# Day 22 — Historical-Only Optimization Adequacy Control

Assess historical optimization adequacy without future-driven tuning.

In [ ]:
# Imports
import json
from pathlib import Path

In [ ]:
# Project paths
def find_project_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "README.md").exists():
            return path
    raise RuntimeError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

def resolve_artifact_paths(value):
    """Rebase legacy project paths in loaded metadata; never rewrite artifacts.

    Keep all non-path values, model identifiers, and scientific settings intact.
    Support the original project name and the public repository name.
    """
    if isinstance(value, dict):
        return {key: resolve_artifact_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_artifact_paths(item) for item in value]
    if isinstance(value, str):
        normalized = value.replace("\\", "/")
        for name in ("FinGPT_Project", "FinGPT-Temporal-Generalization"):
            marker = "/" + name + "/"
            if marker in normalized and "\n" not in normalized:
                # Only standalone filesystem references, never prose containing a path.
                prefix, relative = normalized.split(marker, 1)
                if prefix.startswith("/") or (len(prefix) > 1 and prefix[1] == ":"):
                    if not any(char.isspace() for char in relative):
                        return str(PROJECT_ROOT / relative)
    return value

def load_artifact_json(stream, *args, **kwargs):
    return resolve_artifact_paths(json.load(stream, *args, **kwargs))


In [ ]:
# Reproducibility settings
# Original seeds and deterministic settings remain at their original workflow
# positions below, including stage-specific RNG resets. No new seed is introduced.

## Primary Objective

Improve **historical validation Accuracy** while preserving balanced three-class performance.

Day 22 asks:

> **Can the historical Financial LLM be made materially more accurate and more stable using only historical development data before temporal distribution-shift diagnosis continues?**

The optimization target is:

\[
\boxed{\text{Accuracy}}
\]

with mandatory guardrails:

\[
\boxed{
\text{Macro-F1},\;
\text{Negative Recall},\;
\text{Positive Recall},\;
\text{Invalid-Generation Rate}
}
\]

An Accuracy improvement obtained through severe minority-class degradation will **not** be considered a valid model-quality improvement.

---

## Experimental Boundary

Day 22 is a **historical-development experiment only**.

The previously constructed future windows

\[
F_1,\ldots,F_6
\]

are strictly outside the Day 22 optimization loop.

They must **not** be used for:

- hyperparameter selection;
- checkpoint selection;
- learning-rate selection;
- sampling-strategy selection;
- LoRA-capacity selection;
- prompt selection;
- early stopping;
- seed selection;
- candidate-model ranking.

All optimization decisions will be made using historical data only.

---

## Historical Data Protocol

The frozen Day 14 historical populations remain the upstream reference:

\[
N_{\text{Historical Train}} = 13{,}539
\]

\[
N_{\text{Historical Validation}} = 5{,}859
\]

The 13,539-row Historical Train population will be used to construct a deterministic internal:

\[
\text{Optimization Train}
\quad+\quad
\text{Optimization Dev}
\]

split.

All Day 22 model development and candidate selection will use this internal optimization split.

The frozen 5,859-row Historical Validation population will remain outside candidate ranking and will be evaluated only after the final Day 22 configuration has been frozen.

---

## Optimization Sequence

Day 22 will proceed through the following controlled stages:

1. **Baseline reproduction**
2. **Training-horizon adequacy audit**
3. **Class-aware sampling audit**
4. **Learning-rate audit**
5. **LoRA-capacity audit**
6. **Optional prompt refinement — only if formally triggered**
7. **Multi-seed confirmation**
8. **Final configuration freeze**
9. **Full Historical Train retraining**
10. **One formal Historical Validation evaluation**
11. **Final integrity and improvement audit**

Each stage changes only the intended experimental factor.

---

## Reference Configuration

The current historical reference model uses:

- Base model: **Meta-Llama-3-8B**
- Quantization: **4-bit**
- LoRA rank: \(r=8\)
- LoRA alpha: \(32\)
- LoRA dropout: \(0.1\)
- Learning rate: \(1\times10^{-4}\)
- Training cap: **200 steps**
- Sampling: **natural**
- Seed: **42**
- Task: **3-class financial sentiment classification**

Previous Historical Validation reference:

\[
\text{Accuracy} \approx 0.789
\]

\[
\text{Macro-F1} \approx 0.746
\]

The previous selected checkpoint occurred at the 200-step training boundary, motivating the Day 22 training-adequacy audit.

---

## Day 22 Selection Principle

Among candidates that pass all guardrails, selection is lexicographic:

\[
\boxed{
\text{Accuracy}
\rightarrow
\text{Macro-F1}
\rightarrow
\min(R_{\text{neg}},R_{\text{pos}})
\rightarrow
\text{Validation Loss}
\rightarrow
\text{Lower Complexity}
}
\]

No future-window result may alter this rule.

---

## Notebook Hygiene Rule

Diagnostic or forensic cells may be introduced when necessary.

However:

> **Temporary diagnostic cells that do not contribute to the final reproducible Day 22 workflow will be removed after their purpose has been resolved.**

The final notebook should contain only the cells required to reproduce, audit, and interpret the formal Day 22 experiment.

In [1]:
# ==================================================================================================
# DAY 22 — CELL 1
# ENVIRONMENT BOOTSTRAP + DEPENDENCY INTEGRITY
# ==================================================================================================

import os
import sys
import subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError


# --------------------------------------------------------------------------------------------------
# 1. Install / verify CUDA-13-compatible bitsandbytes BEFORE importing it
# --------------------------------------------------------------------------------------------------

REQUIRED_BNB_VERSION = "0.50.2"

try:
    installed_bnb_version = version("bitsandbytes")
except PackageNotFoundError:
    installed_bnb_version = None


if installed_bnb_version != REQUIRED_BNB_VERSION:

    print(
        f"Installing bitsandbytes=={REQUIRED_BNB_VERSION} "
        f"(current: {installed_bnb_version})..."
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            f"bitsandbytes=={REQUIRED_BNB_VERSION}",
        ]
    )

    # Re-read package metadata after installation
    installed_bnb_version = version("bitsandbytes")


# --------------------------------------------------------------------------------------------------
# 3. Import runtime libraries AFTER dependency installation
# --------------------------------------------------------------------------------------------------

import platform
import torch
import bitsandbytes as bnb

from bitsandbytes.cextension import lib


# --------------------------------------------------------------------------------------------------
# 4. Project paths
# --------------------------------------------------------------------------------------------------

# PROJECT_ROOT is discovered in the project-path setup cell.

NOTEBOOK_DIR = (
    PROJECT_ROOT
    / "notebooks"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
)

MODELS_ROOT = (
    PROJECT_ROOT
    / "models"
)


DAY07_DIR = RESULTS_ROOT / "day07"
DAY14_DIR = RESULTS_ROOT / "day14"
DAY15_DIR = RESULTS_ROOT / "day15"
DAY16_DIR = RESULTS_ROOT / "day16"
DAY17_DIR = RESULTS_ROOT / "day17"
DAY20_DIR = RESULTS_ROOT / "day20"
DAY21_DIR = RESULTS_ROOT / "day21"
DAY22_DIR = RESULTS_ROOT / "day22"


DAY22_DIAGNOSTICS_DIR = (
    DAY22_DIR
    / "diagnostics"
)

DAY22_RUNS_DIR = (
    DAY22_DIR
    / "runs"
)

DAY22_FINAL_DIR = (
    DAY22_DIR
    / "final"
)

DAY22_PROTOCOL_DIR = (
    DAY22_DIR
    / "protocol"
)


DAY22_NOTEBOOK_PATH = (
    NOTEBOOK_DIR
    / "day22_historical_model_optimization_and_training_adequacy_audit.ipynb"
)


for path in [
    NOTEBOOK_DIR,
    RESULTS_ROOT,
    MODELS_ROOT,
    DAY22_DIR,
    DAY22_DIAGNOSTICS_DIR,
    DAY22_RUNS_DIR,
    DAY22_FINAL_DIR,
    DAY22_PROTOCOL_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 5. CUDA / GPU / bitsandbytes backend audit
# --------------------------------------------------------------------------------------------------

cuda_available = bool(
    torch.cuda.is_available()
)

cuda_runtime = (
    torch.version.cuda
)

gpu_name = (
    torch.cuda.get_device_name(0)
    if cuda_available
    else None
)

gpu_total_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
    if cuda_available
    else None
)

bnb_native_type = (
    type(lib).__name__
)


# ErrorHandlerMockBNBNativeLibrary means the native CUDA binary failed to load.
bnb_native_backend_valid = (
    lib is not None
    and bnb_native_type
    != "ErrorHandlerMockBNBNativeLibrary"
)


environment_checks = {

    "project_root_exists":
        PROJECT_ROOT.exists(),

    "cuda_available":
        cuda_available,

    "gpu_detected":
        gpu_name is not None,

    "bnb_version_0_50_2":
        bnb.__version__
        == REQUIRED_BNB_VERSION,

    "bnb_native_backend_loaded":
        bnb_native_backend_valid,

    "day14_dir_exists":
        DAY14_DIR.exists(),

    "day15_dir_exists":
        DAY15_DIR.exists(),

    "day22_dir_ready":
        DAY22_DIR.exists(),
}


all_environment_checks_pass = bool(
    all(
        environment_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — ENVIRONMENT BOOTSTRAP + DEPENDENCY INTEGRITY")
print("=" * 120)

print("\n[RUNTIME]")
print(
    f"Python               : "
    f"{sys.version.split()[0]}"
)
print(
    f"Platform             : "
    f"{platform.platform()}"
)
print(
    f"PyTorch              : "
    f"{torch.__version__}"
)
print(
    f"CUDA available       : "
    f"{cuda_available}"
)
print(
    f"CUDA runtime         : "
    f"{cuda_runtime}"
)
print(
    f"GPU                  : "
    f"{gpu_name}"
)

if gpu_total_memory_gb is not None:
    print(
        f"GPU memory           : "
        f"{gpu_total_memory_gb:.2f} GB"
    )


print("\n[BITSANDBYTES]")
print(
    f"Version              : "
    f"{bnb.__version__}"
)
print(
    f"Native backend type  : "
    f"{bnb_native_type}"
)
print(
    f"Native backend valid : "
    f"{bnb_native_backend_valid}"
)


print("\n[PROJECT]")
print(
    f"Project root         : "
    f"{PROJECT_ROOT}"
)
print(
    f"Day 14               : "
    f"{DAY14_DIR}"
)
print(
    f"Day 15               : "
    f"{DAY15_DIR}"
)
print(
    f"Day 22               : "
    f"{DAY22_DIR}"
)
print(
    f"Notebook             : "
    f"{DAY22_NOTEBOOK_PATH}"
)


print("\n[CHECKS]")

for name, passed in environment_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n" + "-" * 120)

if all_environment_checks_pass:

    print(
        "DAY 22 CELL 1 STATUS: PASS — "
        "ENVIRONMENT AND CUDA QUANTIZATION BACKEND READY"
    )

else:

    print(
        "DAY 22 CELL 1 STATUS: FAIL — "
        "DO NOT CONTINUE TO DATA OR MODEL SETUP"
    )

print("=" * 120)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DAY 22 — ENVIRONMENT BOOTSTRAP + DEPENDENCY INTEGRITY

[RUNTIME]
Python               : 3.13.15
Platform             : Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch              : 2.11.0+cu130
CUDA available       : True
CUDA runtime         : 13.0
GPU                  : Tesla T4
GPU memory           : 14.56 GB

[BITSANDBYTES]
Version              : 0.50.2
Native backend type  : CudaBNBNativeLibrary
Native backend valid : True

[PROJECT]
Project root         : /content/drive/MyDrive/FinGPT_Project
Day 14               : /content/drive/MyDrive/FinGPT_Project/results/day14
Day 15               : /content/drive/MyDrive/FinGPT_Project/results/day15
Day 22               : /content/drive/MyDrive/FinGPT_Project/results/day22
Notebook             : /content/drive/MyDrive/FinGPT_Project/notebooks/day22_historical_model_optimization_and_training_adequacy_audit.ipynb

[C

In [2]:
# ==================================================================================================
# DAY 22 — CELL 2
# FROZEN UPSTREAM PROVENANCE + HISTORICAL POPULATION RECONSTRUCTION
# ==================================================================================================

import json
import hashlib
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Required frozen upstream artifacts
# --------------------------------------------------------------------------------------------------

DATASET_B_PATH = (
    DAY07_DIR
    / "temporal_dataset_clean.csv"
)

DAY14_SPLIT_SPEC_PATH = (
    DAY14_DIR
    / "chronological_split_spec.json"
)

DAY15_TRAIN_MANIFEST_PATH = (
    DAY15_DIR
    / "historical_train_manifest.json"
)

DAY15_VALIDATION_MANIFEST_PATH = (
    DAY15_DIR
    / "historical_validation_manifest.json"
)

DAY15_METRICS_PATH = (
    DAY15_DIR
    / "historical_validation_metrics.json"
)

DAY15_FROZEN_MODEL_MANIFEST_PATH = (
    DAY15_DIR
    / "frozen_model_manifest.json"
)

DAY15_CHECKPOINT_SELECTION_PATH = (
    DAY15_DIR
    / "checkpoint_selection.csv"
)


required_upstream_paths = {
    "dataset_b":
        DATASET_B_PATH,

    "day14_split_spec":
        DAY14_SPLIT_SPEC_PATH,

    "day15_train_manifest":
        DAY15_TRAIN_MANIFEST_PATH,

    "day15_validation_manifest":
        DAY15_VALIDATION_MANIFEST_PATH,

    "day15_metrics":
        DAY15_METRICS_PATH,

    "day15_frozen_model_manifest":
        DAY15_FROZEN_MODEL_MANIFEST_PATH,

    "day15_checkpoint_selection":
        DAY15_CHECKPOINT_SELECTION_PATH,
}


# --------------------------------------------------------------------------------------------------
# 2. File-existence gate
# --------------------------------------------------------------------------------------------------

upstream_file_checks = {
    name: path.exists()
    for name, path in required_upstream_paths.items()
}

assert all(upstream_file_checks.values()), (
    "DAY 22 STOP: one or more required Day 14/15 upstream artifacts are missing."
)


# --------------------------------------------------------------------------------------------------
# 3. Load frozen metadata
# --------------------------------------------------------------------------------------------------

with open(
    DAY14_SPLIT_SPEC_PATH,
    "r",
    encoding="utf-8",
) as f:
    day14_split_spec = load_artifact_json(f)


with open(
    DAY15_TRAIN_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    day15_train_manifest = load_artifact_json(f)


with open(
    DAY15_VALIDATION_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    day15_validation_manifest = load_artifact_json(f)


with open(
    DAY15_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    day15_metrics = load_artifact_json(f)


with open(
    DAY15_FROZEN_MODEL_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    day15_frozen_model_manifest = load_artifact_json(f)


day15_checkpoint_selection = pd.read_csv(
    DAY15_CHECKPOINT_SELECTION_PATH
)


# --------------------------------------------------------------------------------------------------
# 4. Load Dataset B
# --------------------------------------------------------------------------------------------------

dataset_b = pd.read_csv(
    DATASET_B_PATH
)


required_columns = {
    "message_id",
    "channel_username",
    "date",
    "text",
    "sentiment",
}

missing_columns = (
    required_columns
    - set(dataset_b.columns)
)

assert not missing_columns, (
    f"DAY 22 STOP: Dataset B missing columns: {sorted(missing_columns)}"
)


dataset_b["date"] = pd.to_datetime(
    dataset_b["date"],
    utc=True,
    errors="raise",
)


# --------------------------------------------------------------------------------------------------
# 5. Reconstruct canonical observation_id
# --------------------------------------------------------------------------------------------------

dataset_b["observation_id"] = (
    dataset_b["channel_username"].astype(str)
    + "::"
    + dataset_b["message_id"].astype(str)
)


dataset_b = (
    dataset_b
    .sort_values(
        [
            "date",
            "observation_id",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 6. Frozen Day 14 boundaries
# --------------------------------------------------------------------------------------------------

train_start = pd.Timestamp(
    day14_split_spec[
        "historical_train"
    ][
        "start_timestamp_utc"
    ]
)

train_end = pd.Timestamp(
    day14_split_spec[
        "historical_train"
    ][
        "end_timestamp_utc"
    ]
)

validation_start = pd.Timestamp(
    day14_split_spec[
        "historical_validation"
    ][
        "start_timestamp_utc"
    ]
)

validation_end = pd.Timestamp(
    day14_split_spec[
        "historical_validation"
    ][
        "end_timestamp_utc"
    ]
)

future_cutoff = pd.Timestamp(
    day14_split_spec[
        "historical_cutoff"
    ][
        "cutoff_timestamp_utc"
    ]
)


# --------------------------------------------------------------------------------------------------
# 7. Reconstruct exact frozen Historical Train / Validation
# --------------------------------------------------------------------------------------------------

historical_train = (
    dataset_b.loc[
        (dataset_b["date"] >= train_start)
        &
        (dataset_b["date"] <= train_end)
    ]
    .copy()
    .sort_values(
        [
            "date",
            "observation_id",
        ]
    )
    .reset_index(drop=True)
)


historical_validation = (
    dataset_b.loc[
        (dataset_b["date"] >= validation_start)
        &
        (dataset_b["date"] <= validation_end)
    ]
    .copy()
    .sort_values(
        [
            "date",
            "observation_id",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 8. Deterministic identity hashes
# --------------------------------------------------------------------------------------------------

def ordered_observation_hash(df):

    payload = "\n".join(
        df[
            "observation_id"
        ]
        .astype(str)
        .tolist()
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


historical_train_hash = (
    ordered_observation_hash(
        historical_train
    )
)

historical_validation_hash = (
    ordered_observation_hash(
        historical_validation
    )
)


# --------------------------------------------------------------------------------------------------
# 9. Frozen expected quantities
# --------------------------------------------------------------------------------------------------

EXPECTED_TRAIN_N = 13_539
EXPECTED_VALIDATION_N = 5_859


EXPECTED_TRAIN_LABEL_COUNTS = {
    "negative": 2782,
    "neutral": 8436,
    "positive": 2321,
}


EXPECTED_VALIDATION_LABEL_COUNTS = {
    "negative": 1132,
    "neutral": 3687,
    "positive": 1040,
}


EXPECTED_TRAIN_HASH = (
    "2318e6c4c98676cd268a9728349f7a2abeddaccc0a10c925244ac88c71fc6ae1"
)

EXPECTED_VALIDATION_HASH = (
    "09ccb61d3ae6cc4e384c7fe5364043a5f7806ca31ee72830324d149f573748bf"
)


LABEL_ORDER = [
    "negative",
    "neutral",
    "positive",
]


train_label_counts = (
    historical_train[
        "sentiment"
    ]
    .value_counts()
    .reindex(
        LABEL_ORDER,
        fill_value=0,
    )
    .astype(int)
    .to_dict()
)


validation_label_counts = (
    historical_validation[
        "sentiment"
    ]
    .value_counts()
    .reindex(
        LABEL_ORDER,
        fill_value=0,
    )
    .astype(int)
    .to_dict()
)


# --------------------------------------------------------------------------------------------------
# 10. Day 15 reference metrics
# --------------------------------------------------------------------------------------------------

DAY15_REFERENCE_ACCURACY = float(
    day15_metrics["accuracy"]
)

DAY15_REFERENCE_MACRO_F1 = float(
    day15_metrics["macro_f1"]
)

DAY15_REFERENCE_WEIGHTED_F1 = float(
    day15_metrics["weighted_f1"]
)

DAY15_REFERENCE_NEG_RECALL = float(
    day15_metrics[
        "class_recall"
    ][
        "negative"
    ]
)

DAY15_REFERENCE_NEU_RECALL = float(
    day15_metrics[
        "class_recall"
    ][
        "neutral"
    ]
)

DAY15_REFERENCE_POS_RECALL = float(
    day15_metrics[
        "class_recall"
    ][
        "positive"
    ]
)

DAY15_SELECTED_STEP = int(
    day15_metrics[
        "selected_checkpoint_step"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. Formal provenance checks
# --------------------------------------------------------------------------------------------------

train_ids = set(
    historical_train[
        "observation_id"
    ].astype(str)
)

validation_ids = set(
    historical_validation[
        "observation_id"
    ].astype(str)
)


provenance_checks = {

    "dataset_b_exists":
        DATASET_B_PATH.exists(),

    "dataset_b_rows_39965":
        len(dataset_b) == 39_965,

    "historical_train_n_13539":
        len(historical_train)
        == EXPECTED_TRAIN_N,

    "historical_validation_n_5859":
        len(historical_validation)
        == EXPECTED_VALIDATION_N,

    "historical_train_label_counts_exact":
        train_label_counts
        == EXPECTED_TRAIN_LABEL_COUNTS,

    "historical_validation_label_counts_exact":
        validation_label_counts
        == EXPECTED_VALIDATION_LABEL_COUNTS,

    "historical_train_sources_6":
        historical_train[
            "channel_username"
        ].nunique()
        == 6,

    "historical_validation_sources_7":
        historical_validation[
            "channel_username"
        ].nunique()
        == 7,

    "historical_train_hash_exact":
        historical_train_hash
        == EXPECTED_TRAIN_HASH,

    "historical_validation_hash_exact":
        historical_validation_hash
        == EXPECTED_VALIDATION_HASH,

    "historical_train_ids_unique":
        historical_train[
            "observation_id"
        ].is_unique,

    "historical_validation_ids_unique":
        historical_validation[
            "observation_id"
        ].is_unique,

    "train_validation_overlap_zero":
        train_ids.isdisjoint(
            validation_ids
        ),

    "historical_train_before_validation":
        historical_train["date"].max()
        <
        historical_validation["date"].min(),

    "historical_validation_before_future":
        historical_validation["date"].max()
        <
        future_cutoff,

    "train_no_future_rows":
        (
            historical_train["date"]
            < future_cutoff
        ).all(),

    "validation_no_future_rows":
        (
            historical_validation["date"]
            < future_cutoff
        ).all(),

    "train_text_complete":
        (
            historical_train["text"].notna()
            &
            historical_train["text"]
            .astype(str)
            .str.strip()
            .ne("")
        ).all(),

    "validation_text_complete":
        (
            historical_validation["text"].notna()
            &
            historical_validation["text"]
            .astype(str)
            .str.strip()
            .ne("")
        ).all(),

    "train_labels_valid":
        historical_train[
            "sentiment"
        ].isin(
            LABEL_ORDER
        ).all(),

    "validation_labels_valid":
        historical_validation[
            "sentiment"
        ].isin(
            LABEL_ORDER
        ).all(),

    "day15_selected_checkpoint_200":
        DAY15_SELECTED_STEP
        == 200,
}


all_provenance_checks_pass = bool(
    all(
        provenance_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Compact report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — FROZEN UPSTREAM PROVENANCE RECONSTRUCTION")
print("=" * 120)


print("\n[DATASET B]")
print(
    f"Path                  : "
    f"{DATASET_B_PATH}"
)
print(
    f"Total rows            : "
    f"{len(dataset_b):,}"
)


print("\n[HISTORICAL TRAIN]")
print(
    f"Rows                  : "
    f"{len(historical_train):,}"
)
print(
    f"Date range            : "
    f"{historical_train['date'].min()} "
    f"→ {historical_train['date'].max()}"
)
print(
    f"Label counts          : "
    f"{train_label_counts}"
)
print(
    f"Sources               : "
    f"{historical_train['channel_username'].nunique()}"
)
print(
    f"Observation ID SHA256 : "
    f"{historical_train_hash}"
)


print("\n[HISTORICAL VALIDATION]")
print(
    f"Rows                  : "
    f"{len(historical_validation):,}"
)
print(
    f"Date range            : "
    f"{historical_validation['date'].min()} "
    f"→ {historical_validation['date'].max()}"
)
print(
    f"Label counts          : "
    f"{validation_label_counts}"
)
print(
    f"Sources               : "
    f"{historical_validation['channel_username'].nunique()}"
)
print(
    f"Observation ID SHA256 : "
    f"{historical_validation_hash}"
)


print("\n[DAY 15 REFERENCE]")
print(
    f"Accuracy              : "
    f"{DAY15_REFERENCE_ACCURACY:.6f}"
)
print(
    f"Macro-F1              : "
    f"{DAY15_REFERENCE_MACRO_F1:.6f}"
)
print(
    f"Weighted-F1           : "
    f"{DAY15_REFERENCE_WEIGHTED_F1:.6f}"
)
print(
    f"Negative recall       : "
    f"{DAY15_REFERENCE_NEG_RECALL:.6f}"
)
print(
    f"Neutral recall        : "
    f"{DAY15_REFERENCE_NEU_RECALL:.6f}"
)
print(
    f"Positive recall       : "
    f"{DAY15_REFERENCE_POS_RECALL:.6f}"
)
print(
    f"Selected checkpoint   : "
    f"{DAY15_SELECTED_STEP}"
)


print("\n[PROVENANCE CHECKS]")

for name, passed in provenance_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n" + "-" * 120)

if all_provenance_checks_pass:

    print(
        "DAY 22 CELL 2 STATUS: PASS — "
        "FROZEN HISTORICAL POPULATIONS RECONSTRUCTED EXACTLY"
    )

else:

    print(
        "DAY 22 CELL 2 STATUS: FAIL — "
        "DO NOT CONSTRUCT DAY 22 OPTIMIZATION SPLIT"
    )

print("=" * 120)

DAY 22 — FROZEN UPSTREAM PROVENANCE RECONSTRUCTION

[DATASET B]
Path                  : /content/drive/MyDrive/FinGPT_Project/results/day07/temporal_dataset_clean.csv
Total rows            : 39,965

[HISTORICAL TRAIN]
Rows                  : 13,539
Date range            : 2025-06-03 21:06:16+00:00 → 2025-10-31 21:14:14+00:00
Label counts          : {'negative': 2782, 'neutral': 8436, 'positive': 2321}
Sources               : 6
Observation ID SHA256 : 2318e6c4c98676cd268a9728349f7a2abeddaccc0a10c925244ac88c71fc6ae1

[HISTORICAL VALIDATION]
Rows                  : 5,859
Date range            : 2025-11-01 01:50:30+00:00 → 2025-12-31 21:05:31+00:00
Label counts          : {'negative': 1132, 'neutral': 3687, 'positive': 1040}
Sources               : 7
Observation ID SHA256 : 09ccb61d3ae6cc4e384c7fe5364043a5f7806ca31ee72830324d149f573748bf

[DAY 15 REFERENCE]
Accuracy              : 0.789042
Macro-F1              : 0.745678
Weighted-F1           : 0.788907
Negative recall       : 0.639576
Ne

In [3]:
# ==================================================================================================
# DAY 22 — CELL 3
# DETERMINISTIC HISTORICAL OPTIMIZATION SPLIT CONSTRUCTION
# ==================================================================================================

import json
import hashlib
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

OPT_SPLIT_MANIFEST_PATH = (
    DAY22_PROTOCOL_DIR
    / "historical_optimization_split_manifest.csv"
)

OPT_SPLIT_AUDIT_PATH = (
    DAY22_PROTOCOL_DIR
    / "historical_optimization_split_audit.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Start strictly from frozen Historical Train
# --------------------------------------------------------------------------------------------------

optimization_population = (
    historical_train
    .copy()
    .sort_values(
        [
            "date",
            "observation_id",
        ]
    )
    .reset_index(drop=True)
)

N_TOTAL = len(
    optimization_population
)

assert N_TOTAL == 13_539


# --------------------------------------------------------------------------------------------------
# 3. Determine chronological boundary nearest to 80%
#
# Rule:
#   - target train fraction = 80%
#   - never split identical timestamps
#   - choose legal boundary closest to target
#   - ties resolved toward earlier/smaller train partition
# --------------------------------------------------------------------------------------------------

TARGET_TRAIN_FRACTION = 0.80

target_train_n = (
    N_TOTAL
    * TARGET_TRAIN_FRACTION
)


timestamp_counts = (
    optimization_population
    .groupby(
        "date",
        sort=True,
    )
    .size()
    .reset_index(
        name="timestamp_n"
    )
)


timestamp_counts[
    "cumulative_n"
] = (
    timestamp_counts[
        "timestamp_n"
    ]
    .cumsum()
)


# Final timestamp cannot be boundary because dev must remain nonempty.
legal_boundaries = (
    timestamp_counts
    .iloc[:-1]
    .copy()
)


legal_boundaries[
    "distance_from_target"
] = (
    legal_boundaries[
        "cumulative_n"
    ]
    - target_train_n
).abs()


legal_boundaries = (
    legal_boundaries
    .sort_values(
        [
            "distance_from_target",
            "cumulative_n",
            "date",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


selected_boundary = (
    legal_boundaries
    .iloc[0]
)


optimization_train_end = pd.Timestamp(
    selected_boundary[
        "date"
    ]
)


expected_train_n = int(
    selected_boundary[
        "cumulative_n"
    ]
)


# --------------------------------------------------------------------------------------------------
# 4. Construct Optimization Train / Dev
# --------------------------------------------------------------------------------------------------

optimization_train = (
    optimization_population.loc[
        optimization_population[
            "date"
        ]
        <= optimization_train_end
    ]
    .copy()
    .sort_values(
        [
            "date",
            "observation_id",
        ]
    )
    .reset_index(drop=True)
)


optimization_dev = (
    optimization_population.loc[
        optimization_population[
            "date"
        ]
        > optimization_train_end
    ]
    .copy()
    .sort_values(
        [
            "date",
            "observation_id",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 5. Identity hash helper
# --------------------------------------------------------------------------------------------------

def ordered_observation_hash(df):

    payload = "\n".join(
        df[
            "observation_id"
        ]
        .astype(str)
        .tolist()
    )

    return hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()


optimization_population_hash = (
    ordered_observation_hash(
        optimization_population
    )
)

optimization_train_hash = (
    ordered_observation_hash(
        optimization_train
    )
)

optimization_dev_hash = (
    ordered_observation_hash(
        optimization_dev
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Label summaries
# --------------------------------------------------------------------------------------------------

def ordered_label_counts(df):

    return (
        df[
            "sentiment"
        ]
        .value_counts()
        .reindex(
            LABEL_ORDER,
            fill_value=0,
        )
        .astype(int)
        .to_dict()
    )


train_label_counts_day22 = (
    ordered_label_counts(
        optimization_train
    )
)

dev_label_counts_day22 = (
    ordered_label_counts(
        optimization_dev
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Core split integrity
# --------------------------------------------------------------------------------------------------

train_ids = set(
    optimization_train[
        "observation_id"
    ]
    .astype(str)
)

dev_ids = set(
    optimization_dev[
        "observation_id"
    ]
    .astype(str)
)


combined_ids = (
    list(
        optimization_train[
            "observation_id"
        ].astype(str)
    )
    +
    list(
        optimization_dev[
            "observation_id"
        ].astype(str)
    )
)


shared_timestamps = (
    set(
        optimization_train[
            "date"
        ].unique()
    )
    &
    set(
        optimization_dev[
            "date"
        ].unique()
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Exact-text leakage checks
# --------------------------------------------------------------------------------------------------

train_raw_text = set(
    optimization_train[
        "text"
    ]
    .astype(str)
)

dev_raw_text = set(
    optimization_dev[
        "text"
    ]
    .astype(str)
)


raw_exact_overlap = (
    train_raw_text
    &
    dev_raw_text
)


def normalize_exact_text(series):

    return (
        series
        .astype(str)
        .str.lower()
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.strip()
    )


train_normalized_text = set(
    normalize_exact_text(
        optimization_train[
            "text"
        ]
    )
)

dev_normalized_text = set(
    normalize_exact_text(
        optimization_dev[
            "text"
        ]
    )
)


normalized_exact_overlap = (
    train_normalized_text
    &
    dev_normalized_text
)

normalized_exact_overlap.discard("")


# --------------------------------------------------------------------------------------------------
# 9. Formal checks
# --------------------------------------------------------------------------------------------------

split_checks = {

    "parent_population_n_13539":
        N_TOTAL == 13_539,

    "optimization_train_n_10831":
        len(
            optimization_train
        )
        == 10_831,

    "optimization_dev_n_2708":
        len(
            optimization_dev
        )
        == 2_708,

    "selected_boundary_size_exact":
        len(
            optimization_train
        )
        == expected_train_n,

    "train_fraction_near_80pct":
        abs(
            len(
                optimization_train
            )
            / N_TOTAL
            - 0.80
        )
        < 0.001,

    "full_population_partitioned":
        (
            len(
                combined_ids
            )
            == N_TOTAL
            and len(
                set(
                    combined_ids
                )
            )
            == N_TOTAL
        ),

    "train_ids_unique":
        optimization_train[
            "observation_id"
        ].is_unique,

    "dev_ids_unique":
        optimization_dev[
            "observation_id"
        ].is_unique,

    "train_dev_id_overlap_zero":
        train_ids.isdisjoint(
            dev_ids
        ),

    "strict_chronological_order":
        optimization_train[
            "date"
        ].max()
        <
        optimization_dev[
            "date"
        ].min(),

    "boundary_timestamp_not_split":
        len(
            shared_timestamps
        )
        == 0,

    "train_no_future_rows":
        (
            optimization_train[
                "date"
            ]
            < future_cutoff
        ).all(),

    "dev_no_future_rows":
        (
            optimization_dev[
                "date"
            ]
            < future_cutoff
        ).all(),

    "train_text_complete":
        (
            optimization_train[
                "text"
            ].notna()
            &
            optimization_train[
                "text"
            ]
            .astype(str)
            .str.strip()
            .ne("")
        ).all(),

    "dev_text_complete":
        (
            optimization_dev[
                "text"
            ].notna()
            &
            optimization_dev[
                "text"
            ]
            .astype(str)
            .str.strip()
            .ne("")
        ).all(),

    "raw_exact_text_overlap_zero":
        len(
            raw_exact_overlap
        )
        == 0,

    "normalized_exact_text_overlap_zero":
        len(
            normalized_exact_overlap
        )
        == 0,
}


all_split_checks_pass = bool(
    all(
        split_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Save formal manifest
# --------------------------------------------------------------------------------------------------

manifest_train = (
    optimization_train[
        [
            "observation_id",
            "channel_username",
            "message_id",
            "date",
            "sentiment",
        ]
    ]
    .copy()
)

manifest_train[
    "day22_split"
] = "optimization_train"


manifest_dev = (
    optimization_dev[
        [
            "observation_id",
            "channel_username",
            "message_id",
            "date",
            "sentiment",
        ]
    ]
    .copy()
)

manifest_dev[
    "day22_split"
] = "optimization_dev"


optimization_split_manifest = (
    pd.concat(
        [
            manifest_train,
            manifest_dev,
        ],
        axis=0,
        ignore_index=True,
    )
    .sort_values(
        [
            "date",
            "observation_id",
        ]
    )
    .reset_index(drop=True)
)


optimization_split_manifest.to_csv(
    OPT_SPLIT_MANIFEST_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 11. Save formal audit
# --------------------------------------------------------------------------------------------------

optimization_split_audit = {

    "day": 22,

    "artifact":
        "historical_optimization_split_audit",

    "parent_population": {
        "n_observations":
            int(
                N_TOTAL
            ),

        "observation_id_sha256":
            optimization_population_hash,
    },

    "split_protocol": {
        "type":
            "deterministic_chronological",

        "target_train_fraction":
            0.80,

        "target_train_n":
            float(
                target_train_n
            ),

        "boundary_rule":
            (
                "Complete timestamp boundary nearest to 80% target; "
                "identical timestamps cannot be split."
            ),

        "selected_train_end_timestamp_utc":
            optimization_train[
                "date"
            ]
            .max()
            .isoformat(),

        "selected_dev_start_timestamp_utc":
            optimization_dev[
                "date"
            ]
            .min()
            .isoformat(),
    },

    "optimization_train": {
        "n_observations":
            int(
                len(
                    optimization_train
                )
            ),

        "fraction":
            float(
                len(
                    optimization_train
                )
                / N_TOTAL
            ),

        "label_counts":
            train_label_counts_day22,

        "n_sources":
            int(
                optimization_train[
                    "channel_username"
                ].nunique()
            ),

        "observation_id_sha256":
            optimization_train_hash,
    },

    "optimization_dev": {
        "n_observations":
            int(
                len(
                    optimization_dev
                )
            ),

        "fraction":
            float(
                len(
                    optimization_dev
                )
                / N_TOTAL
            ),

        "label_counts":
            dev_label_counts_day22,

        "n_sources":
            int(
                optimization_dev[
                    "channel_username"
                ].nunique()
            ),

        "observation_id_sha256":
            optimization_dev_hash,
    },

    "exact_leakage_screen": {
        "raw_exact_text_overlap_n":
            int(
                len(
                    raw_exact_overlap
                )
            ),

        "normalized_exact_text_overlap_n":
            int(
                len(
                    normalized_exact_overlap
                )
            ),
    },

    "checks": {
        key: bool(
            value
        )
        for key, value
        in split_checks.items()
    },

    "all_checks_pass":
        all_split_checks_pass,

    "near_duplicate_status":
        "PENDING_CELL_4",
}


with open(
    OPT_SPLIT_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        optimization_split_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 12. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — HISTORICAL OPTIMIZATION SPLIT CONSTRUCTION")
print("=" * 120)


print("\n[PARENT POPULATION]")
print(
    f"Historical Train total : "
    f"{N_TOTAL:,}"
)
print(
    f"Target train n         : "
    f"{target_train_n:,.1f}"
)


print("\n[BOUNDARY]")
print(
    f"Optimization Train end : "
    f"{optimization_train['date'].max()}"
)
print(
    f"Optimization Dev start : "
    f"{optimization_dev['date'].min()}"
)


print("\n[OPTIMIZATION TRAIN]")
print(
    f"Rows                   : "
    f"{len(optimization_train):,} "
    f"({len(optimization_train)/N_TOTAL:.4%})"
)
print(
    f"Label counts           : "
    f"{train_label_counts_day22}"
)
print(
    f"Sources                : "
    f"{optimization_train['channel_username'].nunique()}"
)
print(
    f"Observation ID SHA256  : "
    f"{optimization_train_hash}"
)


print("\n[OPTIMIZATION DEV]")
print(
    f"Rows                   : "
    f"{len(optimization_dev):,} "
    f"({len(optimization_dev)/N_TOTAL:.4%})"
)
print(
    f"Label counts           : "
    f"{dev_label_counts_day22}"
)
print(
    f"Sources                : "
    f"{optimization_dev['channel_username'].nunique()}"
)
print(
    f"Observation ID SHA256  : "
    f"{optimization_dev_hash}"
)


print("\n[EXACT LEAKAGE SCREEN]")
print(
    f"Raw exact overlap      : "
    f"{len(raw_exact_overlap)}"
)
print(
    f"Normalized overlap     : "
    f"{len(normalized_exact_overlap)}"
)


print("\n[SPLIT CHECKS]")

for name, passed in split_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Manifest : "
    f"{OPT_SPLIT_MANIFEST_PATH}"
)
print(
    f"Audit    : "
    f"{OPT_SPLIT_AUDIT_PATH}"
)


print("\n" + "-" * 120)

if all_split_checks_pass:

    print(
        "DAY 22 CELL 3 STATUS: PASS — "
        "DETERMINISTIC OPTIMIZATION SPLIT CONSTRUCTED"
    )

else:

    print(
        "DAY 22 CELL 3 STATUS: FAIL — "
        "DO NOT CONTINUE TO NEAR-DUPLICATE AUDIT"
    )

print("=" * 120)

DAY 22 — HISTORICAL OPTIMIZATION SPLIT CONSTRUCTION

[PARENT POPULATION]
Historical Train total : 13,539
Target train n         : 10,831.2

[BOUNDARY]
Optimization Train end : 2025-10-10 01:36:47+00:00
Optimization Dev start : 2025-10-10 01:37:04+00:00

[OPTIMIZATION TRAIN]
Rows                   : 10,831 (79.9985%)
Label counts           : {'negative': 2276, 'neutral': 6767, 'positive': 1788}
Sources                : 6
Observation ID SHA256  : c99dc271d3fab37d7e1160e48297c07cfd1a459c51c75d6d61f79514d0eb20f7

[OPTIMIZATION DEV]
Rows                   : 2,708 (20.0015%)
Label counts           : {'negative': 506, 'neutral': 1669, 'positive': 533}
Sources                : 6
Observation ID SHA256  : 4ca97f36f014198c9d7852d55b6f5ef98b866582c03dfa544f6714d09b5132b4

[EXACT LEAKAGE SCREEN]
Raw exact overlap      : 0
Normalized overlap     : 0

[SPLIT CHECKS]
PASS   — parent_population_n_13539
PASS   — optimization_train_n_10831
PASS   — optimization_dev_n_2708
PASS   — selected_boundary_size_

In [4]:
# ==================================================================================================
# DAY 22 — CELL 4
# CROSS-BOUNDARY NEAR-DUPLICATE AUDIT + FORMAL ADJUDICATION + SPLIT FREEZE
# ==================================================================================================

import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


# --------------------------------------------------------------------------------------------------
# 1. Formal settings / paths
# --------------------------------------------------------------------------------------------------

NEAR_DUPLICATE_THRESHOLD = 0.95


NEAR_DUPLICATE_CANDIDATES_PATH = (
    DAY22_PROTOCOL_DIR
    / "historical_optimization_near_duplicate_candidates.csv"
)

NEAR_DUPLICATE_ADJUDICATION_PATH = (
    DAY22_PROTOCOL_DIR
    / "historical_optimization_near_duplicate_adjudication.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Normalize text for leakage screening
# --------------------------------------------------------------------------------------------------

train_text = (
    optimization_train["text"]
    .astype(str)
    .str.lower()
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)


dev_text = (
    optimization_dev["text"]
    .astype(str)
    .str.lower()
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)


# --------------------------------------------------------------------------------------------------
# 3. Fit TF-IDF on the complete internal historical optimization population
#
# IMPORTANT:
# This is a leakage diagnostic only.
# Historical Validation and Future F1–F6 are not used.
# --------------------------------------------------------------------------------------------------

tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    lowercase=False,
    norm="l2",
    dtype=np.float32,
)


combined_text = pd.concat(
    [
        train_text,
        dev_text,
    ],
    axis=0,
    ignore_index=True,
)


tfidf.fit(
    combined_text
)


X_train_tfidf = tfidf.transform(
    train_text
)

X_dev_tfidf = tfidf.transform(
    dev_text
)


# --------------------------------------------------------------------------------------------------
# 4. For each Optimization Dev row, find nearest Optimization Train row
# --------------------------------------------------------------------------------------------------

nn = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)


nn.fit(
    X_train_tfidf
)


distances, indices = nn.kneighbors(
    X_dev_tfidf,
    return_distance=True,
)


nearest_similarity = (
    1.0
    - distances[:, 0]
)

nearest_train_index = (
    indices[:, 0]
)


candidate_mask = (
    nearest_similarity
    >= NEAR_DUPLICATE_THRESHOLD
)

candidate_dev_indices = np.where(
    candidate_mask
)[0]


# --------------------------------------------------------------------------------------------------
# 5. Build candidate table
# --------------------------------------------------------------------------------------------------

candidate_rows = []


for dev_idx in candidate_dev_indices:

    train_idx = int(
        nearest_train_index[
            dev_idx
        ]
    )

    train_row = (
        optimization_train
        .iloc[
            train_idx
        ]
    )

    dev_row = (
        optimization_dev
        .iloc[
            dev_idx
        ]
    )

    candidate_rows.append(
        {
            "train_observation_id":
                str(
                    train_row[
                        "observation_id"
                    ]
                ),

            "dev_observation_id":
                str(
                    dev_row[
                        "observation_id"
                    ]
                ),

            "train_date":
                train_row[
                    "date"
                ].isoformat(),

            "dev_date":
                dev_row[
                    "date"
                ].isoformat(),

            "train_source":
                str(
                    train_row[
                        "channel_username"
                    ]
                ),

            "dev_source":
                str(
                    dev_row[
                        "channel_username"
                    ]
                ),

            "train_label":
                str(
                    train_row[
                        "sentiment"
                    ]
                ),

            "dev_label":
                str(
                    dev_row[
                        "sentiment"
                    ]
                ),

            "tfidf_cosine_similarity":
                float(
                    nearest_similarity[
                        dev_idx
                    ]
                ),

            "train_text":
                str(
                    train_row[
                        "text"
                    ]
                ),

            "dev_text":
                str(
                    dev_row[
                        "text"
                    ]
                ),
        }
    )


near_duplicate_candidates = pd.DataFrame(
    candidate_rows
)


if not near_duplicate_candidates.empty:

    near_duplicate_candidates = (
        near_duplicate_candidates
        .sort_values(
            [
                "train_observation_id",
                "dev_observation_id",
            ]
        )
        .reset_index(drop=True)
    )


# --------------------------------------------------------------------------------------------------
# 6. Expected candidate identities from the completed forensic review
# --------------------------------------------------------------------------------------------------

EXPECTED_CANDIDATE_PAIRS = {

    ("firstsquaw::54754", "firstsquaw::57623"),
    ("firstsquaw::54757", "firstsquaw::57624"),
    ("firstsquaw::54759", "firstsquaw::57630"),
    ("firstsquaw::54756", "firstsquaw::57631"),
    ("firstsquaw::51620", "firstsquaw::57654"),
    ("firstsquaw::46852", "firstsquaw::58062"),
    ("firstsquaw::51054", "firstsquaw::57660"),
    ("TreeNewsFeed::8088", "TreeNewsFeed::8127"),
    ("firstsquaw::54758", "firstsquaw::57626"),
}


observed_candidate_pairs = set(
    zip(
        near_duplicate_candidates[
            "train_observation_id"
        ],
        near_duplicate_candidates[
            "dev_observation_id"
        ],
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Formal candidate identity checks
# --------------------------------------------------------------------------------------------------

candidate_checks = {

    "candidate_pair_n_9":
        len(
            near_duplicate_candidates
        )
        == 9,

    "candidate_pair_identity_exact":
        observed_candidate_pairs
        == EXPECTED_CANDIDATE_PAIRS,

    "all_candidate_similarity_ge_095":
        bool(
            (
                near_duplicate_candidates[
                    "tfidf_cosine_similarity"
                ]
                >= 0.95
            ).all()
        ),

    "candidate_train_rows_unique":
        near_duplicate_candidates[
            "train_observation_id"
        ].nunique()
        == 9,

    "candidate_dev_rows_unique":
        near_duplicate_candidates[
            "dev_observation_id"
        ].nunique()
        == 9,
}


assert all(
    candidate_checks.values()
), (
    "DAY 22 STOP: near-duplicate candidate set differs from "
    "the previously completed forensic review."
)


# --------------------------------------------------------------------------------------------------
# 8. Formal adjudication
#
# Previous forensic review established:
#
#   - no raw exact duplicate pairs;
#   - differences are economically meaningful release values
#     (actual / estimate / previous);
#   - candidate similarity is driven by recurring macroeconomic-release templates;
#   - observations correspond to distinct events/releases.
#
# Therefore all nine remain in their original chronological split.
# --------------------------------------------------------------------------------------------------

near_duplicate_adjudication = (
    near_duplicate_candidates
    .copy()
)


near_duplicate_adjudication[
    "label_agreement"
] = (
    near_duplicate_adjudication[
        "train_label"
    ]
    ==
    near_duplicate_adjudication[
        "dev_label"
    ]
)


near_duplicate_adjudication[
    "adjudication"
] = (
    "template_similar_but_distinct_event"
)


near_duplicate_adjudication[
    "confirmed_duplicate_leakage"
] = False


near_duplicate_adjudication[
    "retain_in_original_split"
] = True


near_duplicate_adjudication[
    "adjudication_reason"
] = (
    "Recurring macroeconomic-release template with economically meaningful "
    "numeric differences across observations; distinct event/release rather "
    "than duplicated record."
)


# --------------------------------------------------------------------------------------------------
# 9. Summary statistics
# --------------------------------------------------------------------------------------------------

candidate_pair_n = int(
    len(
        near_duplicate_candidates
    )
)


confirmed_duplicate_pair_n = int(
    near_duplicate_adjudication[
        "confirmed_duplicate_leakage"
    ].sum()
)


retained_distinct_event_pair_n = int(
    near_duplicate_adjudication[
        "retain_in_original_split"
    ].sum()
)


different_label_pair_n = int(
    (
        near_duplicate_adjudication[
            "train_label"
        ]
        !=
        near_duplicate_adjudication[
            "dev_label"
        ]
    ).sum()
)


max_similarity = float(
    nearest_similarity.max()
)


p99_similarity = float(
    np.quantile(
        nearest_similarity,
        0.99,
    )
)


p999_similarity = float(
    np.quantile(
        nearest_similarity,
        0.999,
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Save formal candidate / adjudication artifacts
# --------------------------------------------------------------------------------------------------

near_duplicate_candidates.to_csv(
    NEAR_DUPLICATE_CANDIDATES_PATH,
    index=False,
)


near_duplicate_adjudication.to_csv(
    NEAR_DUPLICATE_ADJUDICATION_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 11. Update formal split audit
# --------------------------------------------------------------------------------------------------

with open(
    OPT_SPLIT_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:

    optimization_split_audit = (
        load_artifact_json(f)
    )


optimization_split_audit[
    "near_duplicate_audit"
] = {

    "method":
        "word TF-IDF 1-2gram nearest-neighbor cosine",

    "threshold":
        float(
            NEAR_DUPLICATE_THRESHOLD
        ),

    "maximum_similarity":
        max_similarity,

    "p99_similarity":
        p99_similarity,

    "p999_similarity":
        p999_similarity,

    "candidate_pair_n":
        candidate_pair_n,

    "candidate_pair_identity_matches_prior_review":
        True,

    "confirmed_duplicate_leakage_pair_n":
        confirmed_duplicate_pair_n,

    "retained_distinct_event_pair_n":
        retained_distinct_event_pair_n,

    "different_label_pair_n":
        different_label_pair_n,

    "adjudication":
        (
            "All nine candidates are recurring macroeconomic-release "
            "templates with economically meaningful numeric differences; "
            "none are duplicate leakage."
        ),

    "action":
        "retain_all_rows_and_preserve_original_chronological_split",
}


optimization_split_audit[
    "checks"
][
    "near_duplicate_screen_complete"
] = True


optimization_split_audit[
    "checks"
][
    "candidate_identity_matches_forensic_review"
] = True


optimization_split_audit[
    "checks"
][
    "confirmed_duplicate_leakage_zero"
] = (
    confirmed_duplicate_pair_n
    == 0
)


optimization_split_audit[
    "near_duplicate_status"
] = "ADJUDICATED_PASS"


optimization_split_audit[
    "all_checks_pass"
] = bool(
    all(
        bool(v)
        for v
        in optimization_split_audit[
            "checks"
        ].values()
    )
)


optimization_split_audit[
    "freeze_status"
] = (
    "FROZEN_FOR_DAY22_MODEL_DEVELOPMENT"
    if optimization_split_audit[
        "all_checks_pass"
    ]
    else
    "NOT_FROZEN_REVIEW_REQUIRED"
)


with open(
    OPT_SPLIT_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        optimization_split_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 12. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — NEAR-DUPLICATE AUDIT + FORMAL SPLIT FREEZE")
print("=" * 120)


print("\n[METHOD]")
print(
    "Representation          : word TF-IDF, 1–2 grams"
)
print(
    "Similarity              : cosine"
)
print(
    f"Threshold               : "
    f"{NEAR_DUPLICATE_THRESHOLD:.2f}"
)


print("\n[SIMILARITY SUMMARY]")
print(
    f"Maximum similarity      : "
    f"{max_similarity:.6f}"
)
print(
    f"99th percentile         : "
    f"{p99_similarity:.6f}"
)
print(
    f"99.9th percentile       : "
    f"{p999_similarity:.6f}"
)


print("\n[CANDIDATES]")
print(
    f"Pairs >= 0.95           : "
    f"{candidate_pair_n}"
)
print(
    f"Unique Train rows       : "
    f"{near_duplicate_candidates['train_observation_id'].nunique()}"
)
print(
    f"Unique Dev rows         : "
    f"{near_duplicate_candidates['dev_observation_id'].nunique()}"
)


print("\n[ADJUDICATION]")
print(
    f"Confirmed duplicates    : "
    f"{confirmed_duplicate_pair_n}"
)
print(
    f"Distinct events retained: "
    f"{retained_distinct_event_pair_n}"
)
print(
    f"Different-label pairs   : "
    f"{different_label_pair_n}"
)


print("\n[CANDIDATE CHECKS]")

for name, passed in candidate_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[FINAL SPLIT CHECKS]")

for name, passed in optimization_split_audit[
    "checks"
].items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Candidates   : "
    f"{NEAR_DUPLICATE_CANDIDATES_PATH}"
)
print(
    f"Adjudication : "
    f"{NEAR_DUPLICATE_ADJUDICATION_PATH}"
)
print(
    f"Split audit  : "
    f"{OPT_SPLIT_AUDIT_PATH}"
)


print("\n" + "-" * 120)

if optimization_split_audit[
    "all_checks_pass"
]:

    print(
        "DAY 22 CELL 4 STATUS: PASS — "
        "HISTORICAL OPTIMIZATION SPLIT FROZEN FOR MODEL DEVELOPMENT"
    )

else:

    print(
        "DAY 22 CELL 4 STATUS: FAIL — "
        "DO NOT BEGIN MODEL DEVELOPMENT"
    )

print("=" * 120)

DAY 22 — NEAR-DUPLICATE AUDIT + FORMAL SPLIT FREEZE

[METHOD]
Representation          : word TF-IDF, 1–2 grams
Similarity              : cosine
Threshold               : 0.95

[SIMILARITY SUMMARY]
Maximum similarity      : 1.000000
99th percentile         : 0.764131
99.9th percentile       : 1.000000

[CANDIDATES]
Pairs >= 0.95           : 9
Unique Train rows       : 9
Unique Dev rows         : 9

[ADJUDICATION]
Confirmed duplicates    : 0
Distinct events retained: 9
Different-label pairs   : 3

[CANDIDATE CHECKS]
PASS   — candidate_pair_n_9
PASS   — candidate_pair_identity_exact
PASS   — all_candidate_similarity_ge_095
PASS   — candidate_train_rows_unique
PASS   — candidate_dev_rows_unique

[FINAL SPLIT CHECKS]
PASS   — parent_population_n_13539
PASS   — optimization_train_n_10831
PASS   — optimization_dev_n_2708
PASS   — selected_boundary_size_exact
PASS   — train_fraction_near_80pct
PASS   — full_population_partitioned
PASS   — train_ids_unique
PASS   — dev_ids_unique
PASS   — train

In [5]:
# ==================================================================================================
# DAY 22 — CELL 5
# STAGE A BASELINE REPRODUCTION PROTOCOL FREEZE
# ==================================================================================================

import json
from datetime import datetime, timezone


# --------------------------------------------------------------------------------------------------
# 1. Stage A directories / artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_A_DIR = (
    DAY22_RUNS_DIR
    / "stage_a_baseline"
)

STAGE_A_MODEL_DIR = (
    STAGE_A_DIR
    / "model"
)

STAGE_A_CHECKPOINT_DIR = (
    STAGE_A_MODEL_DIR
    / "checkpoints"
)

STAGE_A_ARTIFACT_DIR = (
    STAGE_A_DIR
    / "artifacts"
)


for path in [
    STAGE_A_DIR,
    STAGE_A_MODEL_DIR,
    STAGE_A_CHECKPOINT_DIR,
    STAGE_A_ARTIFACT_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


STAGE_A_CONFIG_PATH = (
    STAGE_A_ARTIFACT_DIR
    / "stage_a_baseline_config.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Exact Day 15 task interface
#
# IMPORTANT:
# The prompt template intentionally contains literal "\n" characters as text.
# This matches the exact Day 15 implementation.
# --------------------------------------------------------------------------------------------------

EXACT_INSTRUCTION_TEXT = (
    "Classify the sentiment of the following financial news "
    "as negative, neutral, or positive."
)

EXACT_PROMPT_TEMPLATE = (
    r"Instruction:{instruction}\nInput:{input}\nAnswer: "
)


# --------------------------------------------------------------------------------------------------
# 3. Stage A frozen configuration
# --------------------------------------------------------------------------------------------------

STAGE_A_CONFIG = {

    "day":
        22,

    "stage":
        "A",

    "stage_name":
        "baseline_reproduction",

    "objective":
        (
            "Reproduce the Day 15 baseline configuration on the "
            "Day 22 frozen Optimization Train / Optimization Dev split."
        ),

    # ----------------------------------------------------------------------------------------------
    # Data
    # ----------------------------------------------------------------------------------------------

    "data_protocol": {

        "parent_population":
            "Day14 frozen Historical Train",

        "training_split":
            "optimization_train",

        "training_n":
            int(
                len(
                    optimization_train
                )
            ),

        "evaluation_split":
            "optimization_dev",

        "evaluation_n":
            int(
                len(
                    optimization_dev
                )
            ),

        "sampling":
            "natural",

        "historical_validation_used_for_selection":
            False,

        "future_f1_f6_used":
            False,

        "optimization_split_audit":
            str(
                OPT_SPLIT_AUDIT_PATH
            ),
    },

    # ----------------------------------------------------------------------------------------------
    # Base model
    # ----------------------------------------------------------------------------------------------

    "base_model": {

        "model_name":
            "meta-llama/Meta-Llama-3-8B",

        "task":
            "causal_language_modeling",

        "trust_remote_code":
            True,

        "device_map":
            "auto",
    },

    # ----------------------------------------------------------------------------------------------
    # Quantization
    # ----------------------------------------------------------------------------------------------

    "quantization": {

        "enabled":
            True,

        "load_in_4bit":
            True,

        "bnb_4bit_quant_type":
            "nf4",

        "bnb_4bit_use_double_quant":
            True,

        "bnb_4bit_compute_dtype":
            "float16",
    },

    # ----------------------------------------------------------------------------------------------
    # LoRA
    # ----------------------------------------------------------------------------------------------

    "lora": {

        "r":
            8,

        "lora_alpha":
            32,

        "lora_dropout":
            0.1,

        "target_modules":
            [
                "q_proj",
                "v_proj",
            ],

        "bias":
            "none",

        "task_type":
            "CAUSAL_LM",

        "inference_mode":
            False,
    },

    # ----------------------------------------------------------------------------------------------
    # Sequence / task interface
    # ----------------------------------------------------------------------------------------------

    "sequence_protocol": {

        "max_seq_length":
            512,

        "instruction_text":
            EXACT_INSTRUCTION_TEXT,

        "prompt_template":
            EXACT_PROMPT_TEMPLATE,

        "target_labels":
            [
                "negative",
                "neutral",
                "positive",
            ],

        "append_eos_token":
            True,

        "loss_masking":
            (
                "Mask complete prompt with -100; "
                "compute loss only on target label + EOS."
            ),

        "training_padding_side":
            "right",

        "training_pad_token":
            "eos_token",

        "padding_label_value":
            -100,
    },

    # ----------------------------------------------------------------------------------------------
    # Training
    # ----------------------------------------------------------------------------------------------

    "training": {

        "max_steps":
            200,

        "per_device_train_batch_size":
            1,

        "per_device_eval_batch_size":
            1,

        "gradient_accumulation_steps":
            8,

        "effective_train_batch_size":
            8,

        "learning_rate":
            1e-4,

        "warmup_steps":
            10,

        "weight_decay":
            0.0,

        "gradient_checkpointing":
            True,

        "logging_steps":
            10,

        "eval_steps":
            25,

        "save_steps":
            50,

        "seed":
            42,

        "data_seed":
            42,
    },

    # ----------------------------------------------------------------------------------------------
    # Checkpoint selection
    # ----------------------------------------------------------------------------------------------

    "checkpoint_selection": {

        "candidate_steps":
            [
                50,
                100,
                150,
                200,
            ],

        "selection_split":
            "optimization_dev",

        "selection_metric":
            "eval_loss",

        "selection_direction":
            "minimize",

        "early_stopping":
            False,

        "historical_validation_metrics_allowed":
            False,

        "future_metrics_allowed":
            False,
    },

    # ----------------------------------------------------------------------------------------------
    # Inference
    # ----------------------------------------------------------------------------------------------

    "inference": {

        "model_mode":
            "eval",

        "padding_side":
            "left",

        "pad_token":
            "eos_token",

        "max_new_tokens":
            4,

        "do_sample":
            False,

        "postprocessing":
            [
                "decode generated tokens only",
                "skip special tokens",
                "strip whitespace",
                "lowercase",
            ],

        "valid_predictions":
            [
                "negative",
                "neutral",
                "positive",
            ],
    },

    # ----------------------------------------------------------------------------------------------
    # Day 15 reference only
    # ----------------------------------------------------------------------------------------------

    "day15_reference": {

        "accuracy":
            DAY15_REFERENCE_ACCURACY,

        "macro_f1":
            DAY15_REFERENCE_MACRO_F1,

        "weighted_f1":
            DAY15_REFERENCE_WEIGHTED_F1,

        "negative_recall":
            DAY15_REFERENCE_NEG_RECALL,

        "neutral_recall":
            DAY15_REFERENCE_NEU_RECALL,

        "positive_recall":
            DAY15_REFERENCE_POS_RECALL,

        "selected_checkpoint_step":
            DAY15_SELECTED_STEP,
    },

    "freeze_status":
        "FROZEN_BEFORE_STAGE_A_TRAINING",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 4. Formal protocol checks
# --------------------------------------------------------------------------------------------------

protocol_checks = {

    "optimization_train_n_10831":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "training_n"
        ]
        == 10_831,

    "optimization_dev_n_2708":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "evaluation_n"
        ]
        == 2_708,

    "historical_validation_firewall":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used_for_selection"
        ]
        is False,

    "future_firewall":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ]
        is False,

    "natural_sampling":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "sampling"
        ]
        == "natural",

    "base_model_exact":
        STAGE_A_CONFIG[
            "base_model"
        ][
            "model_name"
        ]
        == "meta-llama/Meta-Llama-3-8B",

    "lora_r8":
        STAGE_A_CONFIG[
            "lora"
        ][
            "r"
        ]
        == 8,

    "lora_alpha32":
        STAGE_A_CONFIG[
            "lora"
        ][
            "lora_alpha"
        ]
        == 32,

    "lora_dropout_01":
        STAGE_A_CONFIG[
            "lora"
        ][
            "lora_dropout"
        ]
        == 0.1,

    "learning_rate_1e4":
        STAGE_A_CONFIG[
            "training"
        ][
            "learning_rate"
        ]
        == 1e-4,

    "max_steps_200":
        STAGE_A_CONFIG[
            "training"
        ][
            "max_steps"
        ]
        == 200,

    "seed_42":
        STAGE_A_CONFIG[
            "training"
        ][
            "seed"
        ]
        == 42,

    "candidate_steps_exact":
        STAGE_A_CONFIG[
            "checkpoint_selection"
        ][
            "candidate_steps"
        ]
        == [
            50,
            100,
            150,
            200,
        ],

    "selection_metric_eval_loss":
        STAGE_A_CONFIG[
            "checkpoint_selection"
        ][
            "selection_metric"
        ]
        == "eval_loss",

    "literal_backslash_n_preserved":
        "\\n"
        in STAGE_A_CONFIG[
            "sequence_protocol"
        ][
            "prompt_template"
        ],

    "no_actual_newline_in_prompt_template":
        "\n"
        not in STAGE_A_CONFIG[
            "sequence_protocol"
        ][
            "prompt_template"
        ],
}


all_protocol_checks_pass = bool(
    all(
        protocol_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 5. Save frozen Stage A configuration
# --------------------------------------------------------------------------------------------------

with open(
    STAGE_A_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        STAGE_A_CONFIG,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 6. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE A BASELINE PROTOCOL FREEZE")
print("=" * 120)


print("\n[DATA]")
print(
    f"Optimization Train     : "
    f"{STAGE_A_CONFIG['data_protocol']['training_n']:,}"
)
print(
    f"Optimization Dev       : "
    f"{STAGE_A_CONFIG['data_protocol']['evaluation_n']:,}"
)
print(
    f"Sampling               : "
    f"{STAGE_A_CONFIG['data_protocol']['sampling']}"
)


print("\n[MODEL]")
print(
    f"Base model             : "
    f"{STAGE_A_CONFIG['base_model']['model_name']}"
)
print(
    f"LoRA                   : "
    f"r={STAGE_A_CONFIG['lora']['r']}, "
    f"alpha={STAGE_A_CONFIG['lora']['lora_alpha']}, "
    f"dropout={STAGE_A_CONFIG['lora']['lora_dropout']}"
)


print("\n[TASK INTERFACE]")
print(
    f"Instruction            : "
    f"{STAGE_A_CONFIG['sequence_protocol']['instruction_text']}"
)
print(
    f"Prompt repr            : "
    f"{repr(STAGE_A_CONFIG['sequence_protocol']['prompt_template'])}"
)


print("\n[TRAINING]")
print(
    f"Learning rate          : "
    f"{STAGE_A_CONFIG['training']['learning_rate']}"
)
print(
    f"Max steps              : "
    f"{STAGE_A_CONFIG['training']['max_steps']}"
)
print(
    f"Effective batch        : "
    f"{STAGE_A_CONFIG['training']['effective_train_batch_size']}"
)
print(
    f"Seed                   : "
    f"{STAGE_A_CONFIG['training']['seed']}"
)


print("\n[CHECKPOINT SELECTION]")
print(
    f"Candidate steps        : "
    f"{STAGE_A_CONFIG['checkpoint_selection']['candidate_steps']}"
)
print(
    f"Selection split        : "
    f"{STAGE_A_CONFIG['checkpoint_selection']['selection_split']}"
)
print(
    f"Selection metric       : "
    f"{STAGE_A_CONFIG['checkpoint_selection']['selection_metric']}"
)


print("\n[PROTOCOL CHECKS]")

for name, passed in protocol_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    f"Stage A config         : "
    f"{STAGE_A_CONFIG_PATH}"
)


print("\n" + "-" * 120)

if all_protocol_checks_pass:

    print(
        "DAY 22 CELL 5 STATUS: PASS — "
        "STAGE A BASELINE PROTOCOL FROZEN"
    )

else:

    print(
        "DAY 22 CELL 5 STATUS: FAIL — "
        "DO NOT BUILD TRAINING DATASETS"
    )

print("=" * 120)

DAY 22 — STAGE A BASELINE PROTOCOL FREEZE

[DATA]
Optimization Train     : 10,831
Optimization Dev       : 2,708
Sampling               : natural

[MODEL]
Base model             : meta-llama/Meta-Llama-3-8B
LoRA                   : r=8, alpha=32, dropout=0.1

[TASK INTERFACE]
Instruction            : Classify the sentiment of the following financial news as negative, neutral, or positive.
Prompt repr            : 'Instruction:{instruction}\\nInput:{input}\\nAnswer: '

[TRAINING]
Learning rate          : 0.0001
Max steps              : 200
Effective batch        : 8
Seed                   : 42

[CHECKPOINT SELECTION]
Candidate steps        : [50, 100, 150, 200]
Selection split        : optimization_dev
Selection metric       : eval_loss

[PROTOCOL CHECKS]
PASS   — optimization_train_n_10831
PASS   — optimization_dev_n_2708
PASS   — historical_validation_firewall
PASS   — future_firewall
PASS   — natural_sampling
PASS   — base_model_exact
PASS   — lora_r8
PASS   — lora_alpha32
PASS   — l

In [6]:
# ==================================================================================================
# DAY 22 — CELL 6
# EXACT TOKENIZER + FORMATTER + TARGET-ONLY LOSS MASKING + ENCODED DATASETS
# ==================================================================================================

import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import AutoTokenizer


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage A task interface
# --------------------------------------------------------------------------------------------------

BASE_MODEL_NAME = (
    STAGE_A_CONFIG[
        "base_model"
    ][
        "model_name"
    ]
)

MAX_SEQ_LENGTH = (
    STAGE_A_CONFIG[
        "sequence_protocol"
    ][
        "max_seq_length"
    ]
)

INSTRUCTION_TEXT = (
    STAGE_A_CONFIG[
        "sequence_protocol"
    ][
        "instruction_text"
    ]
)

PROMPT_TEMPLATE = (
    STAGE_A_CONFIG[
        "sequence_protocol"
    ][
        "prompt_template"
    ]
)

VALID_LABELS = set(
    STAGE_A_CONFIG[
        "sequence_protocol"
    ][
        "target_labels"
    ]
)


# --------------------------------------------------------------------------------------------------
# 2. Load tokenizer
# --------------------------------------------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=STAGE_A_CONFIG[
        "base_model"
    ][
        "trust_remote_code"
    ],
)


# Exact Day 15 training convention
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# --------------------------------------------------------------------------------------------------
# 3. Exact Day 15-compatible training encoder
# --------------------------------------------------------------------------------------------------

def encode_training_example(text, label):
    """
    Build one exact Day 15-compatible causal-LM training example.

    Protocol:
      - prompt uses literal "\\n" sequences;
      - target = sentiment label + EOS;
      - prompt tokens are fully masked with -100;
      - only target + EOS contribute to loss;
      - if truncation is required, truncate prompt only.
    """

    text = str(text)
    label = str(label).strip().lower()

    if label not in VALID_LABELS:
        raise ValueError(
            f"Invalid sentiment label: {label}"
        )

    prompt = PROMPT_TEMPLATE.format(
        instruction=INSTRUCTION_TEXT,
        input=text,
    )

    target = (
        label
        + tokenizer.eos_token
    )

    # Tokenize prompt and target separately
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    target_ids = tokenizer(
        target,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    original_prompt_token_count = len(
        prompt_ids
    )

    # Preserve target completely
    max_prompt_length = (
        MAX_SEQ_LENGTH
        - len(target_ids)
    )

    if max_prompt_length <= 0:
        raise ValueError(
            "Target token length exceeds MAX_SEQ_LENGTH."
        )

    prompt_was_truncated = (
        len(prompt_ids)
        > max_prompt_length
    )

    if prompt_was_truncated:
        prompt_ids = (
            prompt_ids[
                :max_prompt_length
            ]
        )

    input_ids = (
        prompt_ids
        + target_ids
    )

    attention_mask = (
        [1]
        * len(input_ids)
    )

    labels = (
        [-100]
        * len(prompt_ids)
        +
        target_ids.copy()
    )

    return {
        "input_ids":
            input_ids,

        "attention_mask":
            attention_mask,

        "labels":
            labels,

        "prompt_token_count":
            len(prompt_ids),

        "original_prompt_token_count":
            original_prompt_token_count,

        "target_token_count":
            len(target_ids),

        "sequence_length":
            len(input_ids),

        "prompt_was_truncated":
            prompt_was_truncated,

        "prompt_text":
            prompt,

        "target_text":
            target,
    }


# --------------------------------------------------------------------------------------------------
# 4. Exact inference prompt builder
# --------------------------------------------------------------------------------------------------

def build_inference_prompt(text):

    return PROMPT_TEMPLATE.format(
        instruction=INSTRUCTION_TEXT,
        input=str(text),
    )


# --------------------------------------------------------------------------------------------------
# 5. Prepare Optimization Train / Dev frames
# --------------------------------------------------------------------------------------------------

DATA_COLUMNS = [
    "observation_id",
    "channel_username",
    "message_id",
    "date",
    "text",
    "sentiment",
]


optimization_train_frame = (
    optimization_train[
        DATA_COLUMNS
    ]
    .copy()
    .reset_index(drop=True)
)


optimization_dev_frame = (
    optimization_dev[
        DATA_COLUMNS
    ]
    .copy()
    .reset_index(drop=True)
)


# Hugging Face Dataset does not need pandas Timestamp objects
optimization_train_frame[
    "date"
] = (
    optimization_train_frame[
        "date"
    ]
    .astype(str)
)


optimization_dev_frame[
    "date"
] = (
    optimization_dev_frame[
        "date"
    ]
    .astype(str)
)


# --------------------------------------------------------------------------------------------------
# 6. Convert to Hugging Face datasets
# --------------------------------------------------------------------------------------------------

hf_opt_train_raw = Dataset.from_pandas(
    optimization_train_frame,
    preserve_index=False,
)


hf_opt_dev_raw = Dataset.from_pandas(
    optimization_dev_frame,
    preserve_index=False,
)


# --------------------------------------------------------------------------------------------------
# 7. Batch encoder
# --------------------------------------------------------------------------------------------------

def encode_batch(batch):

    output = {
        "input_ids": [],
        "attention_mask": [],
        "labels": [],
        "prompt_token_count": [],
        "original_prompt_token_count": [],
        "target_token_count": [],
        "sequence_length": [],
        "prompt_was_truncated": [],
    }

    for text, label in zip(
        batch["text"],
        batch["sentiment"],
    ):

        encoded = (
            encode_training_example(
                text=text,
                label=label,
            )
        )

        for key in output:

            output[
                key
            ].append(
                encoded[
                    key
                ]
            )

    return output


# --------------------------------------------------------------------------------------------------
# 8. Encode full Optimization Train / Dev
# --------------------------------------------------------------------------------------------------

hf_opt_train = hf_opt_train_raw.map(
    encode_batch,
    batched=True,
    desc="Encoding Optimization Train",
)


hf_opt_dev = hf_opt_dev_raw.map(
    encode_batch,
    batched=True,
    desc="Encoding Optimization Dev",
)


# --------------------------------------------------------------------------------------------------
# 9. Sequence diagnostics
# --------------------------------------------------------------------------------------------------

train_lengths = np.asarray(
    hf_opt_train[
        "sequence_length"
    ],
    dtype=int,
)


dev_lengths = np.asarray(
    hf_opt_dev[
        "sequence_length"
    ],
    dtype=int,
)


train_truncated_n = int(
    np.sum(
        hf_opt_train[
            "prompt_was_truncated"
        ]
    )
)


dev_truncated_n = int(
    np.sum(
        hf_opt_dev[
            "prompt_was_truncated"
        ]
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Sample encoding for loss-mask verification
# --------------------------------------------------------------------------------------------------

example_text = (
    optimization_train_frame
    .iloc[0][
        "text"
    ]
)

example_label = (
    optimization_train_frame
    .iloc[0][
        "sentiment"
    ]
)


example_encoded = (
    encode_training_example(
        text=example_text,
        label=example_label,
    )
)


prompt_n = (
    example_encoded[
        "prompt_token_count"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. Formal formatter checks
# --------------------------------------------------------------------------------------------------

formatter_checks = {

    "tokenizer_loaded":
        tokenizer is not None,

    "pad_token_is_eos":
        tokenizer.pad_token_id
        ==
        tokenizer.eos_token_id,

    "training_padding_side_right":
        tokenizer.padding_side
        == "right",

    "max_seq_length_512":
        MAX_SEQ_LENGTH
        == 512,

    "valid_labels_exact":
        VALID_LABELS
        == {
            "negative",
            "neutral",
            "positive",
        },

    "exact_instruction_text":
        INSTRUCTION_TEXT
        == (
            "Classify the sentiment of the following financial news "
            "as negative, neutral, or positive."
        ),

    "literal_backslash_n_preserved":
        "\\n"
        in PROMPT_TEMPLATE,

    "no_actual_newline_in_prompt_template":
        "\n"
        not in PROMPT_TEMPLATE,

    "prompt_loss_fully_masked":
        all(
            value == -100
            for value
            in example_encoded[
                "labels"
            ][
                :prompt_n
            ]
        ),

    "target_loss_unmasked":
        all(
            value != -100
            for value
            in example_encoded[
                "labels"
            ][
                prompt_n:
            ]
        ),

    "target_contains_eos":
        example_encoded[
            "input_ids"
        ][-1]
        ==
        tokenizer.eos_token_id,
}


# --------------------------------------------------------------------------------------------------
# 12. Formal encoded-dataset checks
# --------------------------------------------------------------------------------------------------

encoded_dataset_checks = {

    "optimization_train_n_10831":
        len(
            hf_opt_train
        )
        == 10_831,

    "optimization_dev_n_2708":
        len(
            hf_opt_dev
        )
        == 2_708,

    "train_all_within_512":
        int(
            train_lengths.max()
        )
        <= MAX_SEQ_LENGTH,

    "dev_all_within_512":
        int(
            dev_lengths.max()
        )
        <= MAX_SEQ_LENGTH,

    "train_labels_present":
        "labels"
        in hf_opt_train.column_names,

    "dev_labels_present":
        "labels"
        in hf_opt_dev.column_names,

    "train_attention_mask_present":
        "attention_mask"
        in hf_opt_train.column_names,

    "dev_attention_mask_present":
        "attention_mask"
        in hf_opt_dev.column_names,

    "train_observation_id_preserved":
        "observation_id"
        in hf_opt_train.column_names,

    "dev_observation_id_preserved":
        "observation_id"
        in hf_opt_dev.column_names,

    "train_no_prompt_truncation":
        train_truncated_n
        == 0,

    "dev_no_prompt_truncation":
        dev_truncated_n
        == 0,
}


all_encoding_checks_pass = bool(
    all(
        formatter_checks.values()
    )
    and
    all(
        encoded_dataset_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — EXACT TOKENIZER + FORMATTER + ENCODED OPTIMIZATION DATASETS")
print("=" * 120)


print("\n[TASK INTERFACE]")
print(
    f"Instruction            : "
    f"{INSTRUCTION_TEXT}"
)
print(
    f"Prompt repr            : "
    f"{repr(PROMPT_TEMPLATE)}"
)
print(
    f"EOS token              : "
    f"{repr(tokenizer.eos_token)}"
)
print(
    f"EOS token ID           : "
    f"{tokenizer.eos_token_id}"
)
print(
    f"Pad token ID           : "
    f"{tokenizer.pad_token_id}"
)
print(
    f"Training padding       : "
    f"{tokenizer.padding_side}"
)


print("\n[FORMATTER CHECKS]")

for name, passed in formatter_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[OPTIMIZATION TRAIN]")
print(
    f"Rows                   : "
    f"{len(hf_opt_train):,}"
)
print(
    f"Sequence min           : "
    f"{int(train_lengths.min())}"
)
print(
    f"Sequence mean          : "
    f"{train_lengths.mean():.2f}"
)
print(
    f"Sequence p95           : "
    f"{np.quantile(train_lengths, 0.95):.2f}"
)
print(
    f"Sequence max           : "
    f"{int(train_lengths.max())}"
)
print(
    f"Prompt truncated       : "
    f"{train_truncated_n:,}"
)


print("\n[OPTIMIZATION DEV]")
print(
    f"Rows                   : "
    f"{len(hf_opt_dev):,}"
)
print(
    f"Sequence min           : "
    f"{int(dev_lengths.min())}"
)
print(
    f"Sequence mean          : "
    f"{dev_lengths.mean():.2f}"
)
print(
    f"Sequence p95           : "
    f"{np.quantile(dev_lengths, 0.95):.2f}"
)
print(
    f"Sequence max           : "
    f"{int(dev_lengths.max())}"
)
print(
    f"Prompt truncated       : "
    f"{dev_truncated_n:,}"
)


print("\n[ENCODED DATASET CHECKS]")

for name, passed in encoded_dataset_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[SAMPLE ENCODING]")
print(
    f"Prompt tokens          : "
    f"{example_encoded['prompt_token_count']}"
)
print(
    f"Target tokens          : "
    f"{example_encoded['target_token_count']}"
)
print(
    f"Sequence length        : "
    f"{example_encoded['sequence_length']}"
)
print(
    f"Target text repr       : "
    f"{repr(example_encoded['target_text'])}"
)


print("\n" + "-" * 120)

if all_encoding_checks_pass:

    print(
        "DAY 22 CELL 6 STATUS: PASS — "
        "EXACT DAY 15 TASK INTERFACE AND ENCODED DATASETS READY"
    )

else:

    print(
        "DAY 22 CELL 6 STATUS: FAIL — "
        "DO NOT INITIALIZE MODEL"
    )

print("=" * 120)

Encoding Optimization Train:   0%|          | 0/10831 [00:00<?, ? examples/s]

Encoding Optimization Dev:   0%|          | 0/2708 [00:00<?, ? examples/s]

DAY 22 — EXACT TOKENIZER + FORMATTER + ENCODED OPTIMIZATION DATASETS

[TASK INTERFACE]
Instruction            : Classify the sentiment of the following financial news as negative, neutral, or positive.
Prompt repr            : 'Instruction:{instruction}\\nInput:{input}\\nAnswer: '
EOS token              : '<|end_of_text|>'
EOS token ID           : 128001
Pad token ID           : 128001
Training padding       : right

[FORMATTER CHECKS]
PASS   — tokenizer_loaded
PASS   — pad_token_is_eos
PASS   — training_padding_side_right
PASS   — max_seq_length_512
PASS   — valid_labels_exact
PASS   — exact_instruction_text
PASS   — literal_backslash_n_preserved
PASS   — no_actual_newline_in_prompt_template
PASS   — prompt_loss_fully_masked
PASS   — target_loss_unmasked
PASS   — target_contains_eos

[OPTIMIZATION TRAIN]
Rows                   : 10,831
Sequence min           : 32
Sequence mean          : 56.81
Sequence p95           : 95.00
Sequence max           : 267
Prompt truncated       : 0

[OPT

In [7]:
# ==================================================================================================
# DAY 22 — CELL 7
# FRESH 4-BIT META-LLAMA-3-8B + LoRA INITIALIZATION INTEGRITY GATE
# ==================================================================================================

import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# --------------------------------------------------------------------------------------------------
# 1. Clean GPU state
# --------------------------------------------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


# --------------------------------------------------------------------------------------------------
# 2. Frozen config blocks
# --------------------------------------------------------------------------------------------------

base_model_cfg = (
    STAGE_A_CONFIG[
        "base_model"
    ]
)

quant_cfg = (
    STAGE_A_CONFIG[
        "quantization"
    ]
)

lora_cfg = (
    STAGE_A_CONFIG[
        "lora"
    ]
)

train_cfg = (
    STAGE_A_CONFIG[
        "training"
    ]
)


# --------------------------------------------------------------------------------------------------
# 3. 4-bit quantization config
# --------------------------------------------------------------------------------------------------

bnb_config = BitsAndBytesConfig(

    load_in_4bit=
        quant_cfg[
            "load_in_4bit"
        ],

    bnb_4bit_quant_type=
        quant_cfg[
            "bnb_4bit_quant_type"
        ],

    bnb_4bit_use_double_quant=
        quant_cfg[
            "bnb_4bit_use_double_quant"
        ],

    bnb_4bit_compute_dtype=
        torch.float16,
)


# --------------------------------------------------------------------------------------------------
# 4. Load common base model ONLY
#
# IMPORTANT:
#   - no Day 15 adapter loaded;
#   - no previous Day 22 adapter loaded;
#   - fresh LoRA will be initialized below.
# --------------------------------------------------------------------------------------------------

base_model = AutoModelForCausalLM.from_pretrained(

    base_model_cfg[
        "model_name"
    ],

    quantization_config=
        bnb_config,

    device_map=
        base_model_cfg[
            "device_map"
        ],

    trust_remote_code=
        base_model_cfg[
            "trust_remote_code"
        ],
)


# Exact Day 15 training behavior
base_model.config.use_cache = False


# --------------------------------------------------------------------------------------------------
# 5. Prepare quantized model for LoRA training
# --------------------------------------------------------------------------------------------------

base_model = prepare_model_for_kbit_training(

    base_model,

    use_gradient_checkpointing=
        train_cfg[
            "gradient_checkpointing"
        ],
)


# --------------------------------------------------------------------------------------------------
# 6. Fresh LoRA config
# --------------------------------------------------------------------------------------------------

peft_config = LoraConfig(

    r=
        lora_cfg[
            "r"
        ],

    lora_alpha=
        lora_cfg[
            "lora_alpha"
        ],

    lora_dropout=
        lora_cfg[
            "lora_dropout"
        ],

    bias=
        lora_cfg[
            "bias"
        ],

    target_modules=
        lora_cfg[
            "target_modules"
        ],

    task_type=
        lora_cfg[
            "task_type"
        ],

    inference_mode=
        lora_cfg[
            "inference_mode"
        ],
)


# --------------------------------------------------------------------------------------------------
# 7. Attach fresh LoRA
# --------------------------------------------------------------------------------------------------

model = get_peft_model(
    base_model,
    peft_config,
)


# --------------------------------------------------------------------------------------------------
# 8. Parameter audit
# --------------------------------------------------------------------------------------------------

total_params = 0
trainable_params = 0

trainable_parameter_names = []


for name, param in model.named_parameters():

    n = param.numel()

    total_params += n

    if param.requires_grad:

        trainable_params += n
        trainable_parameter_names.append(
            name
        )


trainable_pct = (
    100.0
    * trainable_params
    / total_params
)


unexpected_trainable_names = [

    name
    for name in trainable_parameter_names

    if "lora_" not in name
]


q_proj_lora_present = any(

    "q_proj" in name
    and
    "lora_" in name

    for name
    in trainable_parameter_names
)


v_proj_lora_present = any(

    "v_proj" in name
    and
    "lora_" in name

    for name
    in trainable_parameter_names
)


# --------------------------------------------------------------------------------------------------
# 9. Formal checks
# --------------------------------------------------------------------------------------------------

model_init_checks = {

    "cuda_available":
        torch.cuda.is_available(),

    "model_loaded":
        model is not None,

    "peft_adapter_present":
        hasattr(
            model,
            "peft_config",
        ),

    "fresh_adapter_named_default":
        (
            hasattr(
                model,
                "peft_config",
            )
            and
            "default"
            in model.peft_config
        ),

    "base_model_exact":
        base_model_cfg[
            "model_name"
        ]
        ==
        "meta-llama/Meta-Llama-3-8B",

    "load_in_4bit":
        quant_cfg[
            "load_in_4bit"
        ]
        is True,

    "quant_type_nf4":
        quant_cfg[
            "bnb_4bit_quant_type"
        ]
        ==
        "nf4",

    "double_quant_enabled":
        quant_cfg[
            "bnb_4bit_use_double_quant"
        ]
        is True,

    "compute_dtype_float16":
        quant_cfg[
            "bnb_4bit_compute_dtype"
        ]
        ==
        "float16",

    "lora_r_matches":
        model.peft_config[
            "default"
        ].r
        ==
        8,

    "lora_alpha_matches":
        model.peft_config[
            "default"
        ].lora_alpha
        ==
        32,

    "lora_dropout_matches":
        abs(
            float(
                model.peft_config[
                    "default"
                ].lora_dropout
            )
            - 0.1
        )
        < 1e-12,

    "q_proj_lora_present":
        q_proj_lora_present,

    "v_proj_lora_present":
        v_proj_lora_present,

    "trainable_params_positive":
        trainable_params
        > 0,

    "trainable_params_less_than_total":
        trainable_params
        < total_params,

    "only_lora_parameters_trainable":
        len(
            unexpected_trainable_names
        )
        == 0,

    "use_cache_disabled":
        model.config.use_cache
        is False,
}


all_model_init_checks_pass = bool(
    all(
        model_init_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 10. GPU memory snapshot
# --------------------------------------------------------------------------------------------------

if torch.cuda.is_available():

    gpu_allocated_gb = (
        torch.cuda.memory_allocated()
        / (1024 ** 3)
    )

    gpu_reserved_gb = (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )

    gpu_peak_gb = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

else:

    gpu_allocated_gb = None
    gpu_reserved_gb = None
    gpu_peak_gb = None


# --------------------------------------------------------------------------------------------------
# 11. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — FRESH 4-BIT LoRA INITIALIZATION")
print("=" * 120)


print("\n[BASE MODEL]")
print(
    f"Model                  : "
    f"{base_model_cfg['model_name']}"
)
print(
    "Initialization         : common base model + fresh LoRA"
)
print(
    "Previous adapters      : NOT LOADED"
)


print("\n[QUANTIZATION]")
print(
    f"Load in 4-bit          : "
    f"{quant_cfg['load_in_4bit']}"
)
print(
    f"Quantization type      : "
    f"{quant_cfg['bnb_4bit_quant_type']}"
)
print(
    f"Double quantization    : "
    f"{quant_cfg['bnb_4bit_use_double_quant']}"
)
print(
    f"Compute dtype          : "
    f"{quant_cfg['bnb_4bit_compute_dtype']}"
)


print("\n[LoRA]")
print(
    f"Rank                   : "
    f"{model.peft_config['default'].r}"
)
print(
    f"Alpha                  : "
    f"{model.peft_config['default'].lora_alpha}"
)
print(
    f"Dropout                : "
    f"{model.peft_config['default'].lora_dropout}"
)
print(
    f"Target modules         : "
    f"{lora_cfg['target_modules']}"
)


print("\n[PARAMETERS]")
print(
    f"Trainable parameters   : "
    f"{trainable_params:,}"
)
print(
    f"Total parameters       : "
    f"{total_params:,}"
)
print(
    f"Trainable percentage   : "
    f"{trainable_pct:.6f}%"
)
print(
    f"Trainable tensors      : "
    f"{len(trainable_parameter_names):,}"
)
print(
    f"Unexpected trainable   : "
    f"{len(unexpected_trainable_names):,}"
)


print("\n[INTEGRITY CHECKS]")

for name, passed in model_init_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[GPU MEMORY]")

if torch.cuda.is_available():

    print(
        f"Allocated              : "
        f"{gpu_allocated_gb:.2f} GB"
    )

    print(
        f"Reserved               : "
        f"{gpu_reserved_gb:.2f} GB"
    )

    print(
        f"Peak allocated         : "
        f"{gpu_peak_gb:.2f} GB"
    )


print("\n" + "-" * 120)

if all_model_init_checks_pass:

    print(
        "DAY 22 CELL 7 STATUS: PASS — "
        "FRESH 4-BIT BASE MODEL + LoRA READY"
    )

else:

    print(
        "DAY 22 CELL 7 STATUS: FAIL — "
        "DO NOT START TRAINER SETUP"
    )

print("=" * 120)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

DAY 22 — FRESH 4-BIT LoRA INITIALIZATION

[BASE MODEL]
Model                  : meta-llama/Meta-Llama-3-8B
Initialization         : common base model + fresh LoRA
Previous adapters      : NOT LOADED

[QUANTIZATION]
Load in 4-bit          : True
Quantization type      : nf4
Double quantization    : True
Compute dtype          : float16

[LoRA]
Rank                   : 8
Alpha                  : 32
Dropout                : 0.1
Target modules         : ['q_proj', 'v_proj']

[PARAMETERS]
Trainable parameters   : 3,407,872
Total parameters       : 4,544,008,192
Trainable percentage   : 0.074997%
Trainable tensors      : 128
Unexpected trainable   : 0

[INTEGRITY CHECKS]
PASS   — cuda_available
PASS   — model_loaded
PASS   — peft_adapter_present
PASS   — fresh_adapter_named_default
PASS   — base_model_exact
PASS   — load_in_4bit
PASS   — quant_type_nf4
PASS   — double_quant_enabled
PASS   — compute_dtype_float16
PASS   — lora_r_matches
PASS   — lora_alpha_matches
PASS   — lora_dropout_matche

In [8]:
# ==================================================================================================
# DAY 22 — CELL 8
# DYNAMIC PADDING COLLATOR + TRAINING ARGUMENTS + TRAINER INFRASTRUCTURE
# ==================================================================================================

from dataclasses import dataclass

import torch

from transformers import (
    Trainer,
    TrainingArguments,
)


# --------------------------------------------------------------------------------------------------
# 1. Remove metadata columns before Trainer
#
# Trainer/model should receive only:
#   - input_ids
#   - attention_mask
#   - labels
#
# Diagnostic / observation metadata remain preserved in hf_opt_train / hf_opt_dev,
# but are removed from the training views.
# --------------------------------------------------------------------------------------------------

MODEL_INPUT_COLUMNS = [
    "input_ids",
    "attention_mask",
    "labels",
]


hf_opt_train_model = (
    hf_opt_train
    .select_columns(
        MODEL_INPUT_COLUMNS
    )
)


hf_opt_dev_model = (
    hf_opt_dev
    .select_columns(
        MODEL_INPUT_COLUMNS
    )
)


# --------------------------------------------------------------------------------------------------
# 2. Exact target-aware dynamic padding collator
#
# Required behavior:
#
#   input_ids:
#       right-pad with tokenizer.pad_token_id (= EOS token ID)
#
#   attention_mask:
#       right-pad with 0
#
#   labels:
#       right-pad with -100
#
# This preserves target-only loss masking.
# --------------------------------------------------------------------------------------------------

@dataclass
class TargetOnlyDynamicPaddingCollator:

    pad_token_id: int

    label_pad_token_id: int = -100

    def __call__(
        self,
        features,
    ):

        max_length = max(
            len(
                feature[
                    "input_ids"
                ]
            )
            for feature
            in features
        )

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for feature in features:

            input_ids = list(
                feature[
                    "input_ids"
                ]
            )

            attention_mask = list(
                feature[
                    "attention_mask"
                ]
            )

            labels = list(
                feature[
                    "labels"
                ]
            )

            pad_length = (
                max_length
                - len(
                    input_ids
                )
            )

            batch_input_ids.append(
                input_ids
                +
                [
                    self.pad_token_id
                ]
                * pad_length
            )

            batch_attention_mask.append(
                attention_mask
                +
                [0]
                * pad_length
            )

            batch_labels.append(
                labels
                +
                [
                    self.label_pad_token_id
                ]
                * pad_length
            )

        return {

            "input_ids":
                torch.tensor(
                    batch_input_ids,
                    dtype=torch.long,
                ),

            "attention_mask":
                torch.tensor(
                    batch_attention_mask,
                    dtype=torch.long,
                ),

            "labels":
                torch.tensor(
                    batch_labels,
                    dtype=torch.long,
                ),
        }


data_collator = (
    TargetOnlyDynamicPaddingCollator(
        pad_token_id=
            tokenizer.pad_token_id,

        label_pad_token_id=
            -100,
    )
)


# --------------------------------------------------------------------------------------------------
# 3. Collator dry-run audit
# --------------------------------------------------------------------------------------------------

dry_run_features = [
    hf_opt_train_model[
        0
    ],
    hf_opt_train_model[
        1
    ],
    hf_opt_train_model[
        2
    ],
]


dry_run_batch = (
    data_collator(
        dry_run_features
    )
)


dry_run_max_original_length = max(
    len(
        feature[
            "input_ids"
        ]
    )
    for feature
    in dry_run_features
)


dry_run_checks = {

    "batch_input_ids_tensor":
        isinstance(
            dry_run_batch[
                "input_ids"
            ],
            torch.Tensor,
        ),

    "batch_attention_mask_tensor":
        isinstance(
            dry_run_batch[
                "attention_mask"
            ],
            torch.Tensor,
        ),

    "batch_labels_tensor":
        isinstance(
            dry_run_batch[
                "labels"
            ],
            torch.Tensor,
        ),

    "batch_size_3":
        dry_run_batch[
            "input_ids"
        ].shape[0]
        == 3,

    "dynamic_length_correct":
        dry_run_batch[
            "input_ids"
        ].shape[1]
        ==
        dry_run_max_original_length,

    "all_batch_shapes_match":
        (
            dry_run_batch[
                "input_ids"
            ].shape
            ==
            dry_run_batch[
                "attention_mask"
            ].shape
            ==
            dry_run_batch[
                "labels"
            ].shape
        ),

    "label_padding_uses_minus100":
        bool(
            (
                dry_run_batch[
                    "labels"
                ]
                ==
                -100
            ).any()
        ),

    "attention_padding_uses_zero":
        bool(
            (
                dry_run_batch[
                    "attention_mask"
                ]
                ==
                0
            ).any()
        ),

    "pad_token_is_eos":
        tokenizer.pad_token_id
        ==
        tokenizer.eos_token_id,
}


# --------------------------------------------------------------------------------------------------
# 4. Stage A TrainingArguments
#
# No training is executed in this cell.
# --------------------------------------------------------------------------------------------------
training_args = TrainingArguments(

    output_dir=
        str(
            STAGE_A_CHECKPOINT_DIR
        ),

    # Training budget
    max_steps=
        STAGE_A_CONFIG[
            "training"
        ][
            "max_steps"
        ],

    # Batch
    per_device_train_batch_size=
        STAGE_A_CONFIG[
            "training"
        ][
            "per_device_train_batch_size"
        ],

    per_device_eval_batch_size=
        STAGE_A_CONFIG[
            "training"
        ][
            "per_device_eval_batch_size"
        ],

    gradient_accumulation_steps=
        STAGE_A_CONFIG[
            "training"
        ][
            "gradient_accumulation_steps"
        ],

    # Optimizer
    learning_rate=
        STAGE_A_CONFIG[
            "training"
        ][
            "learning_rate"
        ],

    weight_decay=
        STAGE_A_CONFIG[
            "training"
        ][
            "weight_decay"
        ],

    warmup_steps=
        STAGE_A_CONFIG[
            "training"
        ][
            "warmup_steps"
        ],

    # Logging
    logging_strategy=
        "steps",

    logging_steps=
        STAGE_A_CONFIG[
            "training"
        ][
            "logging_steps"
        ],

    # Evaluation
    eval_strategy=
        "steps",

    eval_steps=
        STAGE_A_CONFIG[
            "training"
        ][
            "eval_steps"
        ],

    # Saving
    save_strategy=
        "steps",

    save_steps=
        STAGE_A_CONFIG[
            "training"
        ][
            "save_steps"
        ],

    save_total_limit=
        4,

    # Important:
    # checkpoint selection happens AFTER training from
    # candidate checkpoints [50,100,150,200].
    load_best_model_at_end=
        False,

    # Precision
    fp16=
        True,

    bf16=
        False,

    # Reproducibility
    seed=
        STAGE_A_CONFIG[
            "training"
        ][
            "seed"
        ],

    data_seed=
        STAGE_A_CONFIG[
            "training"
        ][
            "data_seed"
        ],

    # Gradient checkpointing already prepared in the model
    gradient_checkpointing=
        STAGE_A_CONFIG[
            "training"
        ][
            "gradient_checkpointing"
        ],

    # Avoid external experiment tracking
    report_to=
        [],

    # Dataset/model interface
    remove_unused_columns=
        True,

    label_names=
        [
            "labels"
        ],

    # Keep logs deterministic / simple
    disable_tqdm=
        False,
)


# --------------------------------------------------------------------------------------------------
# 5. Instantiate Trainer
#
# Still NO training here.
# --------------------------------------------------------------------------------------------------

trainer = Trainer(

    model=
        model,

    args=
        training_args,

    train_dataset=
        hf_opt_train_model,

    eval_dataset=
        hf_opt_dev_model,

    data_collator=
        data_collator,

    processing_class=
        tokenizer,
)


# --------------------------------------------------------------------------------------------------
# 6. Trainer / TrainingArguments integrity checks
# --------------------------------------------------------------------------------------------------

trainer_checks = {

    "trainer_created":
        trainer is not None,

    "train_dataset_n_10831":
        len(
            trainer.train_dataset
        )
        == 10_831,

    "eval_dataset_n_2708":
        len(
            trainer.eval_dataset
        )
        == 2_708,

    "max_steps_200":
        training_args.max_steps
        == 200,

    "train_batch_size_1":
        training_args.per_device_train_batch_size
        == 1,

    "eval_batch_size_1":
        training_args.per_device_eval_batch_size
        == 1,

    "gradient_accumulation_8":
        training_args.gradient_accumulation_steps
        == 8,

    "learning_rate_1e4":
        abs(
            training_args.learning_rate
            - 1e-4
        )
        < 1e-12,

    "warmup_steps_10":
        training_args.warmup_steps
        == 10,

    "eval_every_25":
        training_args.eval_steps
        == 25,

    "save_every_50":
        training_args.save_steps
        == 50,

    "logging_every_10":
        training_args.logging_steps
        == 10,

    "seed_42":
        training_args.seed
        == 42,

    "data_seed_42":
        training_args.data_seed
        == 42,

    "fp16_enabled":
        training_args.fp16
        is True,

    "best_model_not_auto_loaded":
        training_args.load_best_model_at_end
        is False,

    "external_reporting_disabled":
        len(
            training_args.report_to
        )
        == 0,

    "model_still_trainable":
        any(
            param.requires_grad
            for param
            in trainer.model.parameters()
        ),
}


all_collator_checks_pass = bool(
    all(
        dry_run_checks.values()
    )
)


all_trainer_checks_pass = bool(
    all(
        trainer_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE A TRAINER INFRASTRUCTURE")
print("=" * 120)


print("\n[DYNAMIC COLLATOR]")
print(
    f"Dry-run batch shape      : "
    f"{tuple(dry_run_batch['input_ids'].shape)}"
)
print(
    f"Longest sample length    : "
    f"{dry_run_max_original_length}"
)
print(
    f"Input padding ID         : "
    f"{tokenizer.pad_token_id}"
)
print(
    "Attention padding       : 0"
)
print(
    "Label padding           : -100"
)


print("\n[COLLATOR CHECKS]")

for name, passed in dry_run_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[TRAINING ARGUMENTS]")
print(
    f"Max steps                : "
    f"{training_args.max_steps}"
)
print(
    f"Train batch              : "
    f"{training_args.per_device_train_batch_size}"
)
print(
    f"Eval batch               : "
    f"{training_args.per_device_eval_batch_size}"
)
print(
    f"Gradient accumulation    : "
    f"{training_args.gradient_accumulation_steps}"
)
print(
    f"Effective train batch    : "
    f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}"
)
print(
    f"Learning rate            : "
    f"{training_args.learning_rate}"
)
print(
    f"Warmup steps             : "
    f"{training_args.warmup_steps}"
)
print(
    f"Eval interval            : "
    f"{training_args.eval_steps}"
)
print(
    f"Save interval            : "
    f"{training_args.save_steps}"
)
print(
    f"Logging interval         : "
    f"{training_args.logging_steps}"
)
print(
    f"Seed                     : "
    f"{training_args.seed}"
)


print("\n[TRAINER CHECKS]")

for name, passed in trainer_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n" + "-" * 120)

if (
    all_collator_checks_pass
    and
    all_trainer_checks_pass
):

    print(
        "DAY 22 CELL 8 STATUS: PASS — "
        "STAGE A TRAINER READY; TRAINING NOT YET STARTED"
    )

else:

    print(
        "DAY 22 CELL 8 STATUS: FAIL — "
        "DO NOT START TRAINING"
    )

print("=" * 120)

DAY 22 — STAGE A TRAINER INFRASTRUCTURE

[DYNAMIC COLLATOR]
Dry-run batch shape      : (3, 52)
Longest sample length    : 52
Input padding ID         : 128001
Attention padding       : 0
Label padding           : -100

[COLLATOR CHECKS]
PASS   — batch_input_ids_tensor
PASS   — batch_attention_mask_tensor
PASS   — batch_labels_tensor
PASS   — batch_size_3
PASS   — dynamic_length_correct
PASS   — all_batch_shapes_match
PASS   — label_padding_uses_minus100
PASS   — attention_padding_uses_zero
PASS   — pad_token_is_eos

[TRAINING ARGUMENTS]
Max steps                : 200
Train batch              : 1
Eval batch               : 1
Gradient accumulation    : 8
Effective train batch    : 8
Learning rate            : 0.0001
Warmup steps             : 10
Eval interval            : 25
Save interval            : 50
Logging interval         : 10
Seed                     : 42

[TRAINER CHECKS]
PASS   — trainer_created
PASS   — train_dataset_n_10831
PASS   — eval_dataset_n_2708
PASS   — max_steps_200


In [9]:
# ==================================================================================================
# DAY 22 — CELL 9
# STAGE A BASELINE TRAINING — 200 STEPS
# ==================================================================================================

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch

from transformers import set_seed


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_A_TRAINING_LOG_PATH = (
    STAGE_A_ARTIFACT_DIR
    / "stage_a_training_log.csv"
)

STAGE_A_TRAINING_SUMMARY_PATH = (
    STAGE_A_ARTIFACT_DIR
    / "stage_a_training_summary.json"
)

STAGE_A_TRAINING_COMPLETE_MARKER = (
    STAGE_A_ARTIFACT_DIR
    / "stage_a_training_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate
#
# Prevent accidental second training on an already-trained in-memory model.
# --------------------------------------------------------------------------------------------------

if STAGE_A_TRAINING_COMPLETE_MARKER.exists():

    raise RuntimeError(
        "DAY 22 STOP: Stage A training-complete marker already exists. "
        "Do not accidentally retrain the same Stage A run."
    )


# --------------------------------------------------------------------------------------------------
# 3. Confirm no pre-existing Stage A checkpoints
# --------------------------------------------------------------------------------------------------

existing_checkpoint_dirs = sorted(
    [
        p
        for p in STAGE_A_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(existing_checkpoint_dirs) == 0, (
    "DAY 22 STOP: pre-existing Stage A checkpoint directories detected. "
    "Stage A must begin from a clean output directory."
)


# --------------------------------------------------------------------------------------------------
# 4. Pre-training integrity gate
# --------------------------------------------------------------------------------------------------

pretraining_checks = {

    "trainer_exists":
        trainer is not None,

    "trainer_global_step_zero":
        int(
            trainer.state.global_step
        )
        == 0,

    "model_trainable":
        any(
            p.requires_grad
            for p in trainer.model.parameters()
        ),

    "train_n_10831":
        len(
            trainer.train_dataset
        )
        == 10_831,

    "dev_n_2708":
        len(
            trainer.eval_dataset
        )
        == 2_708,

    "max_steps_200":
        trainer.args.max_steps
        == 200,

    "save_steps_50":
        trainer.args.save_steps
        == 50,

    "eval_steps_25":
        trainer.args.eval_steps
        == 25,

    "seed_42":
        trainer.args.seed
        == 42,

    "no_existing_checkpoints":
        len(
            existing_checkpoint_dirs
        )
        == 0,
}


assert all(
    pretraining_checks.values()
), (
    "DAY 22 STOP: Stage A pre-training integrity gate failed."
)


# --------------------------------------------------------------------------------------------------
# 5. Re-establish deterministic seed immediately before training
# --------------------------------------------------------------------------------------------------

set_seed(
    STAGE_A_CONFIG[
        "training"
    ][
        "seed"
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. Reset GPU peak-memory accounting
# --------------------------------------------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.reset_peak_memory_stats()


# --------------------------------------------------------------------------------------------------
# 7. Train Stage A baseline
#
# IMPORTANT:
#   - fresh LoRA
#   - natural sampling
#   - Optimization Train only
#   - Optimization Dev for eval_loss
#   - no Historical Validation
#   - no Future F1–F6
# --------------------------------------------------------------------------------------------------

training_start_time = time.time()


train_result = trainer.train()


training_elapsed_seconds = (
    time.time()
    - training_start_time
)


# --------------------------------------------------------------------------------------------------
# 8. Extract final training state
# --------------------------------------------------------------------------------------------------

final_global_step = int(
    trainer.state.global_step
)

final_train_loss = float(
    train_result.training_loss
)


if torch.cuda.is_available():

    peak_gpu_allocation_gb = float(
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

else:

    peak_gpu_allocation_gb = None


# --------------------------------------------------------------------------------------------------
# 9. Save complete Trainer log history
# --------------------------------------------------------------------------------------------------

training_log_df = pd.DataFrame(
    trainer.state.log_history
)


training_log_df.to_csv(
    STAGE_A_TRAINING_LOG_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 10. Identify saved candidate checkpoints
# --------------------------------------------------------------------------------------------------

EXPECTED_CHECKPOINT_STEPS = [
    50,
    100,
    150,
    200,
]


checkpoint_status = {}


for step in EXPECTED_CHECKPOINT_STEPS:

    checkpoint_path = (
        STAGE_A_CHECKPOINT_DIR
        / f"checkpoint-{step}"
    )

    checkpoint_status[
        step
    ] = {
        "path":
            str(
                checkpoint_path
            ),

        "exists":
            bool(
                checkpoint_path.exists()
            ),

        "adapter_config_exists":
            bool(
                (
                    checkpoint_path
                    / "adapter_config.json"
                ).exists()
            ),

        "adapter_weights_exist":
            bool(
                (
                    checkpoint_path
                    / "adapter_model.safetensors"
                ).exists()
                or
                (
                    checkpoint_path
                    / "adapter_model.bin"
                ).exists()
            ),
    }


# --------------------------------------------------------------------------------------------------
# 11. Extract evaluation log entries
# --------------------------------------------------------------------------------------------------

eval_log_df = (
    training_log_df.loc[
        training_log_df[
            "eval_loss"
        ].notna()
    ].copy()
    if "eval_loss"
    in training_log_df.columns
    else pd.DataFrame()
)


observed_eval_steps = (
    sorted(
        eval_log_df[
            "step"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if not eval_log_df.empty
    else []
)


# --------------------------------------------------------------------------------------------------
# 12. Formal post-training checks
# --------------------------------------------------------------------------------------------------

posttraining_checks = {

    "training_completed_200_steps":
        final_global_step
        == 200,

    "training_loss_finite":
        bool(
            pd.notna(
                final_train_loss
            )
        ),

    "training_log_nonempty":
        len(
            training_log_df
        )
        > 0,

    "evaluation_logs_present":
        not eval_log_df.empty,

    "eval_step_25_present":
        25
        in observed_eval_steps,

    "eval_step_50_present":
        50
        in observed_eval_steps,

    "eval_step_100_present":
        100
        in observed_eval_steps,

    "eval_step_150_present":
        150
        in observed_eval_steps,

    "eval_step_200_present":
        200
        in observed_eval_steps,

    "checkpoint_50_complete":
        (
            checkpoint_status[
                50
            ][
                "exists"
            ]
            and
            checkpoint_status[
                50
            ][
                "adapter_config_exists"
            ]
            and
            checkpoint_status[
                50
            ][
                "adapter_weights_exist"
            ]
        ),

    "checkpoint_100_complete":
        (
            checkpoint_status[
                100
            ][
                "exists"
            ]
            and
            checkpoint_status[
                100
            ][
                "adapter_config_exists"
            ]
            and
            checkpoint_status[
                100
            ][
                "adapter_weights_exist"
            ]
        ),

    "checkpoint_150_complete":
        (
            checkpoint_status[
                150
            ][
                "exists"
            ]
            and
            checkpoint_status[
                150
            ][
                "adapter_config_exists"
            ]
            and
            checkpoint_status[
                150
            ][
                "adapter_weights_exist"
            ]
        ),

    "checkpoint_200_complete":
        (
            checkpoint_status[
                200
            ][
                "exists"
            ]
            and
            checkpoint_status[
                200
            ][
                "adapter_config_exists"
            ]
            and
            checkpoint_status[
                200
            ][
                "adapter_weights_exist"
            ]
        ),

    "historical_validation_not_used":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used_for_selection"
        ]
        is False,

    "future_data_not_used":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


all_posttraining_checks_pass = bool(
    all(
        posttraining_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Save formal training summary
# --------------------------------------------------------------------------------------------------

stage_a_training_summary = {

    "day":
        22,

    "stage":
        "A",

    "artifact":
        "stage_a_training_summary",

    "training_status":
        (
            "PASS"
            if all_posttraining_checks_pass
            else "FAIL"
        ),

    "population": {

        "optimization_train_n":
            int(
                len(
                    trainer.train_dataset
                )
            ),

        "optimization_dev_n":
            int(
                len(
                    trainer.eval_dataset
                )
            ),

        "historical_validation_used":
            False,

        "future_f1_f6_used":
            False,
    },

    "training": {

        "final_global_step":
            final_global_step,

        "final_train_loss":
            final_train_loss,

        "runtime_seconds":
            float(
                training_elapsed_seconds
            ),

        "runtime_minutes":
            float(
                training_elapsed_seconds
                / 60
            ),

        "peak_gpu_allocation_gb":
            peak_gpu_allocation_gb,

        "seed":
            int(
                trainer.args.seed
            ),
    },

    "evaluation": {

        "observed_eval_steps":
            observed_eval_steps,
    },

    "checkpoints":
        checkpoint_status,

    "checks": {
        key:
            bool(
                value
            )
        for key, value
        in posttraining_checks.items()
    },

    "all_checks_pass":
        all_posttraining_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_A_TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_a_training_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 14. Write training-complete marker only if everything passed
# --------------------------------------------------------------------------------------------------

if all_posttraining_checks_pass:

    with open(
        STAGE_A_TRAINING_COMPLETE_MARKER,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            {
                "day": 22,
                "stage": "A",
                "status":
                    "TRAINING_COMPLETE",
                "global_step":
                    final_global_step,
                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            },
            f,
            indent=2,
        )


# --------------------------------------------------------------------------------------------------
# 15. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE A BASELINE TRAINING")
print("=" * 120)


print("\n[TRAINING]")
print(
    f"Final global step       : "
    f"{final_global_step}"
)
print(
    f"Final train loss        : "
    f"{final_train_loss:.6f}"
)
print(
    f"Runtime                 : "
    f"{training_elapsed_seconds / 60:.2f} minutes"
)

if peak_gpu_allocation_gb is not None:

    print(
        f"Peak GPU allocation     : "
        f"{peak_gpu_allocation_gb:.2f} GB"
    )


print("\n[EVALUATION LOG]")
print(
    f"Observed eval steps     : "
    f"{observed_eval_steps}"
)


if not eval_log_df.empty:

    eval_display_columns = [
        col
        for col in [
            "step",
            "eval_loss",
            "eval_runtime",
        ]
        if col
        in eval_log_df.columns
    ]

    display(
        eval_log_df[
            eval_display_columns
        ].reset_index(
            drop=True
        )
    )


print("\n[CHECKPOINTS]")

for step in EXPECTED_CHECKPOINT_STEPS:

    status = (
        checkpoint_status[
            step
        ]
    )

    complete = (
        status[
            "exists"
        ]
        and
        status[
            "adapter_config_exists"
        ]
        and
        status[
            "adapter_weights_exist"
        ]
    )

    print(
        f"{'PASS' if complete else 'FAIL':<6} — "
        f"checkpoint-{step}"
    )


print("\n[POST-TRAINING CHECKS]")

for name, passed in posttraining_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Training log            : "
    f"{STAGE_A_TRAINING_LOG_PATH}"
)
print(
    f"Training summary        : "
    f"{STAGE_A_TRAINING_SUMMARY_PATH}"
)
print(
    f"Checkpoint directory    : "
    f"{STAGE_A_CHECKPOINT_DIR}"
)


print("\n" + "-" * 120)

if all_posttraining_checks_pass:

    print(
        "DAY 22 CELL 9 STATUS: PASS — "
        "STAGE A BASELINE TRAINING COMPLETE; "
        "CHECKPOINT SELECTION PENDING"
    )

else:

    print(
        "DAY 22 CELL 9 STATUS: FAIL — "
        "DO NOT SELECT CHECKPOINT"
    )

print("=" * 120)

RuntimeError: DAY 22 STOP: Stage A training-complete marker already exists. Do not accidentally retrain the same Stage A run.

In [10]:
# ==================================================================================================
# DAY 22 — CELL 10
# STAGE A CHECKPOINT SELECTION + BASELINE FREEZE
# ==================================================================================================

import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_A_CHECKPOINT_SELECTION_PATH = (
    STAGE_A_ARTIFACT_DIR
    / "stage_a_checkpoint_selection.csv"
)

STAGE_A_SELECTED_MODEL_DIR = (
    STAGE_A_MODEL_DIR
    / "selected_adapter"
)

STAGE_A_SELECTION_SUMMARY_PATH = (
    STAGE_A_ARTIFACT_DIR
    / "stage_a_checkpoint_selection_summary.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Load training log
# --------------------------------------------------------------------------------------------------

training_log_df = pd.read_csv(
    STAGE_A_TRAINING_LOG_PATH
)


assert "step" in training_log_df.columns
assert "eval_loss" in training_log_df.columns


# --------------------------------------------------------------------------------------------------
# 3. Frozen formal candidate checkpoints
# --------------------------------------------------------------------------------------------------

CANDIDATE_STEPS = (
    STAGE_A_CONFIG[
        "checkpoint_selection"
    ][
        "candidate_steps"
    ]
)


candidate_eval_rows = (
    training_log_df.loc[
        training_log_df[
            "step"
        ].isin(
            CANDIDATE_STEPS
        )
        &
        training_log_df[
            "eval_loss"
        ].notna(),
        [
            "step",
            "eval_loss",
        ],
    ]
    .copy()
    .sort_values(
        "step"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Require exactly one eval loss for each formal candidate step
# --------------------------------------------------------------------------------------------------

candidate_counts = (
    candidate_eval_rows[
        "step"
    ]
    .astype(int)
    .value_counts()
)


candidate_integrity_checks = {

    "candidate_row_n_4":
        len(
            candidate_eval_rows
        )
        == 4,

    "candidate_steps_exact":
        sorted(
            candidate_eval_rows[
                "step"
            ]
            .astype(int)
            .tolist()
        )
        ==
        [
            50,
            100,
            150,
            200,
        ],

    "one_eval_per_candidate":
        bool(
            (
                candidate_counts
                == 1
            ).all()
        ),

    "candidate_eval_losses_finite":
        bool(
            candidate_eval_rows[
                "eval_loss"
            ]
            .notna()
            .all()
        ),
}


assert all(
    candidate_integrity_checks.values()
), (
    "DAY 22 STOP: formal candidate checkpoint table failed integrity checks."
)


# --------------------------------------------------------------------------------------------------
# 5. Formal checkpoint selection
#
# Frozen rule:
#   choose candidate checkpoint with minimum Optimization Dev eval_loss
#
# Tie-break:
#   smaller step
# --------------------------------------------------------------------------------------------------

candidate_eval_rows = (
    candidate_eval_rows
    .sort_values(
        [
            "eval_loss",
            "step",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


candidate_eval_rows[
    "rank_by_eval_loss"
] = (
    range(
        1,
        len(
            candidate_eval_rows
        )
        + 1
    )
)


selected_row = (
    candidate_eval_rows
    .iloc[0]
)


selected_step = int(
    selected_row[
        "step"
    ]
)

selected_eval_loss = float(
    selected_row[
        "eval_loss"
    ]
)


candidate_eval_rows[
    "selected"
] = (
    candidate_eval_rows[
        "step"
    ].astype(int)
    ==
    selected_step
)


# --------------------------------------------------------------------------------------------------
# 6. Recover chronological candidate table for reporting
# --------------------------------------------------------------------------------------------------

candidate_eval_table = (
    candidate_eval_rows
    .sort_values(
        "step"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Boundary adequacy diagnostics
#
# This does NOT change Stage A selection.
# It determines whether Stage B horizon extension is scientifically justified.
# --------------------------------------------------------------------------------------------------

eval_150 = float(
    candidate_eval_table.loc[
        candidate_eval_table[
            "step"
        ]
        == 150,
        "eval_loss",
    ].iloc[0]
)


eval_200 = float(
    candidate_eval_table.loc[
        candidate_eval_table[
            "step"
        ]
        == 200,
        "eval_loss",
    ].iloc[0]
)


loss_improvement_150_to_200 = (
    eval_150
    - eval_200
)


relative_improvement_150_to_200 = (
    loss_improvement_150_to_200
    / eval_150
)


selected_at_upper_boundary = (
    selected_step
    ==
    max(
        CANDIDATE_STEPS
    )
)


stage_b_horizon_audit_required = bool(
    selected_at_upper_boundary
    and
    loss_improvement_150_to_200
    > 0
)


# --------------------------------------------------------------------------------------------------
# 8. Validate selected checkpoint directory
# --------------------------------------------------------------------------------------------------

selected_checkpoint_dir = (
    STAGE_A_CHECKPOINT_DIR
    / f"checkpoint-{selected_step}"
)


selected_checkpoint_checks = {

    "selected_checkpoint_exists":
        selected_checkpoint_dir.exists(),

    "selected_adapter_config_exists":
        (
            selected_checkpoint_dir
            / "adapter_config.json"
        ).exists(),

    "selected_adapter_weights_exist":
        (
            (
                selected_checkpoint_dir
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                selected_checkpoint_dir
                / "adapter_model.bin"
            ).exists()
        ),
}


assert all(
    selected_checkpoint_checks.values()
), (
    "DAY 22 STOP: selected checkpoint is incomplete."
)


# --------------------------------------------------------------------------------------------------
# 9. Freeze selected adapter
#
# Copy selected checkpoint into a stable final Stage A directory.
# --------------------------------------------------------------------------------------------------

if STAGE_A_SELECTED_MODEL_DIR.exists():

    shutil.rmtree(
        STAGE_A_SELECTED_MODEL_DIR
    )


shutil.copytree(
    selected_checkpoint_dir,
    STAGE_A_SELECTED_MODEL_DIR,
)


# --------------------------------------------------------------------------------------------------
# 10. Verify frozen selected adapter
# --------------------------------------------------------------------------------------------------

frozen_adapter_checks = {

    "selected_adapter_dir_exists":
        STAGE_A_SELECTED_MODEL_DIR.exists(),

    "frozen_adapter_config_exists":
        (
            STAGE_A_SELECTED_MODEL_DIR
            / "adapter_config.json"
        ).exists(),

    "frozen_adapter_weights_exist":
        (
            (
                STAGE_A_SELECTED_MODEL_DIR
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                STAGE_A_SELECTED_MODEL_DIR
                / "adapter_model.bin"
            ).exists()
        ),
}


# --------------------------------------------------------------------------------------------------
# 11. Formal selection checks
# --------------------------------------------------------------------------------------------------

selection_checks = {

    **candidate_integrity_checks,

    **selected_checkpoint_checks,

    **frozen_adapter_checks,

    "selection_metric_eval_loss":
        STAGE_A_CONFIG[
            "checkpoint_selection"
        ][
            "selection_metric"
        ]
        ==
        "eval_loss",

    "selection_direction_minimize":
        STAGE_A_CONFIG[
            "checkpoint_selection"
        ][
            "selection_direction"
        ]
        ==
        "minimize",

    "selected_step_is_argmin":
        selected_step
        ==
        int(
            candidate_eval_table.loc[
                candidate_eval_table[
                    "eval_loss"
                ].idxmin(),
                "step",
            ]
        ),

    "selected_step_200":
        selected_step
        == 200,

    "selected_at_upper_boundary":
        selected_at_upper_boundary,

    "loss_improved_150_to_200":
        loss_improvement_150_to_200
        > 0,

    "historical_validation_not_used":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used_for_selection"
        ]
        is False,

    "future_not_used":
        STAGE_A_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


all_selection_checks_pass = bool(
    all(
        selection_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Save checkpoint selection table
# --------------------------------------------------------------------------------------------------

candidate_eval_table.to_csv(
    STAGE_A_CHECKPOINT_SELECTION_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 13. Save formal Stage A selection summary
# --------------------------------------------------------------------------------------------------

stage_a_selection_summary = {

    "day":
        22,

    "stage":
        "A",

    "artifact":
        "stage_a_checkpoint_selection_summary",

    "selection_protocol": {

        "candidate_steps":
            [
                int(x)
                for x
                in CANDIDATE_STEPS
            ],

        "selection_split":
            "optimization_dev",

        "metric":
            "eval_loss",

        "direction":
            "minimize",

        "tie_break":
            "smaller_step",
    },

    "selected_checkpoint": {

        "step":
            selected_step,

        "eval_loss":
            selected_eval_loss,

        "source_directory":
            str(
                selected_checkpoint_dir
            ),

        "frozen_adapter_directory":
            str(
                STAGE_A_SELECTED_MODEL_DIR
            ),
    },

    "training_adequacy_signal": {

        "selected_at_upper_boundary":
            selected_at_upper_boundary,

        "eval_loss_step_150":
            eval_150,

        "eval_loss_step_200":
            eval_200,

        "absolute_improvement_150_to_200":
            float(
                loss_improvement_150_to_200
            ),

        "relative_improvement_150_to_200":
            float(
                relative_improvement_150_to_200
            ),

        "stage_b_horizon_audit_required":
            stage_b_horizon_audit_required,

        "interpretation":
            (
                "The formal Stage A optimum occurs at the maximum allowed "
                "training horizon and validation loss continues to improve "
                "from step 150 to step 200. The 200-step budget therefore "
                "does not establish convergence; Stage B horizon extension "
                "is required."
                if stage_b_horizon_audit_required
                else
                "No clear upper-bound training adequacy signal detected."
            ),
    },

    "firewall": {

        "historical_validation_used":
            False,

        "future_f1_f6_used":
            False,
    },

    "checks": {
        key:
            bool(
                value
            )
        for key, value
        in selection_checks.items()
    },

    "all_checks_pass":
        all_selection_checks_pass,

    "freeze_status":
        (
            "STAGE_A_BASELINE_FROZEN"
            if all_selection_checks_pass
            else
            "FREEZE_FAILED"
        ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_A_SELECTION_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_a_selection_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 14. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE A CHECKPOINT SELECTION + BASELINE FREEZE")
print("=" * 120)


print("\n[CANDIDATE CHECKPOINTS]")

display(
    candidate_eval_table[
        [
            "step",
            "eval_loss",
            "rank_by_eval_loss",
            "selected",
        ]
    ]
)


print("\n[SELECTED CHECKPOINT]")
print(
    f"Selected step           : "
    f"{selected_step}"
)
print(
    f"Selected eval loss      : "
    f"{selected_eval_loss:.6f}"
)
print(
    f"Frozen adapter          : "
    f"{STAGE_A_SELECTED_MODEL_DIR}"
)


print("\n[TRAINING ADEQUACY SIGNAL]")
print(
    f"Eval loss @150          : "
    f"{eval_150:.6f}"
)
print(
    f"Eval loss @200          : "
    f"{eval_200:.6f}"
)
print(
    f"Absolute improvement    : "
    f"{loss_improvement_150_to_200:.6f}"
)
print(
    f"Relative improvement    : "
    f"{relative_improvement_150_to_200:.2%}"
)
print(
    f"Upper-bound optimum     : "
    f"{selected_at_upper_boundary}"
)
print(
    f"Stage B required        : "
    f"{stage_b_horizon_audit_required}"
)


print("\n[SELECTION CHECKS]")

for name, passed in selection_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Selection table         : "
    f"{STAGE_A_CHECKPOINT_SELECTION_PATH}"
)
print(
    f"Selection summary       : "
    f"{STAGE_A_SELECTION_SUMMARY_PATH}"
)
print(
    f"Frozen adapter          : "
    f"{STAGE_A_SELECTED_MODEL_DIR}"
)


print("\n" + "-" * 120)

if all_selection_checks_pass:

    print(
        "DAY 22 CELL 10 STATUS: PASS — "
        "STAGE A BASELINE FROZEN; "
        "STAGE B HORIZON AUDIT REQUIRED"
    )

else:

    print(
        "DAY 22 CELL 10 STATUS: FAIL — "
        "DO NOT PROCEED TO STAGE B"
    )

print("=" * 120)

DAY 22 — STAGE A CHECKPOINT SELECTION + BASELINE FREEZE

[CANDIDATE CHECKPOINTS]


,step,eval_loss,rank_by_eval_loss,selected
0,50,0.350382,4,False
1,100,0.312700,3,False
2,150,0.279593,2,False
3,200,0.265740,1,True



[SELECTED CHECKPOINT]
Selected step           : 200
Selected eval loss      : 0.265740
Frozen adapter          : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_a_baseline/model/selected_adapter

[TRAINING ADEQUACY SIGNAL]
Eval loss @150          : 0.279593
Eval loss @200          : 0.265740
Absolute improvement    : 0.013853
Relative improvement    : 4.95%
Upper-bound optimum     : True
Stage B required        : True

[SELECTION CHECKS]
PASS   — candidate_row_n_4
PASS   — candidate_steps_exact
PASS   — one_eval_per_candidate
PASS   — candidate_eval_losses_finite
PASS   — selected_checkpoint_exists
PASS   — selected_adapter_config_exists
PASS   — selected_adapter_weights_exist
PASS   — selected_adapter_dir_exists
PASS   — frozen_adapter_config_exists
PASS   — frozen_adapter_weights_exist
PASS   — selection_metric_eval_loss
PASS   — selection_direction_minimize
PASS   — selected_step_is_argmin
PASS   — selected_step_200
PASS   — selected_at_upper_boundary
PASS   — loss_i

In [11]:
# ==================================================================================================
# DAY 22 — CELL 11
# STAGE B TRAINING-HORIZON ADEQUACY AUDIT — PROTOCOL FREEZE
# ==================================================================================================

import json
from datetime import datetime, timezone


# --------------------------------------------------------------------------------------------------
# 1. Stage B directories / formal paths
# --------------------------------------------------------------------------------------------------

STAGE_B_DIR = (
    DAY22_RUNS_DIR
    / "stage_b_horizon_audit"
)

STAGE_B_MODEL_DIR = (
    STAGE_B_DIR
    / "model"
)

STAGE_B_CHECKPOINT_DIR = (
    STAGE_B_MODEL_DIR
    / "checkpoints"
)

STAGE_B_ARTIFACT_DIR = (
    STAGE_B_DIR
    / "artifacts"
)


for path in [
    STAGE_B_DIR,
    STAGE_B_MODEL_DIR,
    STAGE_B_CHECKPOINT_DIR,
    STAGE_B_ARTIFACT_DIR,
]:

    path.mkdir(
        parents=True,
        exist_ok=True,
    )


STAGE_B_CONFIG_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_horizon_audit_config.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen continuation checkpoint
#
# IMPORTANT:
# Stage B is NOT a new training run from step 0.
#
# We resume the COMPLETE optimization state from the formal Stage A checkpoint-200:
#   - LoRA parameters
#   - optimizer state
#   - LR scheduler state
#   - Trainer state
#
# This makes 300 / 400 / 500 genuine extensions of the same training trajectory.
# --------------------------------------------------------------------------------------------------

STAGE_B_RESUME_CHECKPOINT = (
    STAGE_A_CHECKPOINT_DIR
    / "checkpoint-200"
)


# --------------------------------------------------------------------------------------------------
# 3. Formal Stage B horizon
# --------------------------------------------------------------------------------------------------

STAGE_B_HORIZONS = [
    200,
    300,
    400,
    500,
]


STAGE_B_NEW_HORIZONS = [
    300,
    400,
    500,
]


STAGE_B_OPTIONAL_EXTENSION = 600


# --------------------------------------------------------------------------------------------------
# 4. Stage B frozen protocol
# --------------------------------------------------------------------------------------------------

STAGE_B_CONFIG = {

    "day":
        22,

    "stage":
        "B",

    "stage_name":
        "training_horizon_adequacy_audit",

    "objective":
        (
            "Determine whether the original 200-step training budget is "
            "adequate by extending the exact Stage A training trajectory."
        ),

    # ----------------------------------------------------------------------------------------------
    # Scientific rationale
    # ----------------------------------------------------------------------------------------------

    "trigger": {

        "stage_a_selected_step":
            int(
                selected_step
            ),

        "stage_a_selected_eval_loss":
            float(
                selected_eval_loss
            ),

        "selected_at_upper_boundary":
            bool(
                selected_at_upper_boundary
            ),

        "absolute_eval_loss_improvement_150_to_200":
            float(
                loss_improvement_150_to_200
            ),

        "relative_eval_loss_improvement_150_to_200":
            float(
                relative_improvement_150_to_200
            ),

        "stage_b_required":
            bool(
                stage_b_horizon_audit_required
            ),
    },

    # ----------------------------------------------------------------------------------------------
    # Data
    # ----------------------------------------------------------------------------------------------

    "data_protocol": {

        "training_split":
            "optimization_train",

        "training_n":
            int(
                len(
                    optimization_train
                )
            ),

        "evaluation_split":
            "optimization_dev",

        "evaluation_n":
            int(
                len(
                    optimization_dev
                )
            ),

        "sampling":
            "natural",

        "historical_validation_used":
            False,

        "future_f1_f6_used":
            False,
    },

    # ----------------------------------------------------------------------------------------------
    # Training trajectory
    # ----------------------------------------------------------------------------------------------

    "continuation_protocol": {

        "type":
            "exact_checkpoint_resume",

        "resume_checkpoint":
            str(
                STAGE_B_RESUME_CHECKPOINT
            ),

        "resume_step":
            200,

        "preserve_optimizer_state":
            True,

        "preserve_scheduler_state":
            True,

        "fresh_lora_initialization":
            False,

        "restart_from_base_model":
            False,
    },

    # ----------------------------------------------------------------------------------------------
    # Frozen model / optimization hyperparameters
    #
    # Stage B changes ONLY the maximum training horizon.
    # ----------------------------------------------------------------------------------------------

    "model_protocol": {

        "base_model":
            STAGE_A_CONFIG[
                "base_model"
            ][
                "model_name"
            ],

        "quantization":
            STAGE_A_CONFIG[
                "quantization"
            ],

        "lora":
            STAGE_A_CONFIG[
                "lora"
            ],

        "learning_rate":
            STAGE_A_CONFIG[
                "training"
            ][
                "learning_rate"
            ],

        "effective_batch_size":
            STAGE_A_CONFIG[
                "training"
            ][
                "effective_train_batch_size"
            ],

        "warmup_steps":
            STAGE_A_CONFIG[
                "training"
            ][
                "warmup_steps"
            ],

        "seed":
            STAGE_A_CONFIG[
                "training"
            ][
                "seed"
            ],

        "data_seed":
            STAGE_A_CONFIG[
                "training"
            ][
                "data_seed"
            ],
    },

    # ----------------------------------------------------------------------------------------------
    # Horizon audit
    # ----------------------------------------------------------------------------------------------

    "horizon_protocol": {

        "formal_horizons":
            STAGE_B_HORIZONS,

        "already_observed_horizon":
            200,

        "new_horizons":
            STAGE_B_NEW_HORIZONS,

        "maximum_step_stage_b":
            500,

        "evaluation_interval_after_resume":
            100,

        "save_interval_after_resume":
            100,

        "optional_extension_step":
            STAGE_B_OPTIONAL_EXTENSION,

        "optional_extension_rule":
            (
                "Enable step 600 only if step 500 remains the best formal "
                "horizon and Optimization Dev eval_loss is still materially "
                "improving at the upper boundary."
            ),
    },

    # ----------------------------------------------------------------------------------------------
    # Selection
    # ----------------------------------------------------------------------------------------------

    "selection_protocol": {

        "candidate_horizons":
            STAGE_B_HORIZONS,

        "selection_split":
            "optimization_dev",

        "primary_selection_metric":
            "eval_loss",

        "selection_direction":
            "minimize",

        "tie_break":
            "shorter_training_horizon",

        "historical_validation_allowed":
            False,

        "future_metrics_allowed":
            False,
    },

    "freeze_status":
        "FROZEN_BEFORE_STAGE_B_CONTINUATION",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 5. Resume checkpoint integrity
# --------------------------------------------------------------------------------------------------

resume_checkpoint_checks = {

    "checkpoint_200_exists":
        STAGE_B_RESUME_CHECKPOINT.exists(),

    "adapter_config_exists":
        (
            STAGE_B_RESUME_CHECKPOINT
            / "adapter_config.json"
        ).exists(),

    "adapter_weights_exist":
        (
            (
                STAGE_B_RESUME_CHECKPOINT
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                STAGE_B_RESUME_CHECKPOINT
                / "adapter_model.bin"
            ).exists()
        ),

    "optimizer_state_exists":
        (
            STAGE_B_RESUME_CHECKPOINT
            / "optimizer.pt"
        ).exists(),

    "scheduler_state_exists":
        (
            STAGE_B_RESUME_CHECKPOINT
            / "scheduler.pt"
        ).exists(),

    "trainer_state_exists":
        (
            STAGE_B_RESUME_CHECKPOINT
            / "trainer_state.json"
        ).exists(),
}


# --------------------------------------------------------------------------------------------------
# 6. Formal Stage B protocol checks
# --------------------------------------------------------------------------------------------------

stage_b_protocol_checks = {

    **resume_checkpoint_checks,

    "stage_b_trigger_true":
        stage_b_horizon_audit_required
        is True,

    "resume_step_200":
        STAGE_B_CONFIG[
            "continuation_protocol"
        ][
            "resume_step"
        ]
        == 200,

    "formal_horizons_exact":
        STAGE_B_CONFIG[
            "horizon_protocol"
        ][
            "formal_horizons"
        ]
        ==
        [
            200,
            300,
            400,
            500,
        ],

    "new_horizons_exact":
        STAGE_B_CONFIG[
            "horizon_protocol"
        ][
            "new_horizons"
        ]
        ==
        [
            300,
            400,
            500,
        ],

    "max_step_500":
        STAGE_B_CONFIG[
            "horizon_protocol"
        ][
            "maximum_step_stage_b"
        ]
        == 500,

    "eval_interval_100":
        STAGE_B_CONFIG[
            "horizon_protocol"
        ][
            "evaluation_interval_after_resume"
        ]
        == 100,

    "save_interval_100":
        STAGE_B_CONFIG[
            "horizon_protocol"
        ][
            "save_interval_after_resume"
        ]
        == 100,

    "optional_extension_600":
        STAGE_B_CONFIG[
            "horizon_protocol"
        ][
            "optional_extension_step"
        ]
        == 600,

    "natural_sampling_preserved":
        STAGE_B_CONFIG[
            "data_protocol"
        ][
            "sampling"
        ]
        ==
        "natural",

    "learning_rate_preserved":
        STAGE_B_CONFIG[
            "model_protocol"
        ][
            "learning_rate"
        ]
        ==
        1e-4,

    "effective_batch_preserved":
        STAGE_B_CONFIG[
            "model_protocol"
        ][
            "effective_batch_size"
        ]
        ==
        8,

    "lora_r_preserved":
        STAGE_B_CONFIG[
            "model_protocol"
        ][
            "lora"
        ][
            "r"
        ]
        ==
        8,

    "lora_alpha_preserved":
        STAGE_B_CONFIG[
            "model_protocol"
        ][
            "lora"
        ][
            "lora_alpha"
        ]
        ==
        32,

    "optimizer_state_preserved":
        STAGE_B_CONFIG[
            "continuation_protocol"
        ][
            "preserve_optimizer_state"
        ]
        is True,

    "scheduler_state_preserved":
        STAGE_B_CONFIG[
            "continuation_protocol"
        ][
            "preserve_scheduler_state"
        ]
        is True,

    "historical_validation_firewall":
        STAGE_B_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used"
        ]
        is False,

    "future_firewall":
        STAGE_B_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


all_stage_b_protocol_checks_pass = bool(
    all(
        stage_b_protocol_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Save frozen Stage B protocol
# --------------------------------------------------------------------------------------------------

with open(
    STAGE_B_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        STAGE_B_CONFIG,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 8. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE B TRAINING-HORIZON ADEQUACY AUDIT PROTOCOL")
print("=" * 120)


print("\n[TRIGGER]")
print(
    f"Stage A selected step       : "
    f"{selected_step}"
)
print(
    f"Stage A selected eval loss  : "
    f"{selected_eval_loss:.6f}"
)
print(
    f"Upper-bound optimum         : "
    f"{selected_at_upper_boundary}"
)
print(
    f"150→200 relative gain       : "
    f"{relative_improvement_150_to_200:.2%}"
)


print("\n[CONTINUATION]")
print(
    f"Resume checkpoint           : "
    f"{STAGE_B_RESUME_CHECKPOINT}"
)
print(
    "Optimizer state            : PRESERVED"
)
print(
    "Scheduler state            : PRESERVED"
)
print(
    "Fresh LoRA                 : False"
)


print("\n[HORIZON AUDIT]")
print(
    f"Formal horizons             : "
    f"{STAGE_B_HORIZONS}"
)
print(
    f"Already observed            : "
    f"200"
)
print(
    f"New evaluation horizons     : "
    f"{STAGE_B_NEW_HORIZONS}"
)
print(
    f"Stage B max step            : "
    f"500"
)
print(
    f"Eval/save interval          : "
    f"100 steps"
)
print(
    f"Optional extension          : "
    f"{STAGE_B_OPTIONAL_EXTENSION}"
)


print("\n[RESUME CHECKPOINT CHECKS]")

for name, passed in resume_checkpoint_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[PROTOCOL CHECKS]")

for name, passed in stage_b_protocol_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    f"Stage B config              : "
    f"{STAGE_B_CONFIG_PATH}"
)


print("\n" + "-" * 120)

if all_stage_b_protocol_checks_pass:

    print(
        "DAY 22 CELL 11 STATUS: PASS — "
        "STAGE B HORIZON AUDIT PROTOCOL FROZEN"
    )

else:

    print(
        "DAY 22 CELL 11 STATUS: FAIL — "
        "DO NOT CONTINUE TRAINING"
    )

print("=" * 120)

DAY 22 — STAGE B TRAINING-HORIZON ADEQUACY AUDIT PROTOCOL

[TRIGGER]
Stage A selected step       : 200
Stage A selected eval loss  : 0.265740
Upper-bound optimum         : True
150→200 relative gain       : 4.95%

[CONTINUATION]
Resume checkpoint           : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_a_baseline/model/checkpoints/checkpoint-200
Optimizer state            : PRESERVED
Scheduler state            : PRESERVED
Fresh LoRA                 : False

[HORIZON AUDIT]
Formal horizons             : [200, 300, 400, 500]
Already observed            : 200
New evaluation horizons     : [300, 400, 500]
Stage B max step            : 500
Eval/save interval          : 100 steps
Optional extension          : 600

[RESUME CHECKPOINT CHECKS]
PASS   — checkpoint_200_exists
PASS   — adapter_config_exists
PASS   — adapter_weights_exist
PASS   — optimizer_state_exists
PASS   — scheduler_state_exists
PASS   — trainer_state_exists

[PROTOCOL CHECKS]
PASS   — checkpoint_200_exists


In [12]:
# ==================================================================================================
# DAY 22 — CELL 12
# STAGE A LEARNING-RATE SCHEDULER + CONTINUATION INTEGRITY AUDIT
# ==================================================================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact
# --------------------------------------------------------------------------------------------------

STAGE_B_SCHEDULER_AUDIT_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_a_scheduler_continuation_audit.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Recover Stage A TrainingArguments scheduler specification
# --------------------------------------------------------------------------------------------------

scheduler_type_obj = (
    trainer.args.lr_scheduler_type
)


if hasattr(
    scheduler_type_obj,
    "value",
):
    scheduler_type = str(
        scheduler_type_obj.value
    )

else:
    scheduler_type = str(
        scheduler_type_obj
    )


stage_a_max_steps = int(
    trainer.args.max_steps
)

stage_a_warmup_steps = int(
    trainer.args.warmup_steps
)

stage_a_base_lr = float(
    trainer.args.learning_rate
)


# --------------------------------------------------------------------------------------------------
# 3. Recover logged learning-rate trajectory
# --------------------------------------------------------------------------------------------------

stage_a_log = pd.read_csv(
    STAGE_A_TRAINING_LOG_PATH
)


if "learning_rate" in stage_a_log.columns:

    lr_log = (
        stage_a_log.loc[
            stage_a_log[
                "learning_rate"
            ].notna(),
            [
                "step",
                "learning_rate",
            ],
        ]
        .copy()
        .sort_values(
            "step"
        )
        .reset_index(drop=True)
    )

else:

    lr_log = pd.DataFrame(
        columns=[
            "step",
            "learning_rate",
        ]
    )


# --------------------------------------------------------------------------------------------------
# 4. Extract key logged LR values
# --------------------------------------------------------------------------------------------------

def get_logged_lr(step):

    rows = lr_log.loc[
        lr_log[
            "step"
        ].astype(int)
        == int(step)
    ]

    if len(rows) == 0:
        return None

    return float(
        rows.iloc[-1][
            "learning_rate"
        ]
    )


lr_step_10 = get_logged_lr(
    10
)

lr_step_50 = get_logged_lr(
    50
)

lr_step_100 = get_logged_lr(
    100
)

lr_step_150 = get_logged_lr(
    150
)

lr_step_200 = get_logged_lr(
    200
)


# --------------------------------------------------------------------------------------------------
# 5. Inspect live scheduler / optimizer state after Stage A
# --------------------------------------------------------------------------------------------------

live_scheduler_available = (
    getattr(
        trainer,
        "lr_scheduler",
        None,
    )
    is not None
)


if live_scheduler_available:

    live_scheduler_last_epoch = int(
        trainer.lr_scheduler.last_epoch
    )

    live_scheduler_lr = [
        float(x)
        for x
        in trainer.lr_scheduler.get_last_lr()
    ]

else:

    live_scheduler_last_epoch = None
    live_scheduler_lr = []


live_optimizer_available = (
    getattr(
        trainer,
        "optimizer",
        None,
    )
    is not None
)


if live_optimizer_available:

    live_optimizer_lrs = [
        float(
            group[
                "lr"
            ]
        )
        for group
        in trainer.optimizer.param_groups
    ]

else:

    live_optimizer_lrs = []


# --------------------------------------------------------------------------------------------------
# 6. Determine whether Stage A scheduler was exhausted at step 200
# --------------------------------------------------------------------------------------------------

LR_ZERO_TOLERANCE = 1e-12


logged_lr_200_is_zero = (
    lr_step_200 is not None
    and
    abs(
        lr_step_200
    )
    <= LR_ZERO_TOLERANCE
)


live_scheduler_lr_zero = (
    len(
        live_scheduler_lr
    )
    > 0
    and
    all(
        abs(x)
        <= LR_ZERO_TOLERANCE
        for x
        in live_scheduler_lr
    )
)


live_optimizer_lr_zero = (
    len(
        live_optimizer_lrs
    )
    > 0
    and
    all(
        abs(x)
        <= LR_ZERO_TOLERANCE
        for x
        in live_optimizer_lrs
    )
)


scheduler_exhausted_at_200 = bool(
    logged_lr_200_is_zero
    or
    live_scheduler_lr_zero
    or
    live_optimizer_lr_zero
)


# --------------------------------------------------------------------------------------------------
# 7. Scientific continuation assessment
#
# A horizon experiment should not silently change the learning-rate trajectory.
#
# If Stage A's scheduler was constructed for max_steps=200 and reaches zero there,
# then simply extending max_steps to 500 while loading/rebuilding scheduler state
# does NOT provide a clean interpretation of "same trajectory, only longer horizon".
# --------------------------------------------------------------------------------------------------

scheduler_tied_to_stage_a_budget = (
    scheduler_type.lower()
    in {
        "linear",
        "cosine",
        "cosine_with_restarts",
        "polynomial",
    }
)


naive_resume_is_clean_horizon_extension = not (
    scheduler_tied_to_stage_a_budget
    and
    scheduler_exhausted_at_200
)


stage_b_protocol_revision_required = bool(
    not naive_resume_is_clean_horizon_extension
)


# --------------------------------------------------------------------------------------------------
# 8. Formal audit checks
# --------------------------------------------------------------------------------------------------

scheduler_audit_checks = {

    "scheduler_type_identified":
        len(
            scheduler_type
        )
        > 0,

    "stage_a_max_steps_200":
        stage_a_max_steps
        == 200,

    "stage_a_warmup_steps_10":
        stage_a_warmup_steps
        == 10,

    "stage_a_base_lr_1e4":
        abs(
            stage_a_base_lr
            - 1e-4
        )
        < 1e-12,

    "lr_log_available":
        len(
            lr_log
        )
        > 0,

    "step_200_lr_observed":
        lr_step_200
        is not None,

    "live_scheduler_available":
        live_scheduler_available,

    "live_optimizer_available":
        live_optimizer_available,
}


audit_completed = bool(
    all(
        scheduler_audit_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 9. Save formal audit
# --------------------------------------------------------------------------------------------------

scheduler_audit = {

    "day":
        22,

    "stage":
        "B",

    "artifact":
        "stage_a_scheduler_continuation_audit",

    "stage_a_scheduler": {

        "scheduler_type":
            scheduler_type,

        "base_learning_rate":
            stage_a_base_lr,

        "warmup_steps":
            stage_a_warmup_steps,

        "max_steps":
            stage_a_max_steps,
    },

    "logged_learning_rates": {

        "step_10":
            lr_step_10,

        "step_50":
            lr_step_50,

        "step_100":
            lr_step_100,

        "step_150":
            lr_step_150,

        "step_200":
            lr_step_200,
    },

    "live_state": {

        "scheduler_last_epoch":
            live_scheduler_last_epoch,

        "scheduler_learning_rates":
            live_scheduler_lr,

        "optimizer_learning_rates":
            live_optimizer_lrs,
    },

    "continuation_assessment": {

        "scheduler_tied_to_stage_a_budget":
            scheduler_tied_to_stage_a_budget,

        "scheduler_exhausted_at_step_200":
            scheduler_exhausted_at_200,

        "naive_resume_is_clean_horizon_extension":
            naive_resume_is_clean_horizon_extension,

        "stage_b_protocol_revision_required":
            stage_b_protocol_revision_required,
    },

    "checks": {
        key:
            bool(
                value
            )
        for key, value
        in scheduler_audit_checks.items()
    },

    "audit_completed":
        audit_completed,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_B_SCHEDULER_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        scheduler_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 10. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE A SCHEDULER / CONTINUATION INTEGRITY AUDIT")
print("=" * 120)


print("\n[STAGE A SCHEDULER]")
print(
    f"Scheduler type              : "
    f"{scheduler_type}"
)
print(
    f"Base learning rate          : "
    f"{stage_a_base_lr}"
)
print(
    f"Warmup steps                : "
    f"{stage_a_warmup_steps}"
)
print(
    f"Configured max steps        : "
    f"{stage_a_max_steps}"
)


print("\n[LOGGED LEARNING RATE]")
print(
    f"Step 10                     : "
    f"{lr_step_10}"
)
print(
    f"Step 50                     : "
    f"{lr_step_50}"
)
print(
    f"Step 100                    : "
    f"{lr_step_100}"
)
print(
    f"Step 150                    : "
    f"{lr_step_150}"
)
print(
    f"Step 200                    : "
    f"{lr_step_200}"
)


print("\n[LIVE STATE]")
print(
    f"Scheduler last_epoch        : "
    f"{live_scheduler_last_epoch}"
)
print(
    f"Scheduler LR                : "
    f"{live_scheduler_lr}"
)
print(
    f"Optimizer LR                : "
    f"{live_optimizer_lrs}"
)


print("\n[CONTINUATION ASSESSMENT]")
print(
    f"Scheduler tied to max_steps : "
    f"{scheduler_tied_to_stage_a_budget}"
)
print(
    f"Scheduler exhausted @200    : "
    f"{scheduler_exhausted_at_200}"
)
print(
    f"Naive resume scientifically clean : "
    f"{naive_resume_is_clean_horizon_extension}"
)
print(
    f"Protocol revision required  : "
    f"{stage_b_protocol_revision_required}"
)


print("\n[AUDIT CHECKS]")

for name, passed in scheduler_audit_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACT]")
print(
    f"Scheduler audit             : "
    f"{STAGE_B_SCHEDULER_AUDIT_PATH}"
)


print("\n" + "-" * 120)

if not audit_completed:

    print(
        "DAY 22 CELL 12 STATUS: FAIL — "
        "SCHEDULER AUDIT INCOMPLETE"
    )

elif stage_b_protocol_revision_required:

    print(
        "DAY 22 CELL 12 STATUS: PASS — "
        "AUDIT COMPLETE; STAGE B CONTINUATION PROTOCOL REQUIRES REVISION"
    )

else:

    print(
        "DAY 22 CELL 12 STATUS: PASS — "
        "NAIVE CHECKPOINT CONTINUATION IS SCHEDULER-CONSISTENT"
    )

print("=" * 120)

DAY 22 — STAGE A SCHEDULER / CONTINUATION INTEGRITY AUDIT

[STAGE A SCHEDULER]
Scheduler type              : linear
Base learning rate          : 0.0001
Warmup steps                : 10
Configured max steps        : 200

[LOGGED LEARNING RATE]
Step 10                     : 9e-05
Step 50                     : 7.947368421052632e-05
Step 100                    : 5.31578947368421e-05
Step 150                    : 2.6842105263157896e-05
Step 200                    : 5.263157894736843e-07

[LIVE STATE]
Scheduler last_epoch        : None
Scheduler LR                : []
Optimizer LR                : []

[CONTINUATION ASSESSMENT]
Scheduler tied to max_steps : True
Scheduler exhausted @200    : False
Naive resume scientifically clean : True
Protocol revision required  : False

[AUDIT CHECKS]
PASS   — scheduler_type_identified
PASS   — stage_a_max_steps_200
PASS   — stage_a_warmup_steps_10
PASS   — stage_a_base_lr_1e4
PASS   — lr_log_available
PASS   — step_200_lr_observed
FAIL   — live_schedule

In [13]:
# ==================================================================================================
# DAY 22 — CELL 13
# REVISED STAGE B TRAINING-HORIZON ADEQUACY PROTOCOL FREEZE
# ==================================================================================================

import json
from datetime import datetime, timezone


# --------------------------------------------------------------------------------------------------
# 1. Revised Stage B principle
#
# Cell 12 established:
#
#   Stage A scheduler = linear
#   Stage A max_steps = 200
#   LR reaches zero at step 200
#
# Therefore checkpoint-200 cannot be cleanly continued while preserving the original scheduler.
#
# Revised design:
#
#   - start a NEW controlled trajectory from the same base model;
#   - initialize fresh LoRA with the same seed/config;
#   - define ONE linear schedule over 500 total steps;
#   - observe checkpoints at 200 / 300 / 400 / 500;
#   - all candidate horizons therefore lie on the SAME optimization trajectory.
#
# Stage A remains the original baseline and is NOT overwritten.
# --------------------------------------------------------------------------------------------------


REVISED_STAGE_B_CONFIG_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_horizon_audit_config_revised.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Formal revised Stage B configuration
# --------------------------------------------------------------------------------------------------

REVISED_STAGE_B_CONFIG = {

    "day":
        22,

    "stage":
        "B",

    "stage_name":
        "controlled_training_horizon_audit",

    "protocol_version":
        "revised_after_scheduler_audit",

    "supersedes":
        str(
            STAGE_B_CONFIG_PATH
        ),

    "revision_reason":
        (
            "Stage A used a linear learning-rate scheduler parameterized "
            "for max_steps=200 and the optimizer learning rate reached zero "
            "at step 200. Direct checkpoint continuation would therefore "
            "not constitute a clean controlled horizon extension."
        ),

    # ----------------------------------------------------------------------------------------------
    # Experimental question
    # ----------------------------------------------------------------------------------------------

    "objective":
        (
            "Measure training adequacy along one common 500-step optimization "
            "trajectory and determine whether performance continues improving "
            "beyond 200 steps."
        ),

    # ----------------------------------------------------------------------------------------------
    # Relationship to Stage A
    # ----------------------------------------------------------------------------------------------

    "stage_a_role": {

        "status":
            "frozen_original_baseline",

        "selected_step":
            int(
                selected_step
            ),

        "selected_eval_loss":
            float(
                selected_eval_loss
            ),

        "comparison_role":
            (
                "External reference only. Stage A step 200 and Stage B step 200 "
                "are not expected to be numerically identical because their "
                "linear schedulers are parameterized by different total budgets."
            ),

        "stage_a_adapter_modified":
            False,
    },

    # ----------------------------------------------------------------------------------------------
    # Initialization
    # ----------------------------------------------------------------------------------------------

    "initialization": {

        "base_model":
            "meta-llama/Meta-Llama-3-8B",

        "start_from_base_model":
            True,

        "load_stage_a_adapter":
            False,

        "load_stage_a_optimizer":
            False,

        "load_stage_a_scheduler":
            False,

        "fresh_lora":
            True,

        "seed":
            42,

        "data_seed":
            42,
    },

    # ----------------------------------------------------------------------------------------------
    # Data
    # ----------------------------------------------------------------------------------------------

    "data_protocol": {

        "training_split":
            "optimization_train",

        "training_n":
            int(
                len(
                    optimization_train
                )
            ),

        "evaluation_split":
            "optimization_dev",

        "evaluation_n":
            int(
                len(
                    optimization_dev
                )
            ),

        "sampling":
            "natural",

        "historical_validation_used":
            False,

        "future_f1_f6_used":
            False,
    },

    # ----------------------------------------------------------------------------------------------
    # Model / LoRA
    # ----------------------------------------------------------------------------------------------

    "model_protocol": {

        "quantization":
            STAGE_A_CONFIG[
                "quantization"
            ],

        "lora":
            STAGE_A_CONFIG[
                "lora"
            ],

        "max_seq_length":
            STAGE_A_CONFIG[
                "sequence_protocol"
            ][
                "max_seq_length"
            ],

        "prompt_template":
            STAGE_A_CONFIG[
                "sequence_protocol"
            ][
                "prompt_template"
            ],
    },

    # ----------------------------------------------------------------------------------------------
    # Optimization
    #
    # IMPORTANT:
    # One scheduler is defined ONCE for the complete 500-step trajectory.
    # Candidate horizons are observation points along this SAME trajectory.
    # ----------------------------------------------------------------------------------------------

    "training_protocol": {

        "max_steps":
            500,

        "per_device_train_batch_size":
            1,

        "per_device_eval_batch_size":
            1,

        "gradient_accumulation_steps":
            8,

        "effective_train_batch_size":
            8,

        "learning_rate":
            1e-4,

        "lr_scheduler_type":
            "linear",

        "warmup_steps":
            10,

        "weight_decay":
            0.0,

        "gradient_checkpointing":
            True,

        "seed":
            42,

        "data_seed":
            42,
    },

    # ----------------------------------------------------------------------------------------------
    # Horizon audit
    # ----------------------------------------------------------------------------------------------

    "horizon_protocol": {

        "trajectory_total_steps":
            500,

        "formal_candidate_horizons":
            [
                200,
                300,
                400,
                500,
            ],

        "evaluation_steps":
            [
                200,
                300,
                400,
                500,
            ],

        "checkpoint_steps":
            [
                200,
                300,
                400,
                500,
            ],

        "optional_extension_step":
            600,

        "optional_extension_status":
            "NOT_ENABLED",

        "optional_extension_rule":
            (
                "Do not extend automatically. Reassess only if step 500 remains "
                "the best candidate and the boundary signal indicates material "
                "continued improvement."
            ),
    },

    # ----------------------------------------------------------------------------------------------
    # Selection
    # ----------------------------------------------------------------------------------------------

    "selection_protocol": {

        "selection_split":
            "optimization_dev",

        "primary_metric":
            "eval_loss",

        "direction":
            "minimize",

        "tie_break":
            "shorter_training_horizon",

        "historical_validation_allowed":
            False,

        "future_metrics_allowed":
            False,
    },

    # ----------------------------------------------------------------------------------------------
    # Scientific interpretation
    # ----------------------------------------------------------------------------------------------

    "interpretation_protocol": {

        "valid_internal_comparison":
            (
                "Compare Stage B checkpoints 200, 300, 400, and 500 because "
                "they belong to the same model initialization, dataset, optimizer "
                "configuration, and 500-step learning-rate schedule."
            ),

        "invalid_equivalence":
            (
                "Do not interpret Stage A step 200 and Stage B step 200 as "
                "identical training states."
            ),

        "primary_question":
            (
                "Does Optimization Dev performance continue improving after "
                "the 200-step observation point under a common longer training trajectory?"
            ),
    },

    "freeze_status":
        "REVISED_STAGE_B_PROTOCOL_FROZEN",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 3. Formal revised protocol checks
# --------------------------------------------------------------------------------------------------

revised_stage_b_checks = {

    "scheduler_audit_complete":
        audit_completed
        is True,

    "scheduler_problem_confirmed":
        stage_b_protocol_revision_required
        is True,

    "fresh_base_initialization":
        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "start_from_base_model"
        ]
        is True,

    "stage_a_adapter_not_loaded":
        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "load_stage_a_adapter"
        ]
        is False,

    "stage_a_optimizer_not_loaded":
        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "load_stage_a_optimizer"
        ]
        is False,

    "stage_a_scheduler_not_loaded":
        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "load_stage_a_scheduler"
        ]
        is False,

    "fresh_lora_required":
        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "fresh_lora"
        ]
        is True,

    "seed_42":
        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "seed"
        ]
        == 42,

    "training_n_10831":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "training_n"
        ]
        == 10_831,

    "dev_n_2708":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "evaluation_n"
        ]
        == 2_708,

    "natural_sampling":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "sampling"
        ]
        ==
        "natural",

    "max_steps_500":
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "max_steps"
        ]
        == 500,

    "linear_scheduler":
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "lr_scheduler_type"
        ]
        ==
        "linear",

    "learning_rate_1e4":
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "learning_rate"
        ]
        ==
        1e-4,

    "warmup_steps_10":
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "warmup_steps"
        ]
        == 10,

    "effective_batch_8":
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "effective_train_batch_size"
        ]
        == 8,

    "candidate_horizons_exact":
        REVISED_STAGE_B_CONFIG[
            "horizon_protocol"
        ][
            "formal_candidate_horizons"
        ]
        ==
        [
            200,
            300,
            400,
            500,
        ],

    "historical_validation_firewall":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used"
        ]
        is False,

    "future_firewall":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


all_revised_stage_b_checks_pass = bool(
    all(
        revised_stage_b_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Mark original Cell 11 protocol as superseded
# --------------------------------------------------------------------------------------------------

if STAGE_B_CONFIG_PATH.exists():

    with open(
        STAGE_B_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        original_stage_b_config = load_artifact_json(
            f
        )

    original_stage_b_config[
        "freeze_status"
    ] = (
        "SUPERSEDED_AFTER_SCHEDULER_AUDIT"
    )

    original_stage_b_config[
        "superseded_by"
    ] = str(
        REVISED_STAGE_B_CONFIG_PATH
    )

    original_stage_b_config[
        "superseded_reason"
    ] = (
        "Linear scheduler reached zero learning rate at Stage A step 200; "
        "direct continuation does not provide a clean horizon comparison."
    )

    with open(
        STAGE_B_CONFIG_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            original_stage_b_config,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 5. Save revised protocol
# --------------------------------------------------------------------------------------------------

with open(
    REVISED_STAGE_B_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        REVISED_STAGE_B_CONFIG,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 6. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — REVISED STAGE B TRAINING-HORIZON PROTOCOL")
print("=" * 120)


print("\n[REVISION]")
print(
    "Original continuation plan : SUPERSEDED"
)
print(
    "Reason                     : Stage A linear scheduler exhausted at step 200"
)
print(
    "New design                 : fresh controlled 500-step trajectory"
)


print("\n[INITIALIZATION]")
print(
    "Base model                 : meta-llama/Meta-Llama-3-8B"
)
print(
    "Fresh LoRA                 : True"
)
print(
    "Load Stage A adapter       : False"
)
print(
    "Load Stage A optimizer     : False"
)
print(
    "Load Stage A scheduler     : False"
)
print(
    f"Seed                       : "
    f"{REVISED_STAGE_B_CONFIG['initialization']['seed']}"
)


print("\n[TRAINING TRAJECTORY]")
print(
    f"Total trajectory           : "
    f"{REVISED_STAGE_B_CONFIG['training_protocol']['max_steps']} steps"
)
print(
    f"Learning rate              : "
    f"{REVISED_STAGE_B_CONFIG['training_protocol']['learning_rate']}"
)
print(
    f"Scheduler                  : "
    f"{REVISED_STAGE_B_CONFIG['training_protocol']['lr_scheduler_type']}"
)
print(
    f"Warmup                     : "
    f"{REVISED_STAGE_B_CONFIG['training_protocol']['warmup_steps']} steps"
)
print(
    f"Effective batch            : "
    f"{REVISED_STAGE_B_CONFIG['training_protocol']['effective_train_batch_size']}"
)


print("\n[FORMAL HORIZONS]")
print(
    f"Candidate checkpoints      : "
    f"{REVISED_STAGE_B_CONFIG['horizon_protocol']['formal_candidate_horizons']}"
)
print(
    "All candidates belong to ONE common 500-step LR trajectory."
)


print("\n[SCIENTIFIC INTERPRETATION]")
print(
    "Stage A step 200           : original 200-step baseline"
)
print(
    "Stage B step 200           : observation on new 500-step trajectory"
)
print(
    "Stage A 200 == Stage B 200 : FALSE by design"
)


print("\n[PROTOCOL CHECKS]")

for name, passed in revised_stage_b_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Superseded protocol        : "
    f"{STAGE_B_CONFIG_PATH}"
)
print(
    f"Revised protocol           : "
    f"{REVISED_STAGE_B_CONFIG_PATH}"
)


print("\n" + "-" * 120)

if all_revised_stage_b_checks_pass:

    print(
        "DAY 22 CELL 13 STATUS: PASS — "
        "REVISED STAGE B CONTROLLED HORIZON PROTOCOL FROZEN"
    )

else:

    print(
        "DAY 22 CELL 13 STATUS: FAIL — "
        "DO NOT INITIALIZE STAGE B MODEL"
    )

print("=" * 120)

DAY 22 — REVISED STAGE B TRAINING-HORIZON PROTOCOL

[REVISION]
Original continuation plan : SUPERSEDED
Reason                     : Stage A linear scheduler exhausted at step 200
New design                 : fresh controlled 500-step trajectory

[INITIALIZATION]
Base model                 : meta-llama/Meta-Llama-3-8B
Fresh LoRA                 : True
Load Stage A adapter       : False
Load Stage A optimizer     : False
Load Stage A scheduler     : False
Seed                       : 42

[TRAINING TRAJECTORY]
Total trajectory           : 500 steps
Learning rate              : 0.0001
Scheduler                  : linear
Warmup                     : 10 steps
Effective batch            : 8

[FORMAL HORIZONS]
Candidate checkpoints      : [200, 300, 400, 500]
All candidates belong to ONE common 500-step LR trajectory.

[SCIENTIFIC INTERPRETATION]
Stage A step 200           : original 200-step baseline
Stage B step 200           : observation on new 500-step trajectory
Stage A 200 == Stage B 20

In [14]:
# ==================================================================================================
# DAY 22 — CELL 14
# REVISED STAGE B — FRESH 4-BIT BASE MODEL + DETERMINISTIC LoRA INITIALIZATION
# ==================================================================================================

import gc
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# --------------------------------------------------------------------------------------------------
# 1. Stage B model directories
# --------------------------------------------------------------------------------------------------

STAGE_B_CHECKPOINT_DIR = (
    STAGE_B_MODEL_DIR
    / "checkpoints"
)

STAGE_B_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate — Stage B must start clean
# --------------------------------------------------------------------------------------------------

existing_stage_b_checkpoints = sorted(
    [
        p
        for p in STAGE_B_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_stage_b_checkpoints
) == 0, (
    "DAY 22 STOP: pre-existing Stage B checkpoints detected. "
    "Revised Stage B must start from a clean trajectory."
)


# --------------------------------------------------------------------------------------------------
# 3. Release Stage A in-memory training objects
#
# Stage A artifacts/checkpoints on Drive are untouched.
# --------------------------------------------------------------------------------------------------

objects_to_delete = [
    "trainer",
    "model",
    "base_model",
    "peft_config",
    "training_args",
    "train_result",
]


for object_name in objects_to_delete:

    if object_name in globals():

        del globals()[
            object_name
        ]


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


gpu_after_cleanup_gb = (
    torch.cuda.memory_allocated()
    / (1024 ** 3)
    if torch.cuda.is_available()
    else None
)


# --------------------------------------------------------------------------------------------------
# 4. Re-establish deterministic Stage B initialization seed
# --------------------------------------------------------------------------------------------------

STAGE_B_SEED = int(
    REVISED_STAGE_B_CONFIG[
        "initialization"
    ][
        "seed"
    ]
)


random.seed(
    STAGE_B_SEED
)

np.random.seed(
    STAGE_B_SEED
)

torch.manual_seed(
    STAGE_B_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_B_SEED
    )


set_seed(
    STAGE_B_SEED
)


# --------------------------------------------------------------------------------------------------
# 5. Reconstruct frozen 4-bit quantization configuration
# --------------------------------------------------------------------------------------------------

stage_b_quant_cfg = (
    REVISED_STAGE_B_CONFIG[
        "model_protocol"
    ][
        "quantization"
    ]
)


stage_b_bnb_config = BitsAndBytesConfig(

    load_in_4bit=
        stage_b_quant_cfg[
            "load_in_4bit"
        ],

    bnb_4bit_quant_type=
        stage_b_quant_cfg[
            "bnb_4bit_quant_type"
        ],

    bnb_4bit_use_double_quant=
        stage_b_quant_cfg[
            "bnb_4bit_use_double_quant"
        ],

    bnb_4bit_compute_dtype=
        torch.float16,
)


# --------------------------------------------------------------------------------------------------
# 6. Load a fresh common base model
#
# IMPORTANT:
#   - no Stage A adapter
#   - no Stage A optimizer
#   - no Stage A scheduler
#   - no Stage A trainer state
# --------------------------------------------------------------------------------------------------

stage_b_base_model = (
    AutoModelForCausalLM.from_pretrained(

        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "base_model"
        ],

        quantization_config=
            stage_b_bnb_config,

        device_map=
            "auto",

        trust_remote_code=
            True,
    )
)


stage_b_base_model.config.use_cache = False


# --------------------------------------------------------------------------------------------------
# 7. Prepare base model for k-bit LoRA training
# --------------------------------------------------------------------------------------------------

stage_b_base_model = (
    prepare_model_for_kbit_training(

        stage_b_base_model,

        use_gradient_checkpointing=
            REVISED_STAGE_B_CONFIG[
                "training_protocol"
            ][
                "gradient_checkpointing"
            ],
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Re-establish seed immediately before LoRA initialization
#
# This is intentionally repeated so the LoRA initialization itself is deterministic.
# --------------------------------------------------------------------------------------------------

set_seed(
    STAGE_B_SEED
)


# --------------------------------------------------------------------------------------------------
# 9. Fresh Stage B LoRA
# --------------------------------------------------------------------------------------------------

stage_b_lora_cfg = (
    REVISED_STAGE_B_CONFIG[
        "model_protocol"
    ][
        "lora"
    ]
)


stage_b_peft_config = LoraConfig(

    r=
        stage_b_lora_cfg[
            "r"
        ],

    lora_alpha=
        stage_b_lora_cfg[
            "lora_alpha"
        ],

    lora_dropout=
        stage_b_lora_cfg[
            "lora_dropout"
        ],

    target_modules=
        stage_b_lora_cfg[
            "target_modules"
        ],

    bias=
        stage_b_lora_cfg[
            "bias"
        ],

    task_type=
        stage_b_lora_cfg[
            "task_type"
        ],

    inference_mode=
        False,
)


stage_b_model = get_peft_model(
    stage_b_base_model,
    stage_b_peft_config,
)


# --------------------------------------------------------------------------------------------------
# 10. Parameter audit
# --------------------------------------------------------------------------------------------------

stage_b_total_params = 0
stage_b_trainable_params = 0

stage_b_trainable_names = []


for name, param in stage_b_model.named_parameters():

    n_params = param.numel()

    stage_b_total_params += n_params

    if param.requires_grad:

        stage_b_trainable_params += n_params

        stage_b_trainable_names.append(
            name
        )


stage_b_trainable_pct = (
    100.0
    * stage_b_trainable_params
    / stage_b_total_params
)


stage_b_unexpected_trainable = [

    name
    for name
    in stage_b_trainable_names

    if "lora_" not in name
]


stage_b_q_proj_present = any(

    "q_proj" in name
    and
    "lora_" in name

    for name
    in stage_b_trainable_names
)


stage_b_v_proj_present = any(

    "v_proj" in name
    and
    "lora_" in name

    for name
    in stage_b_trainable_names
)


# --------------------------------------------------------------------------------------------------
# 11. Explicitly confirm no Stage A adapter is attached
# --------------------------------------------------------------------------------------------------

stage_b_adapter_names = (
    sorted(
        list(
            stage_b_model.peft_config.keys()
        )
    )
)


stage_a_selected_adapter_path_loaded = False

# By construction this model was created from the base model via get_peft_model,
# not PeftModel.from_pretrained(...), so no Stage A adapter path was loaded.


# --------------------------------------------------------------------------------------------------
# 12. Formal integrity checks
# --------------------------------------------------------------------------------------------------

stage_b_model_checks = {

    "cuda_available":
        torch.cuda.is_available(),

    "stage_a_objects_released":
        (
            "trainer" not in globals()
            and
            "model" not in globals()
            and
            "base_model" not in globals()
        ),

    "stage_b_model_loaded":
        stage_b_model is not None,

    "fresh_peft_adapter_present":
        hasattr(
            stage_b_model,
            "peft_config",
        ),

    "adapter_named_default":
        "default"
        in stage_b_model.peft_config,

    "stage_a_adapter_not_loaded":
        stage_a_selected_adapter_path_loaded
        is False,

    "base_model_exact":
        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "base_model"
        ]
        ==
        "meta-llama/Meta-Llama-3-8B",

    "fresh_lora_protocol":
        REVISED_STAGE_B_CONFIG[
            "initialization"
        ][
            "fresh_lora"
        ]
        is True,

    "seed_42":
        STAGE_B_SEED
        == 42,

    "load_in_4bit":
        stage_b_quant_cfg[
            "load_in_4bit"
        ]
        is True,

    "quant_type_nf4":
        stage_b_quant_cfg[
            "bnb_4bit_quant_type"
        ]
        ==
        "nf4",

    "double_quant_enabled":
        stage_b_quant_cfg[
            "bnb_4bit_use_double_quant"
        ]
        is True,

    "lora_r8":
        stage_b_model.peft_config[
            "default"
        ].r
        == 8,

    "lora_alpha32":
        stage_b_model.peft_config[
            "default"
        ].lora_alpha
        == 32,

    "lora_dropout_01":
        abs(
            float(
                stage_b_model.peft_config[
                    "default"
                ].lora_dropout
            )
            - 0.1
        )
        < 1e-12,

    "q_proj_lora_present":
        stage_b_q_proj_present,

    "v_proj_lora_present":
        stage_b_v_proj_present,

    "trainable_params_positive":
        stage_b_trainable_params
        > 0,

    "only_lora_trainable":
        len(
            stage_b_unexpected_trainable
        )
        == 0,

    "use_cache_disabled":
        stage_b_model.config.use_cache
        is False,

    "stage_b_checkpoint_dir_clean":
        len(
            existing_stage_b_checkpoints
        )
        == 0,
}


all_stage_b_model_checks_pass = bool(
    all(
        stage_b_model_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 13. GPU snapshot
# --------------------------------------------------------------------------------------------------

if torch.cuda.is_available():

    stage_b_gpu_allocated_gb = (
        torch.cuda.memory_allocated()
        / (1024 ** 3)
    )

    stage_b_gpu_reserved_gb = (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )

    stage_b_gpu_peak_gb = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

else:

    stage_b_gpu_allocated_gb = None
    stage_b_gpu_reserved_gb = None
    stage_b_gpu_peak_gb = None


# --------------------------------------------------------------------------------------------------
# 14. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — REVISED STAGE B FRESH MODEL INITIALIZATION")
print("=" * 120)


print("\n[MEMORY CLEANUP]")

if gpu_after_cleanup_gb is not None:

    print(
        f"GPU allocated after cleanup : "
        f"{gpu_after_cleanup_gb:.2f} GB"
    )


print("\n[INITIALIZATION]")
print(
    "Source                     : fresh base model"
)
print(
    "Stage A adapter loaded     : False"
)
print(
    "Stage A optimizer loaded   : False"
)
print(
    "Stage A scheduler loaded   : False"
)
print(
    f"Seed                       : "
    f"{STAGE_B_SEED}"
)


print("\n[BASE MODEL]")
print(
    f"Model                      : "
    f"{REVISED_STAGE_B_CONFIG['initialization']['base_model']}"
)
print(
    "4-bit                     : True"
)
print(
    "Quantization              : NF4 + double quantization"
)


print("\n[LoRA]")
print(
    f"Adapter names              : "
    f"{stage_b_adapter_names}"
)
print(
    f"Rank                       : "
    f"{stage_b_model.peft_config['default'].r}"
)
print(
    f"Alpha                      : "
    f"{stage_b_model.peft_config['default'].lora_alpha}"
)
print(
    f"Dropout                    : "
    f"{stage_b_model.peft_config['default'].lora_dropout}"
)


print("\n[PARAMETERS]")
print(
    f"Trainable parameters       : "
    f"{stage_b_trainable_params:,}"
)
print(
    f"Total parameters           : "
    f"{stage_b_total_params:,}"
)
print(
    f"Trainable percentage       : "
    f"{stage_b_trainable_pct:.6f}%"
)
print(
    f"Trainable tensors          : "
    f"{len(stage_b_trainable_names):,}"
)
print(
    f"Unexpected trainable       : "
    f"{len(stage_b_unexpected_trainable):,}"
)


print("\n[INTEGRITY CHECKS]")

for name, passed in stage_b_model_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[GPU MEMORY]")

if torch.cuda.is_available():

    print(
        f"Allocated                  : "
        f"{stage_b_gpu_allocated_gb:.2f} GB"
    )

    print(
        f"Reserved                   : "
        f"{stage_b_gpu_reserved_gb:.2f} GB"
    )

    print(
        f"Peak allocated             : "
        f"{stage_b_gpu_peak_gb:.2f} GB"
    )


print("\n" + "-" * 120)

if all_stage_b_model_checks_pass:

    print(
        "DAY 22 CELL 14 STATUS: PASS — "
        "FRESH STAGE B 500-STEP TRAJECTORY MODEL READY"
    )

else:

    print(
        "DAY 22 CELL 14 STATUS: FAIL — "
        "DO NOT BUILD STAGE B TRAINER"
    )

print("=" * 120)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

DAY 22 — REVISED STAGE B FRESH MODEL INITIALIZATION

[MEMORY CLEANUP]
GPU allocated after cleanup : 1.96 GB

[INITIALIZATION]
Source                     : fresh base model
Stage A adapter loaded     : False
Stage A optimizer loaded   : False
Stage A scheduler loaded   : False
Seed                       : 42

[BASE MODEL]
Model                      : meta-llama/Meta-Llama-3-8B
4-bit                     : True
Quantization              : NF4 + double quantization

[LoRA]
Adapter names              : ['default']
Rank                       : 8
Alpha                      : 32
Dropout                    : 0.1

[PARAMETERS]
Trainable parameters       : 3,407,872
Total parameters           : 4,544,008,192
Trainable percentage       : 0.074997%
Trainable tensors          : 128
Unexpected trainable       : 0

[INTEGRITY CHECKS]
PASS   — cuda_available
PASS   — stage_a_objects_released
PASS   — stage_b_model_loaded
PASS   — fresh_peft_adapter_present
PASS   — adapter_named_default
PASS   — stage_

In [15]:
# ==================================================================================================
# DAY 22 — CELL 15
# REVISED STAGE B TRAINER INFRASTRUCTURE + EXACT HORIZON CALLBACK
# ==================================================================================================

from pathlib import Path

import torch

from transformers import (
    Trainer,
    TrainerCallback,
    TrainingArguments,
)


# --------------------------------------------------------------------------------------------------
# 1. Formal Stage B horizons
# --------------------------------------------------------------------------------------------------

STAGE_B_FORMAL_HORIZONS = (
    REVISED_STAGE_B_CONFIG[
        "horizon_protocol"
    ][
        "formal_candidate_horizons"
    ]
)

assert STAGE_B_FORMAL_HORIZONS == [
    200,
    300,
    400,
    500,
]


STAGE_B_MAX_STEPS = int(
    REVISED_STAGE_B_CONFIG[
        "training_protocol"
    ][
        "max_steps"
    ]
)


# --------------------------------------------------------------------------------------------------
# 2. Reuse the already frozen encoded datasets
#
# No re-tokenization and no prompt change.
# --------------------------------------------------------------------------------------------------

assert len(
    hf_opt_train_model
) == 10_831

assert len(
    hf_opt_dev_model
) == 2_708


# --------------------------------------------------------------------------------------------------
# 3. Exact-horizon callback
#
# TrainingArguments itself will NOT use periodic evaluation or saving.
#
# This callback explicitly triggers both actions only at:
#
#   200 / 300 / 400 / 500
#
# This prevents an unnecessary full Dev pass at step 100.
# --------------------------------------------------------------------------------------------------

class ExactHorizonEvaluationSaveCallback(
    TrainerCallback
):

    def __init__(
        self,
        target_steps,
    ):

        self.target_steps = set(
            int(x)
            for x
            in target_steps
        )

        self.triggered_eval_steps = []
        self.triggered_save_steps = []


    def on_step_end(
        self,
        args,
        state,
        control,
        **kwargs,
    ):

        current_step = int(
            state.global_step
        )

        # Explicitly suppress periodic evaluation/save
        # except at the frozen formal horizons.
        control.should_evaluate = False
        control.should_save = False

        if current_step in self.target_steps:

            control.should_evaluate = True
            control.should_save = True

            self.triggered_eval_steps.append(
                current_step
            )

            self.triggered_save_steps.append(
                current_step
            )

        return control


stage_b_horizon_callback = (
    ExactHorizonEvaluationSaveCallback(
        target_steps=
            STAGE_B_FORMAL_HORIZONS
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Stage B TrainingArguments
#
# IMPORTANT:
#
#   eval_strategy="no"
#   save_strategy="no"
#
# Periodic Trainer behavior is disabled.
# The exact-horizon callback above controls formal eval/save events.
# --------------------------------------------------------------------------------------------------

stage_b_training_args = TrainingArguments(

    # Output
    output_dir=
        str(
            STAGE_B_CHECKPOINT_DIR
        ),

    # ----------------------------------------------------------------------------------------------
    # Training horizon
    # ----------------------------------------------------------------------------------------------

    max_steps=
        STAGE_B_MAX_STEPS,

    # ----------------------------------------------------------------------------------------------
    # Batch
    # ----------------------------------------------------------------------------------------------

    per_device_train_batch_size=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "per_device_train_batch_size"
        ],

    per_device_eval_batch_size=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "per_device_eval_batch_size"
        ],

    gradient_accumulation_steps=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "gradient_accumulation_steps"
        ],

    # ----------------------------------------------------------------------------------------------
    # Optimization
    # ----------------------------------------------------------------------------------------------

    learning_rate=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "learning_rate"
        ],

    lr_scheduler_type=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "lr_scheduler_type"
        ],

    warmup_steps=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "warmup_steps"
        ],

    weight_decay=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "weight_decay"
        ],

    # ----------------------------------------------------------------------------------------------
    # Logging
    # ----------------------------------------------------------------------------------------------

    logging_strategy=
        "steps",

    logging_steps=
        10,

    # ----------------------------------------------------------------------------------------------
    # Formal evaluation/save events are callback-controlled
    # ----------------------------------------------------------------------------------------------

    eval_strategy=
        "no",

    save_strategy=
        "no",

    load_best_model_at_end=
        False,

    # ----------------------------------------------------------------------------------------------
    # Precision
    # ----------------------------------------------------------------------------------------------

    fp16=
        True,

    bf16=
        False,

    # ----------------------------------------------------------------------------------------------
    # Reproducibility
    # ----------------------------------------------------------------------------------------------

    seed=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "seed"
        ],

    data_seed=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "data_seed"
        ],

    # ----------------------------------------------------------------------------------------------
    # Memory
    # ----------------------------------------------------------------------------------------------

    gradient_checkpointing=
        REVISED_STAGE_B_CONFIG[
            "training_protocol"
        ][
            "gradient_checkpointing"
        ],

    # ----------------------------------------------------------------------------------------------
    # External integrations
    # ----------------------------------------------------------------------------------------------

    report_to=
        [],

    # ----------------------------------------------------------------------------------------------
    # Dataset / model interface
    # ----------------------------------------------------------------------------------------------

    remove_unused_columns=
        True,

    label_names=
        [
            "labels"
        ],

    disable_tqdm=
        False,
)


# --------------------------------------------------------------------------------------------------
# 5. Instantiate Stage B Trainer
# --------------------------------------------------------------------------------------------------

stage_b_trainer = Trainer(

    model=
        stage_b_model,

    args=
        stage_b_training_args,

    train_dataset=
        hf_opt_train_model,

    eval_dataset=
        hf_opt_dev_model,

    data_collator=
        data_collator,

    processing_class=
        tokenizer,

    callbacks=[
        stage_b_horizon_callback
    ],
)


# --------------------------------------------------------------------------------------------------
# 6. Trainer infrastructure audit
# --------------------------------------------------------------------------------------------------

stage_b_trainer_checks = {

    "trainer_created":
        stage_b_trainer
        is not None,

    "stage_b_model_attached":
        stage_b_trainer.model
        is stage_b_model,

    "train_n_10831":
        len(
            stage_b_trainer.train_dataset
        )
        == 10_831,

    "dev_n_2708":
        len(
            stage_b_trainer.eval_dataset
        )
        == 2_708,

    "max_steps_500":
        stage_b_training_args.max_steps
        == 500,

    "train_batch_1":
        stage_b_training_args.per_device_train_batch_size
        == 1,

    "eval_batch_1":
        stage_b_training_args.per_device_eval_batch_size
        == 1,

    "gradient_accumulation_8":
        stage_b_training_args.gradient_accumulation_steps
        == 8,

    "effective_batch_8":
        (
            stage_b_training_args.per_device_train_batch_size
            *
            stage_b_training_args.gradient_accumulation_steps
        )
        == 8,

    "learning_rate_1e4":
        abs(
            stage_b_training_args.learning_rate
            - 1e-4
        )
        < 1e-12,

    "linear_scheduler":
        str(
            stage_b_training_args.lr_scheduler_type.value
            if hasattr(
                stage_b_training_args.lr_scheduler_type,
                "value",
            )
            else stage_b_training_args.lr_scheduler_type
        )
        ==
        "linear",

    "warmup_steps_10":
        stage_b_training_args.warmup_steps
        == 10,

    "seed_42":
        stage_b_training_args.seed
        == 42,

    "data_seed_42":
        stage_b_training_args.data_seed
        == 42,

    "fp16_enabled":
        stage_b_training_args.fp16
        is True,

    "periodic_eval_disabled":
        str(
            stage_b_training_args.eval_strategy.value
            if hasattr(
                stage_b_training_args.eval_strategy,
                "value",
            )
            else stage_b_training_args.eval_strategy
        )
        ==
        "no",

    "periodic_save_disabled":
        str(
            stage_b_training_args.save_strategy.value
            if hasattr(
                stage_b_training_args.save_strategy,
                "value",
            )
            else stage_b_training_args.save_strategy
        )
        ==
        "no",

    "formal_horizons_exact":
        sorted(
            stage_b_horizon_callback.target_steps
        )
        ==
        [
            200,
            300,
            400,
            500,
        ],

    "no_eval_triggered_yet":
        len(
            stage_b_horizon_callback.triggered_eval_steps
        )
        == 0,

    "no_save_triggered_yet":
        len(
            stage_b_horizon_callback.triggered_save_steps
        )
        == 0,

    "trainer_global_step_zero":
        int(
            stage_b_trainer.state.global_step
        )
        == 0,

    "model_trainable":
        any(
            parameter.requires_grad
            for parameter
            in stage_b_trainer.model.parameters()
        ),
}


all_stage_b_trainer_checks_pass = bool(
    all(
        stage_b_trainer_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Scheduler pre-construction sanity check
#
# Trainer normally creates optimizer/scheduler when train() begins.
# At this point they should not yet represent a progressed training trajectory.
# --------------------------------------------------------------------------------------------------

optimizer_precreated = (
    getattr(
        stage_b_trainer,
        "optimizer",
        None,
    )
    is not None
)

scheduler_precreated = (
    getattr(
        stage_b_trainer,
        "lr_scheduler",
        None,
    )
    is not None
)


# --------------------------------------------------------------------------------------------------
# 8. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — REVISED STAGE B TRAINER INFRASTRUCTURE")
print("=" * 120)


print("\n[TRAJECTORY]")
print(
    f"Total training steps        : "
    f"{stage_b_training_args.max_steps}"
)
print(
    f"Formal horizons             : "
    f"{STAGE_B_FORMAL_HORIZONS}"
)
print(
    "Evaluation mechanism       : exact-horizon callback"
)
print(
    "Periodic evaluation        : DISABLED"
)
print(
    "Periodic saving            : DISABLED"
)


print("\n[OPTIMIZATION]")
print(
    f"Learning rate              : "
    f"{stage_b_training_args.learning_rate}"
)
print(
    f"LR scheduler               : "
    f"{stage_b_training_args.lr_scheduler_type}"
)
print(
    f"Warmup steps               : "
    f"{stage_b_training_args.warmup_steps}"
)
print(
    f"Effective batch            : "
    f"{stage_b_training_args.per_device_train_batch_size * stage_b_training_args.gradient_accumulation_steps}"
)
print(
    f"Seed                       : "
    f"{stage_b_training_args.seed}"
)


print("\n[PRE-TRAINING STATE]")
print(
    f"Global step                : "
    f"{stage_b_trainer.state.global_step}"
)
print(
    f"Optimizer already created  : "
    f"{optimizer_precreated}"
)
print(
    f"Scheduler already created  : "
    f"{scheduler_precreated}"
)
print(
    f"Triggered eval steps       : "
    f"{stage_b_horizon_callback.triggered_eval_steps}"
)
print(
    f"Triggered save steps       : "
    f"{stage_b_horizon_callback.triggered_save_steps}"
)


print("\n[TRAINER CHECKS]")

for name, passed in stage_b_trainer_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n" + "-" * 120)

if all_stage_b_trainer_checks_pass:

    print(
        "DAY 22 CELL 15 STATUS: PASS — "
        "STAGE B 500-STEP TRAINER READY; TRAINING NOT YET STARTED"
    )

else:

    print(
        "DAY 22 CELL 15 STATUS: FAIL — "
        "DO NOT START STAGE B TRAINING"
    )

print("=" * 120)

DAY 22 — REVISED STAGE B TRAINER INFRASTRUCTURE

[TRAJECTORY]
Total training steps        : 500
Formal horizons             : [200, 300, 400, 500]
Evaluation mechanism       : exact-horizon callback
Periodic evaluation        : DISABLED
Periodic saving            : DISABLED

[OPTIMIZATION]
Learning rate              : 0.0001
LR scheduler               : SchedulerType.LINEAR
Warmup steps               : 10
Effective batch            : 8
Seed                       : 42

[PRE-TRAINING STATE]
Global step                : 0
Optimizer already created  : False
Scheduler already created  : False
Triggered eval steps       : []
Triggered save steps       : []

[TRAINER CHECKS]
PASS   — trainer_created
PASS   — stage_b_model_attached
PASS   — train_n_10831
PASS   — dev_n_2708
PASS   — max_steps_500
PASS   — train_batch_1
PASS   — eval_batch_1
PASS   — gradient_accumulation_8
PASS   — effective_batch_8
PASS   — learning_rate_1e4
PASS   — linear_scheduler
PASS   — warmup_steps_10
PASS   — seed_42


In [16]:
# ==================================================================================================
# DAY 22 — CELL 16
# REVISED STAGE B CONTROLLED 500-STEP TRAINING TRAJECTORY
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from transformers import set_seed


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_B_TRAINING_LOG_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_training_log.csv"
)

STAGE_B_TRAINING_SUMMARY_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_training_summary.json"
)

STAGE_B_TRAINING_COMPLETE_MARKER = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_training_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Safety gate
#
# Prevent accidental second training on the same in-memory Stage B model.
# --------------------------------------------------------------------------------------------------

if STAGE_B_TRAINING_COMPLETE_MARKER.exists():

    raise RuntimeError(
        "DAY 22 STOP: Stage B training-complete marker already exists. "
        "Do not accidentally retrain this trajectory."
    )


# --------------------------------------------------------------------------------------------------
# 3. Pre-training checkpoint cleanliness
# --------------------------------------------------------------------------------------------------

existing_checkpoints = sorted(
    [
        path
        for path
        in STAGE_B_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if path.is_dir()
    ]
)


assert len(
    existing_checkpoints
) == 0, (
    "DAY 22 STOP: Stage B checkpoint directory is not clean."
)


# --------------------------------------------------------------------------------------------------
# 4. Pre-training integrity gate
# --------------------------------------------------------------------------------------------------

pretraining_checks = {

    "trainer_exists":
        stage_b_trainer
        is not None,

    "global_step_zero":
        int(
            stage_b_trainer.state.global_step
        )
        == 0,

    "model_trainable":
        any(
            parameter.requires_grad
            for parameter
            in stage_b_trainer.model.parameters()
        ),

    "train_n_10831":
        len(
            stage_b_trainer.train_dataset
        )
        == 10_831,

    "dev_n_2708":
        len(
            stage_b_trainer.eval_dataset
        )
        == 2_708,

    "max_steps_500":
        stage_b_trainer.args.max_steps
        == 500,

    "callback_horizons_exact":
        sorted(
            stage_b_horizon_callback.target_steps
        )
        ==
        [
            200,
            300,
            400,
            500,
        ],

    "no_eval_triggered_before_training":
        len(
            stage_b_horizon_callback.triggered_eval_steps
        )
        == 0,

    "no_save_triggered_before_training":
        len(
            stage_b_horizon_callback.triggered_save_steps
        )
        == 0,

    "checkpoint_directory_clean":
        len(
            existing_checkpoints
        )
        == 0,

    "historical_validation_firewall":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used"
        ]
        is False,

    "future_firewall":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


assert all(
    pretraining_checks.values()
), (
    "DAY 22 STOP: Stage B pre-training integrity gate failed."
)


# --------------------------------------------------------------------------------------------------
# 5. Re-establish deterministic seed immediately before training
# --------------------------------------------------------------------------------------------------

set_seed(
    REVISED_STAGE_B_CONFIG[
        "training_protocol"
    ][
        "seed"
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. Reset GPU peak-memory accounting
# --------------------------------------------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.reset_peak_memory_stats()


# --------------------------------------------------------------------------------------------------
# 7. Train the complete controlled 500-step trajectory
#
# Formal evaluation/save events:
#
#   step 200
#   step 300
#   step 400
#   step 500
#
# No Historical Validation.
# No Future F1–F6.
# --------------------------------------------------------------------------------------------------

training_start_time = time.time()


stage_b_train_result = (
    stage_b_trainer.train()
)


training_elapsed_seconds = (
    time.time()
    - training_start_time
)


# --------------------------------------------------------------------------------------------------
# 8. Final training state
# --------------------------------------------------------------------------------------------------

final_global_step = int(
    stage_b_trainer.state.global_step
)


final_training_loss = float(
    stage_b_train_result.training_loss
)


if torch.cuda.is_available():

    peak_gpu_allocation_gb = float(
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

else:

    peak_gpu_allocation_gb = None


# --------------------------------------------------------------------------------------------------
# 9. Persist complete Trainer log
# --------------------------------------------------------------------------------------------------

stage_b_log_df = pd.DataFrame(
    stage_b_trainer.state.log_history
)


stage_b_log_df.to_csv(
    STAGE_B_TRAINING_LOG_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 10. Extract formal evaluation observations
# --------------------------------------------------------------------------------------------------

if "eval_loss" in stage_b_log_df.columns:

    stage_b_eval_df = (
        stage_b_log_df.loc[
            stage_b_log_df[
                "eval_loss"
            ].notna()
        ]
        .copy()
        .sort_values(
            "step"
        )
        .reset_index(drop=True)
    )

else:

    stage_b_eval_df = pd.DataFrame()


observed_eval_steps = (
    sorted(
        stage_b_eval_df[
            "step"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if not stage_b_eval_df.empty
    else []
)


# --------------------------------------------------------------------------------------------------
# 11. Recover logged learning-rate trajectory
# --------------------------------------------------------------------------------------------------

if "learning_rate" in stage_b_log_df.columns:

    stage_b_lr_df = (
        stage_b_log_df.loc[
            stage_b_log_df[
                "learning_rate"
            ].notna(),
            [
                "step",
                "learning_rate",
            ],
        ]
        .copy()
        .sort_values(
            "step"
        )
        .reset_index(drop=True)
    )

else:

    stage_b_lr_df = pd.DataFrame(
        columns=[
            "step",
            "learning_rate",
        ]
    )


def get_stage_b_logged_lr(step):

    rows = (
        stage_b_lr_df.loc[
            stage_b_lr_df[
                "step"
            ].astype(int)
            == int(step)
        ]
    )

    if len(rows) == 0:

        return None

    return float(
        rows.iloc[-1][
            "learning_rate"
        ]
    )


lr_step_10 = get_stage_b_logged_lr(
    10
)

lr_step_100 = get_stage_b_logged_lr(
    100
)

lr_step_200 = get_stage_b_logged_lr(
    200
)

lr_step_300 = get_stage_b_logged_lr(
    300
)

lr_step_400 = get_stage_b_logged_lr(
    400
)

lr_step_500 = get_stage_b_logged_lr(
    500
)


# --------------------------------------------------------------------------------------------------
# 12. Inspect final live scheduler state
# --------------------------------------------------------------------------------------------------

stage_b_scheduler_available = (
    getattr(
        stage_b_trainer,
        "lr_scheduler",
        None,
    )
    is not None
)


if stage_b_scheduler_available:

    final_scheduler_epoch = int(
        stage_b_trainer.lr_scheduler.last_epoch
    )

    final_scheduler_lrs = [
        float(x)
        for x
        in stage_b_trainer.lr_scheduler.get_last_lr()
    ]

else:

    final_scheduler_epoch = None
    final_scheduler_lrs = []


# --------------------------------------------------------------------------------------------------
# 13. Formal checkpoint audit
# --------------------------------------------------------------------------------------------------

FORMAL_HORIZONS = [
    200,
    300,
    400,
    500,
]


checkpoint_status = {}


for step in FORMAL_HORIZONS:

    checkpoint_path = (
        STAGE_B_CHECKPOINT_DIR
        / f"checkpoint-{step}"
    )

    checkpoint_status[
        step
    ] = {

        "path":
            str(
                checkpoint_path
            ),

        "exists":
            bool(
                checkpoint_path.exists()
            ),

        "adapter_config_exists":
            bool(
                (
                    checkpoint_path
                    / "adapter_config.json"
                ).exists()
            ),

        "adapter_weights_exist":
            bool(
                (
                    checkpoint_path
                    / "adapter_model.safetensors"
                ).exists()
                or
                (
                    checkpoint_path
                    / "adapter_model.bin"
                ).exists()
            ),

        "trainer_state_exists":
            bool(
                (
                    checkpoint_path
                    / "trainer_state.json"
                ).exists()
            ),

        "optimizer_state_exists":
            bool(
                (
                    checkpoint_path
                    / "optimizer.pt"
                ).exists()
            ),

        "scheduler_state_exists":
            bool(
                (
                    checkpoint_path
                    / "scheduler.pt"
                ).exists()
            ),
    }


# --------------------------------------------------------------------------------------------------
# 14. Callback event audit
# --------------------------------------------------------------------------------------------------

triggered_eval_steps = sorted(
    set(
        int(x)
        for x
        in stage_b_horizon_callback.triggered_eval_steps
    )
)


triggered_save_steps = sorted(
    set(
        int(x)
        for x
        in stage_b_horizon_callback.triggered_save_steps
    )
)


# --------------------------------------------------------------------------------------------------
# 15. Formal post-training checks
# --------------------------------------------------------------------------------------------------

posttraining_checks = {

    "training_completed_500_steps":
        final_global_step
        == 500,

    "training_loss_finite":
        bool(
            np.isfinite(
                final_training_loss
            )
        ),

    "training_log_nonempty":
        len(
            stage_b_log_df
        )
        > 0,

    # Exact evaluation protocol
    "observed_eval_steps_exact":
        observed_eval_steps
        ==
        [
            200,
            300,
            400,
            500,
        ],

    "callback_eval_steps_exact":
        triggered_eval_steps
        ==
        [
            200,
            300,
            400,
            500,
        ],

    "callback_save_steps_exact":
        triggered_save_steps
        ==
        [
            200,
            300,
            400,
            500,
        ],

    # Explicitly confirm there was no unwanted step-100 evaluation
    "no_step_100_evaluation":
        100
        not in observed_eval_steps,

    # Scheduler integrity
    "lr_step_200_observed":
        lr_step_200
        is not None,

    "lr_step_300_observed":
        lr_step_300
        is not None,

    "lr_step_400_observed":
        lr_step_400
        is not None,

    "lr_step_200_positive":
        (
            lr_step_200
            is not None
            and
            lr_step_200
            > 0
        ),

    "lr_step_300_positive":
        (
            lr_step_300
            is not None
            and
            lr_step_300
            > 0
        ),

    "lr_step_400_positive":
        (
            lr_step_400
            is not None
            and
            lr_step_400
            > 0
        ),

    "scheduler_reaches_end_at_500":
        (
            stage_b_scheduler_available
            and
            final_scheduler_epoch
            == 500
        ),

    # Checkpoints
    "checkpoint_200_complete":
        all(
            [
                checkpoint_status[
                    200
                ][
                    "exists"
                ],
                checkpoint_status[
                    200
                ][
                    "adapter_config_exists"
                ],
                checkpoint_status[
                    200
                ][
                    "adapter_weights_exist"
                ],
            ]
        ),

    "checkpoint_300_complete":
        all(
            [
                checkpoint_status[
                    300
                ][
                    "exists"
                ],
                checkpoint_status[
                    300
                ][
                    "adapter_config_exists"
                ],
                checkpoint_status[
                    300
                ][
                    "adapter_weights_exist"
                ],
            ]
        ),

    "checkpoint_400_complete":
        all(
            [
                checkpoint_status[
                    400
                ][
                    "exists"
                ],
                checkpoint_status[
                    400
                ][
                    "adapter_config_exists"
                ],
                checkpoint_status[
                    400
                ][
                    "adapter_weights_exist"
                ],
            ]
        ),

    "checkpoint_500_complete":
        all(
            [
                checkpoint_status[
                    500
                ][
                    "exists"
                ],
                checkpoint_status[
                    500
                ][
                    "adapter_config_exists"
                ],
                checkpoint_status[
                    500
                ][
                    "adapter_weights_exist"
                ],
            ]
        ),

    "historical_validation_not_used":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used"
        ]
        is False,

    "future_data_not_used":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


all_posttraining_checks_pass = bool(
    all(
        posttraining_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 16. Compact formal horizon table
# --------------------------------------------------------------------------------------------------

formal_horizon_table = (
    stage_b_eval_df.loc[
        stage_b_eval_df[
            "step"
        ].astype(int)
        .isin(
            FORMAL_HORIZONS
        )
    ]
    .copy()
)


formal_horizon_table[
    "step"
] = (
    formal_horizon_table[
        "step"
    ]
    .astype(int)
)


formal_horizon_table = (
    formal_horizon_table
    .sort_values(
        "step"
    )
    .reset_index(drop=True)
)


formal_horizon_table[
    "learning_rate"
] = (
    formal_horizon_table[
        "step"
    ].map(
        {
            200:
                lr_step_200,

            300:
                lr_step_300,

            400:
                lr_step_400,

            500:
                lr_step_500,
        }
    )
)


# --------------------------------------------------------------------------------------------------
# 17. Save formal training summary
# --------------------------------------------------------------------------------------------------

stage_b_training_summary = {

    "day":
        22,

    "stage":
        "B",

    "artifact":
        "stage_b_training_summary",

    "protocol":
        "controlled_common_500_step_trajectory",

    "status":
        (
            "PASS"
            if all_posttraining_checks_pass
            else
            "FAIL"
        ),

    "training": {

        "final_global_step":
            final_global_step,

        "final_training_loss":
            final_training_loss,

        "runtime_seconds":
            float(
                training_elapsed_seconds
            ),

        "runtime_minutes":
            float(
                training_elapsed_seconds
                / 60
            ),

        "peak_gpu_allocation_gb":
            peak_gpu_allocation_gb,
    },

    "formal_horizons": {

        "expected":
            FORMAL_HORIZONS,

        "observed_eval_steps":
            observed_eval_steps,

        "callback_eval_steps":
            triggered_eval_steps,

        "callback_save_steps":
            triggered_save_steps,
    },

    "learning_rate_trajectory": {

        "scheduler_type":
            "linear",

        "base_learning_rate":
            float(
                stage_b_training_args.learning_rate
            ),

        "warmup_steps":
            int(
                stage_b_training_args.warmup_steps
            ),

        "step_10":
            lr_step_10,

        "step_100":
            lr_step_100,

        "step_200":
            lr_step_200,

        "step_300":
            lr_step_300,

        "step_400":
            lr_step_400,

        "step_500":
            lr_step_500,

        "final_scheduler_epoch":
            final_scheduler_epoch,

        "final_scheduler_lrs":
            final_scheduler_lrs,
    },

    "checkpoints":
        checkpoint_status,

    "checks": {
        key:
            bool(
                value
            )
        for key, value
        in posttraining_checks.items()
    },

    "all_checks_pass":
        all_posttraining_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_B_TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_b_training_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 18. Write completion marker only after full integrity PASS
# --------------------------------------------------------------------------------------------------

if all_posttraining_checks_pass:

    with open(
        STAGE_B_TRAINING_COMPLETE_MARKER,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            {
                "day":
                    22,

                "stage":
                    "B",

                "status":
                    "CONTROLLED_500_STEP_TRAJECTORY_COMPLETE",

                "global_step":
                    final_global_step,

                "formal_horizons":
                    FORMAL_HORIZONS,

                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            },
            f,
            indent=2,
        )


# --------------------------------------------------------------------------------------------------
# 19. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — REVISED STAGE B CONTROLLED 500-STEP TRAINING")
print("=" * 120)


print("\n[TRAINING]")
print(
    f"Final global step          : "
    f"{final_global_step}"
)
print(
    f"Final training loss        : "
    f"{final_training_loss:.6f}"
)
print(
    f"Runtime                    : "
    f"{training_elapsed_seconds / 60:.2f} minutes"
)

if peak_gpu_allocation_gb is not None:

    print(
        f"Peak GPU allocation        : "
        f"{peak_gpu_allocation_gb:.2f} GB"
    )


print("\n[FORMAL HORIZON RESULTS]")

display_columns = [
    column
    for column
    in [
        "step",
        "eval_loss",
        "learning_rate",
        "eval_runtime",
    ]
    if column
    in formal_horizon_table.columns
]

display(
    formal_horizon_table[
        display_columns
    ]
)


print("\n[LEARNING-RATE TRAJECTORY]")
print(
    f"Step 10                    : "
    f"{lr_step_10}"
)
print(
    f"Step 100                   : "
    f"{lr_step_100}"
)
print(
    f"Step 200                   : "
    f"{lr_step_200}"
)
print(
    f"Step 300                   : "
    f"{lr_step_300}"
)
print(
    f"Step 400                   : "
    f"{lr_step_400}"
)
print(
    f"Step 500                   : "
    f"{lr_step_500}"
)
print(
    f"Final scheduler epoch      : "
    f"{final_scheduler_epoch}"
)
print(
    f"Final scheduler LR         : "
    f"{final_scheduler_lrs}"
)


print("\n[CALLBACK EVENTS]")
print(
    f"Observed eval steps        : "
    f"{observed_eval_steps}"
)
print(
    f"Callback eval steps        : "
    f"{triggered_eval_steps}"
)
print(
    f"Callback save steps        : "
    f"{triggered_save_steps}"
)


print("\n[CHECKPOINTS]")

for step in FORMAL_HORIZONS:

    status = (
        checkpoint_status[
            step
        ]
    )

    complete = (
        status[
            "exists"
        ]
        and
        status[
            "adapter_config_exists"
        ]
        and
        status[
            "adapter_weights_exist"
        ]
    )

    print(
        f"{'PASS' if complete else 'FAIL':<6} — "
        f"checkpoint-{step}"
    )


print("\n[POST-TRAINING CHECKS]")

for name, passed in posttraining_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Training log              : "
    f"{STAGE_B_TRAINING_LOG_PATH}"
)
print(
    f"Training summary          : "
    f"{STAGE_B_TRAINING_SUMMARY_PATH}"
)
print(
    f"Checkpoint directory      : "
    f"{STAGE_B_CHECKPOINT_DIR}"
)


print("\n" + "-" * 120)

if all_posttraining_checks_pass:

    print(
        "DAY 22 CELL 16 STATUS: PASS — "
        "CONTROLLED 500-STEP TRAJECTORY COMPLETE; "
        "HORIZON SELECTION PENDING"
    )

else:

    print(
        "DAY 22 CELL 16 STATUS: FAIL — "
        "DO NOT SELECT STAGE B HORIZON"
    )

print("=" * 120)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


DAY 22 — REVISED STAGE B CONTROLLED 500-STEP TRAINING

[TRAINING]
Final global step          : 500
Final training loss        : 0.457656
Runtime                    : 87.23 minutes
Peak GPU allocation        : 8.81 GB

[FORMAL HORIZON RESULTS]


,step,eval_loss,learning_rate,eval_runtime
0,200,0.266584,6.142857e-05,515.4582
1,300,0.219981,4.102041e-05,515.7255
2,400,0.219206,2.061224e-05,515.1818
3,500,0.215775,2.040816e-07,514.4298



[LEARNING-RATE TRAJECTORY]
Step 10                    : 9e-05
Step 100                   : 8.183673469387756e-05
Step 200                   : 6.142857142857143e-05
Step 300                   : 4.102040816326531e-05
Step 400                   : 2.0612244897959186e-05
Step 500                   : 2.0408163265306124e-07
Final scheduler epoch      : 500
Final scheduler LR         : [0.0, 0.0]

[CALLBACK EVENTS]
Observed eval steps        : [200, 300, 400, 500]
Callback eval steps        : [200, 300, 400, 500]
Callback save steps        : [200, 300, 400, 500]

[CHECKPOINTS]
PASS   — checkpoint-200
PASS   — checkpoint-300
PASS   — checkpoint-400
PASS   — checkpoint-500

[POST-TRAINING CHECKS]
PASS   — training_completed_500_steps
PASS   — training_loss_finite
PASS   — training_log_nonempty
PASS   — observed_eval_steps_exact
PASS   — callback_eval_steps_exact
PASS   — callback_save_steps_exact
PASS   — no_step_100_evaluation
PASS   — lr_step_200_observed
PASS   — lr_step_300_observed
PASS   

In [17]:
# ==================================================================================================
# DAY 22 — CELL 17
# STAGE B FORMAL HORIZON SELECTION + OPTIONAL 600-STEP EXTENSION DECISION
# ==================================================================================================

import json
import shutil
from datetime import datetime, timezone

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_B_HORIZON_SELECTION_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_horizon_selection.csv"
)

STAGE_B_SELECTION_SUMMARY_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_horizon_selection_summary.json"
)

STAGE_B_SELECTED_MODEL_DIR = (
    STAGE_B_MODEL_DIR
    / "selected_adapter"
)


# --------------------------------------------------------------------------------------------------
# 2. Load Stage B training log
# --------------------------------------------------------------------------------------------------

stage_b_log_df = pd.read_csv(
    STAGE_B_TRAINING_LOG_PATH
)


FORMAL_HORIZONS = [
    200,
    300,
    400,
    500,
]


# --------------------------------------------------------------------------------------------------
# 3. Extract exactly one formal eval-loss observation per horizon
# --------------------------------------------------------------------------------------------------

formal_selection_table = (
    stage_b_log_df.loc[
        stage_b_log_df["step"].isin(
            FORMAL_HORIZONS
        )
        &
        stage_b_log_df["eval_loss"].notna(),
        [
            "step",
            "eval_loss",
        ],
    ]
    .copy()
)


formal_selection_table[
    "step"
] = (
    formal_selection_table[
        "step"
    ]
    .astype(int)
)


formal_selection_table = (
    formal_selection_table
    .sort_values(
        "step"
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 4. Candidate integrity
# --------------------------------------------------------------------------------------------------

candidate_counts = (
    formal_selection_table[
        "step"
    ]
    .value_counts()
)


candidate_checks = {

    "candidate_n_4":
        len(
            formal_selection_table
        )
        == 4,

    "candidate_horizons_exact":
        formal_selection_table[
            "step"
        ].tolist()
        ==
        [
            200,
            300,
            400,
            500,
        ],

    "one_eval_per_horizon":
        bool(
            (
                candidate_counts
                == 1
            ).all()
        ),

    "eval_losses_complete":
        formal_selection_table[
            "eval_loss"
        ]
        .notna()
        .all(),
}


assert all(
    candidate_checks.values()
), (
    "DAY 22 STOP: Stage B formal horizon table failed integrity checks."
)


# --------------------------------------------------------------------------------------------------
# 5. Formal Stage B selection
#
# Frozen rule:
#
#   minimize Optimization Dev eval_loss
#
# tie break:
#
#   shorter training horizon
# --------------------------------------------------------------------------------------------------

ranked_table = (
    formal_selection_table
    .sort_values(
        [
            "eval_loss",
            "step",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


ranked_table[
    "rank_by_eval_loss"
] = range(
    1,
    len(
        ranked_table
    )
    + 1
)


selected_step = int(
    ranked_table.iloc[0][
        "step"
    ]
)

selected_eval_loss = float(
    ranked_table.iloc[0][
        "eval_loss"
    ]
)


formal_selection_table = (
    formal_selection_table
    .merge(
        ranked_table[
            [
                "step",
                "rank_by_eval_loss",
            ]
        ],
        on="step",
        how="left",
    )
)


formal_selection_table[
    "selected"
] = (
    formal_selection_table[
        "step"
    ]
    ==
    selected_step
)


# --------------------------------------------------------------------------------------------------
# 6. Improvement diagnostics
# --------------------------------------------------------------------------------------------------

loss_by_step = {

    int(row.step):
        float(row.eval_loss)

    for row
    in formal_selection_table.itertuples()
}


loss_200 = loss_by_step[
    200
]

loss_300 = loss_by_step[
    300
]

loss_400 = loss_by_step[
    400
]

loss_500 = loss_by_step[
    500
]


def improvement(old_loss, new_loss):

    absolute = (
        old_loss
        - new_loss
    )

    relative = (
        absolute
        / old_loss
    )

    return (
        float(
            absolute
        ),
        float(
            relative
        ),
    )


imp_200_300_abs, imp_200_300_rel = improvement(
    loss_200,
    loss_300,
)

imp_300_400_abs, imp_300_400_rel = improvement(
    loss_300,
    loss_400,
)

imp_400_500_abs, imp_400_500_rel = improvement(
    loss_400,
    loss_500,
)

imp_200_500_abs, imp_200_500_rel = improvement(
    loss_200,
    loss_500,
)


# --------------------------------------------------------------------------------------------------
# 7. Upper-bound diagnostic
# --------------------------------------------------------------------------------------------------

selected_at_upper_boundary = (
    selected_step
    ==
    max(
        FORMAL_HORIZONS
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Optional 600-step decision
#
# IMPORTANT METHODOLOGICAL RULE:
#
# The frozen protocol did NOT define a numerical threshold for what constitutes
# "material continued improvement".
#
# Therefore we must NOT invent a post-hoc threshold after observing the results.
#
# In addition, the 500-step linear scheduler is exhausted at step 500.
# A scientifically clean 600-step experiment would require a NEW common
# 600-step trajectory from fresh initialization; checkpoint-500 cannot simply
# be resumed and interpreted as the same controlled horizon experiment.
#
# Decision:
#   - select step 500 as Stage B winner;
#   - do NOT automatically activate 600;
#   - preserve 600 only as a possible separate sensitivity experiment.
# --------------------------------------------------------------------------------------------------

OPTIONAL_600_AUTOMATICALLY_ACTIVATED = False

OPTIONAL_600_DECISION = (
    "NOT_ACTIVATED"
)

OPTIONAL_600_REASON = (
    "Step 500 is the best formal Stage B horizon, but the pre-frozen protocol "
    "did not specify a numerical threshold for 'material' boundary improvement. "
    "Introducing such a threshold after observing the results would be post-hoc. "
    "Furthermore, the common 500-step linear scheduler is exhausted at step 500; "
    "a clean 600-step comparison would require a new common 600-step trajectory "
    "from fresh initialization rather than continuation from checkpoint-500."
)


# --------------------------------------------------------------------------------------------------
# 9. Validate selected checkpoint
# --------------------------------------------------------------------------------------------------

selected_checkpoint_dir = (
    STAGE_B_CHECKPOINT_DIR
    / f"checkpoint-{selected_step}"
)


selected_checkpoint_checks = {

    "selected_checkpoint_exists":
        selected_checkpoint_dir.exists(),

    "selected_adapter_config_exists":
        (
            selected_checkpoint_dir
            / "adapter_config.json"
        ).exists(),

    "selected_adapter_weights_exist":
        (
            (
                selected_checkpoint_dir
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                selected_checkpoint_dir
                / "adapter_model.bin"
            ).exists()
        ),
}


assert all(
    selected_checkpoint_checks.values()
), (
    "DAY 22 STOP: selected Stage B checkpoint is incomplete."
)


# --------------------------------------------------------------------------------------------------
# 10. Freeze selected adapter
# --------------------------------------------------------------------------------------------------

if STAGE_B_SELECTED_MODEL_DIR.exists():

    shutil.rmtree(
        STAGE_B_SELECTED_MODEL_DIR
    )


shutil.copytree(
    selected_checkpoint_dir,
    STAGE_B_SELECTED_MODEL_DIR,
)


# --------------------------------------------------------------------------------------------------
# 11. Frozen adapter checks
# --------------------------------------------------------------------------------------------------

frozen_adapter_checks = {

    "selected_adapter_dir_exists":
        STAGE_B_SELECTED_MODEL_DIR.exists(),

    "frozen_adapter_config_exists":
        (
            STAGE_B_SELECTED_MODEL_DIR
            / "adapter_config.json"
        ).exists(),

    "frozen_adapter_weights_exist":
        (
            (
                STAGE_B_SELECTED_MODEL_DIR
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                STAGE_B_SELECTED_MODEL_DIR
                / "adapter_model.bin"
            ).exists()
        ),
}


# --------------------------------------------------------------------------------------------------
# 12. Formal selection checks
# --------------------------------------------------------------------------------------------------

selection_checks = {

    **candidate_checks,

    **selected_checkpoint_checks,

    **frozen_adapter_checks,

    "selection_metric_eval_loss":
        REVISED_STAGE_B_CONFIG[
            "selection_protocol"
        ][
            "primary_metric"
        ]
        ==
        "eval_loss",

    "selection_direction_minimize":
        REVISED_STAGE_B_CONFIG[
            "selection_protocol"
        ][
            "direction"
        ]
        ==
        "minimize",

    "selected_step_is_argmin":
        selected_step
        ==
        int(
            formal_selection_table.loc[
                formal_selection_table[
                    "eval_loss"
                ].idxmin(),
                "step",
            ]
        ),

    "selected_step_500":
        selected_step
        == 500,

    "selected_at_upper_boundary":
        selected_at_upper_boundary,

    "loss_400_to_500_improved":
        imp_400_500_abs
        > 0,

    "historical_validation_not_used":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used"
        ]
        is False,

    "future_not_used":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ]
        is False,

    "600_not_automatically_activated":
        OPTIONAL_600_AUTOMATICALLY_ACTIVATED
        is False,
}


all_selection_checks_pass = bool(
    all(
        selection_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Save formal horizon table
# --------------------------------------------------------------------------------------------------

formal_selection_table.to_csv(
    STAGE_B_HORIZON_SELECTION_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 14. Save formal summary
# --------------------------------------------------------------------------------------------------

stage_b_selection_summary = {

    "day":
        22,

    "stage":
        "B",

    "artifact":
        "stage_b_horizon_selection_summary",

    "selection_protocol": {

        "candidate_horizons":
            FORMAL_HORIZONS,

        "selection_split":
            "optimization_dev",

        "metric":
            "eval_loss",

        "direction":
            "minimize",

        "tie_break":
            "shorter_training_horizon",
    },

    "selected_horizon": {

        "step":
            selected_step,

        "eval_loss":
            selected_eval_loss,

        "source_checkpoint":
            str(
                selected_checkpoint_dir
            ),

        "frozen_adapter":
            str(
                STAGE_B_SELECTED_MODEL_DIR
            ),
    },

    "horizon_diagnostics": {

        "eval_loss_200":
            loss_200,

        "eval_loss_300":
            loss_300,

        "eval_loss_400":
            loss_400,

        "eval_loss_500":
            loss_500,

        "improvement_200_to_300_absolute":
            imp_200_300_abs,

        "improvement_200_to_300_relative":
            imp_200_300_rel,

        "improvement_300_to_400_absolute":
            imp_300_400_abs,

        "improvement_300_to_400_relative":
            imp_300_400_rel,

        "improvement_400_to_500_absolute":
            imp_400_500_abs,

        "improvement_400_to_500_relative":
            imp_400_500_rel,

        "improvement_200_to_500_absolute":
            imp_200_500_abs,

        "improvement_200_to_500_relative":
            imp_200_500_rel,

        "selected_at_upper_boundary":
            selected_at_upper_boundary,
    },

    "optional_600_extension": {

        "status":
            OPTIONAL_600_DECISION,

        "automatically_activated":
            OPTIONAL_600_AUTOMATICALLY_ACTIVATED,

        "reason":
            OPTIONAL_600_REASON,

        "scientifically_clean_requirement":
            (
                "If later performed, train a fresh common 600-step trajectory "
                "with the same initialization protocol and evaluate formal "
                "horizons under that shared schedule."
            ),
    },

    "checks": {
        key:
            bool(
                value
            )

        for key, value
        in selection_checks.items()
    },

    "all_checks_pass":
        all_selection_checks_pass,

    "freeze_status":
        (
            "STAGE_B_HORIZON_500_FROZEN"
            if all_selection_checks_pass
            else
            "STAGE_B_SELECTION_FAILED"
        ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_B_SELECTION_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_b_selection_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE B FORMAL HORIZON SELECTION")
print("=" * 120)


print("\n[FORMAL CANDIDATES]")

display(
    formal_selection_table[
        [
            "step",
            "eval_loss",
            "rank_by_eval_loss",
            "selected",
        ]
    ]
)


print("\n[SELECTED HORIZON]")
print(
    f"Selected step              : "
    f"{selected_step}"
)
print(
    f"Selected eval loss         : "
    f"{selected_eval_loss:.6f}"
)
print(
    f"Frozen adapter             : "
    f"{STAGE_B_SELECTED_MODEL_DIR}"
)


print("\n[IMPROVEMENT DIAGNOSTICS]")
print(
    f"200 → 300 absolute         : "
    f"{imp_200_300_abs:.6f}"
)
print(
    f"200 → 300 relative         : "
    f"{imp_200_300_rel:.2%}"
)

print(
    f"300 → 400 absolute         : "
    f"{imp_300_400_abs:.6f}"
)
print(
    f"300 → 400 relative         : "
    f"{imp_300_400_rel:.2%}"
)

print(
    f"400 → 500 absolute         : "
    f"{imp_400_500_abs:.6f}"
)
print(
    f"400 → 500 relative         : "
    f"{imp_400_500_rel:.2%}"
)

print(
    f"200 → 500 absolute         : "
    f"{imp_200_500_abs:.6f}"
)
print(
    f"200 → 500 relative         : "
    f"{imp_200_500_rel:.2%}"
)


print("\n[600-STEP EXTENSION DECISION]")
print(
    f"500 is upper-bound winner  : "
    f"{selected_at_upper_boundary}"
)
print(
    f"Automatic 600 extension    : "
    f"{OPTIONAL_600_AUTOMATICALLY_ACTIVATED}"
)
print(
    f"Decision                   : "
    f"{OPTIONAL_600_DECISION}"
)
print(
    "Reason                     : no pre-specified numerical "
    "materiality threshold; 600 would require a new common trajectory"
)


print("\n[SELECTION CHECKS]")

for name, passed in selection_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Horizon table              : "
    f"{STAGE_B_HORIZON_SELECTION_PATH}"
)
print(
    f"Selection summary          : "
    f"{STAGE_B_SELECTION_SUMMARY_PATH}"
)
print(
    f"Frozen adapter             : "
    f"{STAGE_B_SELECTED_MODEL_DIR}"
)


print("\n" + "-" * 120)

if all_selection_checks_pass:

    print(
        "DAY 22 CELL 17 STATUS: PASS — "
        "STAGE B 500-STEP HORIZON FROZEN; "
        "600-STEP EXTENSION NOT ACTIVATED"
    )

else:

    print(
        "DAY 22 CELL 17 STATUS: FAIL — "
        "DO NOT PROCEED TO STAGE C"
    )

print("=" * 120)

DAY 22 — STAGE B FORMAL HORIZON SELECTION

[FORMAL CANDIDATES]


,step,eval_loss,rank_by_eval_loss,selected
0,200,0.266584,4,False
1,300,0.219981,3,False
2,400,0.219206,2,False
3,500,0.215775,1,True



[SELECTED HORIZON]
Selected step              : 500
Selected eval loss         : 0.215775
Frozen adapter             : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_b_horizon_audit/model/selected_adapter

[IMPROVEMENT DIAGNOSTICS]
200 → 300 absolute         : 0.046603
200 → 300 relative         : 17.48%
300 → 400 absolute         : 0.000774
300 → 400 relative         : 0.35%
400 → 500 absolute         : 0.003431
400 → 500 relative         : 1.57%
200 → 500 absolute         : 0.050809
200 → 500 relative         : 19.06%

[600-STEP EXTENSION DECISION]
500 is upper-bound winner  : True
Automatic 600 extension    : False
Decision                   : NOT_ACTIVATED
Reason                     : no pre-specified numerical materiality threshold; 600 would require a new common trajectory

[SELECTION CHECKS]
PASS   — candidate_n_4
PASS   — candidate_horizons_exact
PASS   — one_eval_per_horizon
PASS   — eval_losses_complete
PASS   — selected_checkpoint_exists
PASS   — selected_ad

In [18]:
# ==================================================================================================
# DAY 22 — CELL 18
# STAGE B-500 OPTIMIZATION DEV CLASSIFICATION INFERENCE + METRICS
# ==================================================================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_B_DEV_PREDICTIONS_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_500_optimization_dev_predictions.csv"
)

STAGE_B_DEV_METRICS_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_500_optimization_dev_metrics.json"
)

STAGE_B_DEV_CLASS_METRICS_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_500_optimization_dev_class_metrics.csv"
)

STAGE_B_DEV_CONFUSION_PATH = (
    STAGE_B_ARTIFACT_DIR
    / "stage_b_500_optimization_dev_confusion_matrix.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Inference protocol
# --------------------------------------------------------------------------------------------------

INFERENCE_BATCH_SIZE = 8

MAX_NEW_TOKENS = 4

VALID_PREDICTIONS = [
    "negative",
    "neutral",
    "positive",
]


# --------------------------------------------------------------------------------------------------
# 3. Formal safety gate
# --------------------------------------------------------------------------------------------------

inference_prechecks = {

    "stage_b_selected_step_500":
        selected_step == 500,

    "dev_n_2708":
        len(
            optimization_dev
        ) == 2_708,

    "historical_validation_not_used":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "historical_validation_used"
        ] is False,

    "future_not_used":
        REVISED_STAGE_B_CONFIG[
            "data_protocol"
        ][
            "future_f1_f6_used"
        ] is False,

    "stage_b_model_available":
        stage_b_model is not None,
}


assert all(
    inference_prechecks.values()
), (
    "DAY 22 STOP: Stage B inference prechecks failed."
)


# --------------------------------------------------------------------------------------------------
# 4. Prepare model for inference
# --------------------------------------------------------------------------------------------------

stage_b_model.eval()

stage_b_model.config.use_cache = True


tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token


# --------------------------------------------------------------------------------------------------
# 5. Build exact inference prompts
# --------------------------------------------------------------------------------------------------

dev_inference_frame = (
    optimization_dev[
        [
            "observation_id",
            "channel_username",
            "message_id",
            "date",
            "text",
            "sentiment",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


dev_prompts = [
    build_inference_prompt(
        text
    )
    for text
    in dev_inference_frame[
        "text"
    ].tolist()
]


# --------------------------------------------------------------------------------------------------
# 6. Greedy batched inference
# --------------------------------------------------------------------------------------------------

raw_predictions = []


for start_idx in range(
    0,
    len(
        dev_prompts
    ),
    INFERENCE_BATCH_SIZE,
):

    end_idx = min(
        start_idx
        + INFERENCE_BATCH_SIZE,
        len(
            dev_prompts
        ),
    )

    batch_prompts = (
        dev_prompts[
            start_idx:end_idx
        ]
    )

    encoded = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )


    encoded = {
        key:
            value.to(
                stage_b_model.device
            )
        for key, value
        in encoded.items()
    }


    input_length = (
        encoded[
            "input_ids"
        ].shape[1]
    )


    with torch.inference_mode():

        generated = stage_b_model.generate(

            **encoded,

            max_new_tokens=
                MAX_NEW_TOKENS,

            do_sample=
                False,

            pad_token_id=
                tokenizer.eos_token_id,

            eos_token_id=
                tokenizer.eos_token_id,
        )


    generated_only = (
        generated[
            :,
            input_length:
        ]
    )


    decoded = tokenizer.batch_decode(

        generated_only,

        skip_special_tokens=True,
    )


    raw_predictions.extend(
        [
            text.strip().lower()
            for text
            in decoded
        ]
    )


# --------------------------------------------------------------------------------------------------
# 7. Prediction validity
# --------------------------------------------------------------------------------------------------

assert len(
    raw_predictions
) == len(
    dev_inference_frame
)


dev_predictions = (
    dev_inference_frame
    .copy()
)


dev_predictions[
    "prediction_raw"
] = raw_predictions


dev_predictions[
    "prediction_valid"
] = (
    dev_predictions[
        "prediction_raw"
    ]
    .isin(
        VALID_PREDICTIONS
    )
)


dev_predictions[
    "prediction"
] = (
    dev_predictions[
        "prediction_raw"
    ]
    .where(
        dev_predictions[
            "prediction_valid"
        ],
        other="INVALID",
    )
)


invalid_prediction_n = int(
    (
        ~dev_predictions[
            "prediction_valid"
        ]
    ).sum()
)


invalid_prediction_rate = float(
    invalid_prediction_n
    / len(
        dev_predictions
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Metrics
#
# Invalid generations are counted as incorrect predictions.
# --------------------------------------------------------------------------------------------------

y_true = (
    dev_predictions[
        "sentiment"
    ]
    .astype(str)
    .tolist()
)


y_pred = (
    dev_predictions[
        "prediction"
    ]
    .astype(str)
    .tolist()
)


accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)


macro_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=
            VALID_PREDICTIONS,
        average="macro",
        zero_division=0,
    )
)


weighted_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=
            VALID_PREDICTIONS,
        average="weighted",
        zero_division=0,
    )
)


precision, recall, class_f1, support = (
    precision_recall_fscore_support(

        y_true,
        y_pred,

        labels=
            VALID_PREDICTIONS,

        zero_division=0,
    )
)


class_metrics = pd.DataFrame(
    {
        "label":
            VALID_PREDICTIONS,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            class_f1,

        "support":
            support.astype(int),
    }
)


negative_recall = float(
    class_metrics.loc[
        class_metrics[
            "label"
        ] == "negative",
        "recall",
    ].iloc[0]
)


neutral_recall = float(
    class_metrics.loc[
        class_metrics[
            "label"
        ] == "neutral",
        "recall",
    ].iloc[0]
)


positive_recall = float(
    class_metrics.loc[
        class_metrics[
            "label"
        ] == "positive",
        "recall",
    ].iloc[0]
)


# --------------------------------------------------------------------------------------------------
# 9. Confusion matrix
# --------------------------------------------------------------------------------------------------

CONFUSION_LABELS = [
    "negative",
    "neutral",
    "positive",
    "INVALID",
]


cm = confusion_matrix(

    y_true,
    y_pred,

    labels=
        CONFUSION_LABELS,
)


confusion_df = pd.DataFrame(

    cm,

    index=[
        f"true_{label}"
        for label
        in CONFUSION_LABELS
    ],

    columns=[
        f"pred_{label}"
        for label
        in CONFUSION_LABELS
    ],
)


# --------------------------------------------------------------------------------------------------
# 10. Save artifacts
# --------------------------------------------------------------------------------------------------

dev_predictions.to_csv(
    STAGE_B_DEV_PREDICTIONS_PATH,
    index=False,
)


class_metrics.to_csv(
    STAGE_B_DEV_CLASS_METRICS_PATH,
    index=False,
)


confusion_df.to_csv(
    STAGE_B_DEV_CONFUSION_PATH,
    index=True,
)


stage_b_dev_metrics = {

    "day":
        22,

    "stage":
        "B",

    "checkpoint_step":
        500,

    "evaluation_population":
        "optimization_dev",

    "n":
        int(
            len(
                dev_predictions
            )
        ),

    "accuracy":
        accuracy,

    "macro_f1":
        macro_f1,

    "weighted_f1":
        weighted_f1,

    "class_recall": {

        "negative":
            negative_recall,

        "neutral":
            neutral_recall,

        "positive":
            positive_recall,
    },

    "invalid_prediction_n":
        invalid_prediction_n,

    "invalid_prediction_rate":
        invalid_prediction_rate,

    "inference_protocol": {

        "batch_size":
            INFERENCE_BATCH_SIZE,

        "max_new_tokens":
            MAX_NEW_TOKENS,

        "do_sample":
            False,

        "padding_side":
            "left",

        "valid_predictions":
            VALID_PREDICTIONS,
    },

    "firewall": {

        "historical_validation_used":
            False,

        "future_f1_f6_used":
            False,
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_B_DEV_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_b_dev_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 11. Formal integrity checks
# --------------------------------------------------------------------------------------------------

inference_checks = {

    "prediction_n_2708":
        len(
            dev_predictions
        ) == 2_708,

    "all_observation_ids_preserved":
        dev_predictions[
            "observation_id"
        ].tolist()
        ==
        optimization_dev[
            "observation_id"
        ].tolist(),

    "accuracy_finite":
        np.isfinite(
            accuracy
        ),

    "macro_f1_finite":
        np.isfinite(
            macro_f1
        ),

    "negative_recall_finite":
        np.isfinite(
            negative_recall
        ),

    "positive_recall_finite":
        np.isfinite(
            positive_recall
        ),

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_inference_checks_pass = bool(
    all(
        inference_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE B-500 OPTIMIZATION DEV CLASSIFICATION INFERENCE")
print("=" * 120)


print("\n[POPULATION]")
print(
    f"Optimization Dev n         : "
    f"{len(dev_predictions):,}"
)


print("\n[METRICS]")
print(
    f"Accuracy                   : "
    f"{accuracy:.6f}"
)
print(
    f"Macro-F1                   : "
    f"{macro_f1:.6f}"
)
print(
    f"Weighted-F1                : "
    f"{weighted_f1:.6f}"
)
print(
    f"Negative recall            : "
    f"{negative_recall:.6f}"
)
print(
    f"Neutral recall             : "
    f"{neutral_recall:.6f}"
)
print(
    f"Positive recall            : "
    f"{positive_recall:.6f}"
)


print("\n[GENERATION VALIDITY]")
print(
    f"Invalid predictions        : "
    f"{invalid_prediction_n:,}"
)
print(
    f"Invalid rate               : "
    f"{invalid_prediction_rate:.4%}"
)


print("\n[CLASS METRICS]")

display(
    class_metrics
)


print("\n[CONFUSION MATRIX]")

display(
    confusion_df
)


print("\n[INFERENCE CHECKS]")

for name, passed in inference_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Predictions                : "
    f"{STAGE_B_DEV_PREDICTIONS_PATH}"
)
print(
    f"Metrics                    : "
    f"{STAGE_B_DEV_METRICS_PATH}"
)
print(
    f"Class metrics              : "
    f"{STAGE_B_DEV_CLASS_METRICS_PATH}"
)
print(
    f"Confusion matrix           : "
    f"{STAGE_B_DEV_CONFUSION_PATH}"
)


print("\n" + "-" * 120)

if all_inference_checks_pass:

    print(
        "DAY 22 CELL 18 STATUS: PASS — "
        "STAGE B-500 OPTIMIZATION DEV CLASSIFICATION METRICS COMPLETE"
    )

else:

    print(
        "DAY 22 CELL 18 STATUS: FAIL — "
        "DO NOT INTERPRET STAGE B CLASSIFICATION PERFORMANCE"
    )

print("=" * 120)

[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=4) and `max_le

DAY 22 — STAGE B-500 OPTIMIZATION DEV CLASSIFICATION INFERENCE

[POPULATION]
Optimization Dev n         : 2,708

[METRICS]
Accuracy                   : 0.840842
Macro-F1                   : 0.808201
Weighted-F1                : 0.840209
Negative recall            : 0.721344
Neutral recall             : 0.889754
Positive recall            : 0.801126

[GENERATION VALIDITY]
Invalid predictions        : 0
Invalid rate               : 0.0000%

[CLASS METRICS]


,label,precision,recall,f1,support
0,negative,0.807522,0.721344,0.762004,506
1,neutral,0.877141,0.889754,0.883403,1669
2,positive,0.758437,0.801126,0.779197,533



[CONFUSION MATRIX]


,pred_negative,pred_neutral,pred_positive,pred_INVALID
true_negative,365,115,26,0
true_neutral,74,1485,110,0
true_positive,13,93,427,0
true_INVALID,0,0,0,0



[INFERENCE CHECKS]
PASS   — prediction_n_2708
PASS   — all_observation_ids_preserved
PASS   — accuracy_finite
PASS   — macro_f1_finite
PASS   — negative_recall_finite
PASS   — positive_recall_finite
PASS   — historical_validation_not_used
PASS   — future_not_used

[ARTIFACTS]
Predictions                : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_b_horizon_audit/artifacts/stage_b_500_optimization_dev_predictions.csv
Metrics                    : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_b_horizon_audit/artifacts/stage_b_500_optimization_dev_metrics.json
Class metrics              : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_b_horizon_audit/artifacts/stage_b_500_optimization_dev_class_metrics.csv
Confusion matrix           : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_b_horizon_audit/artifacts/stage_b_500_optimization_dev_confusion_matrix.csv

-----------------------------------------------------------------

In [19]:
# ==================================================================================================
# DAY 22 — CELL 19
# FAIR OPTIMIZATION-DEV COMPARISON: STAGE A-200 VS STAGE B-500
# ==================================================================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)


# --------------------------------------------------------------------------------------------------
# 1. Artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_A_DEV_PREDICTIONS_PATH = (
    STAGE_A_ARTIFACT_DIR
    / "stage_a_200_optimization_dev_predictions.csv"
)

STAGE_A_DEV_METRICS_PATH = (
    STAGE_A_ARTIFACT_DIR
    / "stage_a_200_optimization_dev_metrics.json"
)

STAGE_AB_COMPARISON_PATH = (
    DAY22_FINAL_DIR
    / "stage_a_vs_stage_b_optimization_dev_comparison.csv"
)

STAGE_AB_COMPARISON_SUMMARY_PATH = (
    DAY22_FINAL_DIR
    / "stage_a_vs_stage_b_optimization_dev_comparison.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Load Stage A adapter onto current Stage B base model
#
# Current:
#   "default"      = Stage B-500 adapter
#
# Add:
#   "stage_a_200"  = Stage A selected adapter
# --------------------------------------------------------------------------------------------------

STAGE_A_ADAPTER_NAME = "stage_a_200"
STAGE_B_ADAPTER_NAME = "default"


if STAGE_A_ADAPTER_NAME not in stage_b_model.peft_config:

    stage_b_model.load_adapter(
        str(
            STAGE_A_SELECTED_MODEL_DIR
        ),
        adapter_name=
            STAGE_A_ADAPTER_NAME,
        is_trainable=
            False,
    )


# --------------------------------------------------------------------------------------------------
# 3. Adapter integrity
# --------------------------------------------------------------------------------------------------

adapter_checks = {

    "stage_b_adapter_present":
        STAGE_B_ADAPTER_NAME
        in stage_b_model.peft_config,

    "stage_a_adapter_present":
        STAGE_A_ADAPTER_NAME
        in stage_b_model.peft_config,

    "stage_a_adapter_source_exists":
        STAGE_A_SELECTED_MODEL_DIR.exists(),

    "same_base_model_reused":
        stage_b_model is not None,
}


assert all(
    adapter_checks.values()
), (
    "DAY 22 STOP: multi-adapter comparison setup failed."
)


# --------------------------------------------------------------------------------------------------
# 4. Shared inference helper
# --------------------------------------------------------------------------------------------------

def run_adapter_inference(
    model,
    adapter_name,
    prompts,
    batch_size=8,
):

    model.set_adapter(
        adapter_name
    )

    model.eval()
    model.config.use_cache = True

    predictions = []

    for start_idx in range(
        0,
        len(prompts),
        batch_size,
    ):

        batch_prompts = prompts[
            start_idx:
            start_idx + batch_size
        ]

        encoded = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        )

        encoded = {
            key:
                value.to(
                    model.device
                )
            for key, value
            in encoded.items()
        }

        input_length = (
            encoded[
                "input_ids"
            ].shape[1]
        )

        with torch.inference_mode():

            generated = model.generate(
                **encoded,
                max_new_tokens=4,
                do_sample=False,
                pad_token_id=
                    tokenizer.eos_token_id,
                eos_token_id=
                    tokenizer.eos_token_id,
            )

        generated_only = (
            generated[
                :,
                input_length:
            ]
        )

        decoded = tokenizer.batch_decode(
            generated_only,
            skip_special_tokens=True,
        )

        predictions.extend(
            [
                text.strip().lower()
                for text
                in decoded
            ]
        )

    return predictions


# --------------------------------------------------------------------------------------------------
# 5. Exact same Optimization Dev prompts
# --------------------------------------------------------------------------------------------------

comparison_dev_frame = (
    optimization_dev[
        [
            "observation_id",
            "channel_username",
            "message_id",
            "date",
            "text",
            "sentiment",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


comparison_prompts = [
    build_inference_prompt(
        text
    )
    for text
    in comparison_dev_frame[
        "text"
    ].tolist()
]


# --------------------------------------------------------------------------------------------------
# 6. Run Stage A-200 inference
# --------------------------------------------------------------------------------------------------

stage_a_raw_predictions = (
    run_adapter_inference(
        model=
            stage_b_model,

        adapter_name=
            STAGE_A_ADAPTER_NAME,

        prompts=
            comparison_prompts,

        batch_size=
            8,
    )
)


assert len(
    stage_a_raw_predictions
) == 2_708


# --------------------------------------------------------------------------------------------------
# 7. Stage A prediction frame
# --------------------------------------------------------------------------------------------------

stage_a_predictions = (
    comparison_dev_frame
    .copy()
)


stage_a_predictions[
    "prediction_raw"
] = stage_a_raw_predictions


stage_a_predictions[
    "prediction_valid"
] = (
    stage_a_predictions[
        "prediction_raw"
    ]
    .isin(
        VALID_PREDICTIONS
    )
)


stage_a_predictions[
    "prediction"
] = (
    stage_a_predictions[
        "prediction_raw"
    ]
    .where(
        stage_a_predictions[
            "prediction_valid"
        ],
        other="INVALID",
    )
)


stage_a_invalid_n = int(
    (
        ~stage_a_predictions[
            "prediction_valid"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 8. Stage A metrics
# --------------------------------------------------------------------------------------------------

stage_a_y_true = (
    stage_a_predictions[
        "sentiment"
    ]
    .astype(str)
    .tolist()
)


stage_a_y_pred = (
    stage_a_predictions[
        "prediction"
    ]
    .astype(str)
    .tolist()
)


stage_a_accuracy = float(
    accuracy_score(
        stage_a_y_true,
        stage_a_y_pred,
    )
)


stage_a_macro_f1 = float(
    f1_score(
        stage_a_y_true,
        stage_a_y_pred,
        labels=
            VALID_PREDICTIONS,
        average=
            "macro",
        zero_division=
            0,
    )
)


stage_a_weighted_f1 = float(
    f1_score(
        stage_a_y_true,
        stage_a_y_pred,
        labels=
            VALID_PREDICTIONS,
        average=
            "weighted",
        zero_division=
            0,
    )
)


stage_a_precision, stage_a_recall, stage_a_class_f1, stage_a_support = (
    precision_recall_fscore_support(
        stage_a_y_true,
        stage_a_y_pred,
        labels=
            VALID_PREDICTIONS,
        zero_division=
            0,
    )
)


stage_a_class_metrics = pd.DataFrame(
    {
        "label":
            VALID_PREDICTIONS,

        "precision":
            stage_a_precision,

        "recall":
            stage_a_recall,

        "f1":
            stage_a_class_f1,

        "support":
            stage_a_support.astype(int),
    }
)


stage_a_neg_recall = float(
    stage_a_class_metrics.loc[
        stage_a_class_metrics[
            "label"
        ] == "negative",
        "recall",
    ].iloc[0]
)

stage_a_neu_recall = float(
    stage_a_class_metrics.loc[
        stage_a_class_metrics[
            "label"
        ] == "neutral",
        "recall",
    ].iloc[0]
)

stage_a_pos_recall = float(
    stage_a_class_metrics.loc[
        stage_a_class_metrics[
            "label"
        ] == "positive",
        "recall",
    ].iloc[0]
)


# --------------------------------------------------------------------------------------------------
# 9. Save Stage A predictions / metrics
# --------------------------------------------------------------------------------------------------

stage_a_predictions.to_csv(
    STAGE_A_DEV_PREDICTIONS_PATH,
    index=False,
)


stage_a_metrics = {

    "checkpoint":
        "stage_a_200",

    "population":
        "optimization_dev",

    "n":
        2708,

    "accuracy":
        stage_a_accuracy,

    "macro_f1":
        stage_a_macro_f1,

    "weighted_f1":
        stage_a_weighted_f1,

    "negative_recall":
        stage_a_neg_recall,

    "neutral_recall":
        stage_a_neu_recall,

    "positive_recall":
        stage_a_pos_recall,

    "invalid_n":
        stage_a_invalid_n,
}


with open(
    STAGE_A_DEV_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_a_metrics,
        f,
        indent=2,
    )


# --------------------------------------------------------------------------------------------------
# 10. Recover Stage B-500 metrics from Cell 18
# --------------------------------------------------------------------------------------------------

with open(
    STAGE_B_DEV_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:

    stage_b_metrics = load_artifact_json(
        f
    )


stage_b_accuracy = float(
    stage_b_metrics[
        "accuracy"
    ]
)

stage_b_macro_f1 = float(
    stage_b_metrics[
        "macro_f1"
    ]
)

stage_b_weighted_f1 = float(
    stage_b_metrics[
        "weighted_f1"
    ]
)

stage_b_neg_recall = float(
    stage_b_metrics[
        "class_recall"
    ][
        "negative"
    ]
)

stage_b_neu_recall = float(
    stage_b_metrics[
        "class_recall"
    ][
        "neutral"
    ]
)

stage_b_pos_recall = float(
    stage_b_metrics[
        "class_recall"
    ][
        "positive"
    ]
)

stage_b_invalid_n = int(
    stage_b_metrics[
        "invalid_prediction_n"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. Build fair comparison table
# --------------------------------------------------------------------------------------------------

comparison_table = pd.DataFrame(
    [
        {
            "model":
                "Stage A-200",

            "accuracy":
                stage_a_accuracy,

            "macro_f1":
                stage_a_macro_f1,

            "weighted_f1":
                stage_a_weighted_f1,

            "negative_recall":
                stage_a_neg_recall,

            "neutral_recall":
                stage_a_neu_recall,

            "positive_recall":
                stage_a_pos_recall,

            "invalid_n":
                stage_a_invalid_n,
        },

        {
            "model":
                "Stage B-500",

            "accuracy":
                stage_b_accuracy,

            "macro_f1":
                stage_b_macro_f1,

            "weighted_f1":
                stage_b_weighted_f1,

            "negative_recall":
                stage_b_neg_recall,

            "neutral_recall":
                stage_b_neu_recall,

            "positive_recall":
                stage_b_pos_recall,

            "invalid_n":
                stage_b_invalid_n,
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. Improvements
# --------------------------------------------------------------------------------------------------

accuracy_gain = (
    stage_b_accuracy
    - stage_a_accuracy
)

macro_f1_gain = (
    stage_b_macro_f1
    - stage_a_macro_f1
)

negative_recall_gain = (
    stage_b_neg_recall
    - stage_a_neg_recall
)

neutral_recall_gain = (
    stage_b_neu_recall
    - stage_a_neu_recall
)

positive_recall_gain = (
    stage_b_pos_recall
    - stage_a_pos_recall
)


# --------------------------------------------------------------------------------------------------
# 13. Formal improvement assessment
# --------------------------------------------------------------------------------------------------

stage_b_improves_accuracy = (
    accuracy_gain
    > 0
)

stage_b_improves_macro_f1 = (
    macro_f1_gain
    > 0
)

minority_recall_floor_a = min(
    stage_a_neg_recall,
    stage_a_pos_recall,
)

minority_recall_floor_b = min(
    stage_b_neg_recall,
    stage_b_pos_recall,
)

minority_floor_gain = (
    minority_recall_floor_b
    - minority_recall_floor_a
)


# --------------------------------------------------------------------------------------------------
# 14. Save fair comparison artifacts
# --------------------------------------------------------------------------------------------------

comparison_table.to_csv(
    STAGE_AB_COMPARISON_PATH,
    index=False,
)


comparison_summary = {

    "population":
        "optimization_dev",

    "n":
        2708,

    "stage_a":
        stage_a_metrics,

    "stage_b":
        stage_b_metrics,

    "delta_stage_b_minus_stage_a": {

        "accuracy":
            float(
                accuracy_gain
            ),

        "macro_f1":
            float(
                macro_f1_gain
            ),

        "negative_recall":
            float(
                negative_recall_gain
            ),

        "neutral_recall":
            float(
                neutral_recall_gain
            ),

        "positive_recall":
            float(
                positive_recall_gain
            ),

        "minority_recall_floor":
            float(
                minority_floor_gain
            ),
    },

    "interpretation": {

        "accuracy_improved":
            bool(
                stage_b_improves_accuracy
            ),

        "macro_f1_improved":
            bool(
                stage_b_improves_macro_f1
            ),

        "minority_recall_floor_improved":
            bool(
                minority_floor_gain
                > 0
            ),
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_AB_COMPARISON_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        comparison_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Restore Stage B adapter as active after comparison
# --------------------------------------------------------------------------------------------------

stage_b_model.set_adapter(
    STAGE_B_ADAPTER_NAME
)

stage_b_model.eval()


# --------------------------------------------------------------------------------------------------
# 16. Formal checks
# --------------------------------------------------------------------------------------------------

comparison_checks = {

    **adapter_checks,

    "stage_a_prediction_n_2708":
        len(
            stage_a_predictions
        )
        == 2708,

    "stage_b_prediction_n_2708":
        int(
            stage_b_metrics[
                "n"
            ]
        )
        == 2708,

    "same_evaluation_population":
        True,

    "stage_a_accuracy_finite":
        np.isfinite(
            stage_a_accuracy
        ),

    "stage_b_accuracy_finite":
        np.isfinite(
            stage_b_accuracy
        ),

    "stage_b_adapter_restored":
        (
            stage_b_model.active_adapter
            == STAGE_B_ADAPTER_NAME
        ),
}


all_comparison_checks_pass = bool(
    all(
        comparison_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 17. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — FAIR STAGE A-200 VS STAGE B-500 OPTIMIZATION DEV COMPARISON")
print("=" * 120)


print("\n[COMMON EVALUATION POPULATION]")
print(
    "Population                  : Optimization Dev"
)
print(
    "n                           : 2,708"
)


print("\n[MODEL COMPARISON]")

display(
    comparison_table
)


print("\n[STAGE B - STAGE A DELTA]")
print(
    f"Accuracy                    : "
    f"{accuracy_gain:+.6f} "
    f"({accuracy_gain * 100:+.2f} pp)"
)
print(
    f"Macro-F1                    : "
    f"{macro_f1_gain:+.6f}"
)
print(
    f"Negative recall             : "
    f"{negative_recall_gain:+.6f}"
)
print(
    f"Neutral recall              : "
    f"{neutral_recall_gain:+.6f}"
)
print(
    f"Positive recall             : "
    f"{positive_recall_gain:+.6f}"
)
print(
    f"Minority recall floor       : "
    f"{minority_floor_gain:+.6f}"
)


print("\n[INTERPRETATION]")
print(
    f"Accuracy improved           : "
    f"{stage_b_improves_accuracy}"
)
print(
    f"Macro-F1 improved           : "
    f"{stage_b_improves_macro_f1}"
)
print(
    f"Minority recall improved    : "
    f"{minority_floor_gain > 0}"
)


print("\n[CHECKS]")

for name, passed in comparison_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Stage A predictions         : "
    f"{STAGE_A_DEV_PREDICTIONS_PATH}"
)
print(
    f"Stage A metrics             : "
    f"{STAGE_A_DEV_METRICS_PATH}"
)
print(
    f"Comparison table            : "
    f"{STAGE_AB_COMPARISON_PATH}"
)
print(
    f"Comparison summary          : "
    f"{STAGE_AB_COMPARISON_SUMMARY_PATH}"
)


print("\n" + "-" * 120)

if all_comparison_checks_pass:

    print(
        "DAY 22 CELL 19 STATUS: PASS — "
        "FAIR STAGE A VS STAGE B CLASSIFICATION COMPARISON COMPLETE"
    )

else:

    print(
        "DAY 22 CELL 19 STATUS: FAIL — "
        "DO NOT INTERPRET STAGE A VS STAGE B PERFORMANCE"
    )

print("=" * 120)

[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

DAY 22 — FAIR STAGE A-200 VS STAGE B-500 OPTIMIZATION DEV COMPARISON

[COMMON EVALUATION POPULATION]
Population                  : Optimization Dev
n                           : 2,708

[MODEL COMPARISON]


,model,accuracy,macro_f1,weighted_f1,negative_recall,neutral_recall,positive_recall,invalid_n
0,Stage A-200,0.779542,0.726037,0.774613,0.608696,0.880168,0.626642,0
1,Stage B-500,0.840842,0.808201,0.840209,0.721344,0.889754,0.801126,0



[STAGE B - STAGE A DELTA]
Accuracy                    : +0.061300 (+6.13 pp)
Macro-F1                    : +0.082165
Negative recall             : +0.112648
Neutral recall              : +0.009587
Positive recall             : +0.174484
Minority recall floor       : +0.112648

[INTERPRETATION]
Accuracy improved           : True
Macro-F1 improved           : True
Minority recall improved    : True

[CHECKS]
PASS   — stage_b_adapter_present
PASS   — stage_a_adapter_present
PASS   — stage_a_adapter_source_exists
PASS   — same_base_model_reused
PASS   — stage_a_prediction_n_2708
PASS   — stage_b_prediction_n_2708
PASS   — same_evaluation_population
PASS   — stage_a_accuracy_finite
PASS   — stage_b_accuracy_finite
PASS   — stage_b_adapter_restored

[ARTIFACTS]
Stage A predictions         : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_a_baseline/artifacts/stage_a_200_optimization_dev_predictions.csv
Stage A metrics             : /content/drive/MyDrive/FinGPT_Project/result

In [21]:
# ==================================================================================================
# DAY 22 — CELL 20
# STAGE C SAMPLING INTERVENTION PROTOCOL FREEZE
# RUNTIME-INDEPENDENT PATH VERSION
# ==================================================================================================

import json
import math
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Explicit Day 22 root
#
# Do not depend on DAY22_RESULTS_DIR from an earlier runtime state.
# --------------------------------------------------------------------------------------------------

DAY22_RESULTS_ROOT = (PROJECT_ROOT / 'results' / 'day22')


assert DAY22_RESULTS_ROOT.exists(), (
    f"DAY 22 STOP: results root does not exist: {DAY22_RESULTS_ROOT}"
)


STAGE_C_DIR = (
    DAY22_RESULTS_ROOT
    / "runs"
    / "stage_c_sampling_audit"
)

STAGE_C_ARTIFACT_DIR = (
    STAGE_C_DIR
    / "artifacts"
)

STAGE_C_MODEL_DIR = (
    STAGE_C_DIR
    / "model"
)


STAGE_C_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

STAGE_C_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE_C_PROTOCOL_PATH = (
    STAGE_C_ARTIFACT_DIR
    / "stage_c_sampling_protocol.json"
)

STAGE_C_CLASS_WEIGHTS_PATH = (
    STAGE_C_ARTIFACT_DIR
    / "stage_c_sampling_class_weights.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen Optimization Train class counts
# --------------------------------------------------------------------------------------------------

STAGE_C_LABELS = [
    "negative",
    "neutral",
    "positive",
]


train_label_counts = (
    optimization_train[
        "sentiment"
    ]
    .value_counts()
    .reindex(
        STAGE_C_LABELS
    )
)


class_counts = {
    label:
        int(
            train_label_counts[
                label
            ]
        )
    for label
    in STAGE_C_LABELS
}


# --------------------------------------------------------------------------------------------------
# 3. Integrity check against frozen Day 22 split
# --------------------------------------------------------------------------------------------------

EXPECTED_CLASS_COUNTS = {
    "negative": 2276,
    "neutral": 6767,
    "positive": 1788,
}


assert class_counts == EXPECTED_CLASS_COUNTS, (
    f"DAY 22 STOP: Optimization Train class counts changed: {class_counts}"
)


# --------------------------------------------------------------------------------------------------
# 4. Natural sampling weights
# --------------------------------------------------------------------------------------------------

natural_class_weights = {
    label:
        1.0
    for label
    in STAGE_C_LABELS
}


# --------------------------------------------------------------------------------------------------
# 5. Moderate inverse-square-root frequency weighting
#
# w_c proportional to 1 / sqrt(n_c)
#
# Normalize relative to neutral = 1.
# --------------------------------------------------------------------------------------------------

sqrt_raw = {
    label:
        1.0
        /
        math.sqrt(
            class_counts[
                label
            ]
        )
    for label
    in STAGE_C_LABELS
}


sqrt_reference = (
    sqrt_raw[
        "neutral"
    ]
)


inverse_sqrt_class_weights = {
    label:
        float(
            sqrt_raw[
                label
            ]
            /
            sqrt_reference
        )
    for label
    in STAGE_C_LABELS
}


# --------------------------------------------------------------------------------------------------
# 6. Capped balanced weighting
#
# raw_c = n_max / n_c
# w_c   = min(raw_c, 3.0)
# --------------------------------------------------------------------------------------------------

MAX_CLASS_WEIGHT_RATIO = 3.0


n_max = max(
    class_counts.values()
)


balanced_raw = {
    label:
        float(
            n_max
            /
            class_counts[
                label
            ]
        )
    for label
    in STAGE_C_LABELS
}


capped_balanced_class_weights = {
    label:
        float(
            min(
                balanced_raw[
                    label
                ],
                MAX_CLASS_WEIGHT_RATIO,
            )
        )
    for label
    in STAGE_C_LABELS
}


# --------------------------------------------------------------------------------------------------
# 7. Weight table
# --------------------------------------------------------------------------------------------------

weight_table = pd.DataFrame(
    {
        "label":
            STAGE_C_LABELS,

        "count":
            [
                class_counts[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "natural_weight":
            [
                natural_class_weights[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "inverse_sqrt_weight":
            [
                inverse_sqrt_class_weights[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "inverse_frequency_raw":
            [
                balanced_raw[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "capped_balanced_weight":
            [
                capped_balanced_class_weights[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],
    }
)


weight_table.to_csv(
    STAGE_C_CLASS_WEIGHTS_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 8. Stage C conditions
#
# Natural condition reuses frozen Stage B-500.
# Only inverse_sqrt and capped_balanced require new training.
# --------------------------------------------------------------------------------------------------

STAGE_C_CONDITIONS = {

    "natural": {

        "role":
            "incumbent_reference",

        "training_required":
            False,

        "source":
            "Stage B-500",

        "sampling":
            "natural",

        "class_weights":
            natural_class_weights,
    },

    "inverse_sqrt": {

        "role":
            "sampling_candidate",

        "training_required":
            True,

        "sampling":
            "weighted_random_sampling",

        "weight_rule":
            "1/sqrt(class_frequency)",

        "class_weights":
            inverse_sqrt_class_weights,
    },

    "capped_balanced": {

        "role":
            "sampling_candidate",

        "training_required":
            True,

        "sampling":
            "weighted_random_sampling",

        "weight_rule":
            "min(n_max/n_c, 3.0)",

        "cap":
            MAX_CLASS_WEIGHT_RATIO,

        "class_weights":
            capped_balanced_class_weights,
    },
}


# --------------------------------------------------------------------------------------------------
# 9. Freeze Stage C training protocol
# --------------------------------------------------------------------------------------------------

STAGE_C_PROTOCOL = {

    "day":
        22,

    "stage":
        "C",

    "stage_name":
        "sampling_intervention_audit",

    "objective":
        (
            "Test whether controlled class-sampling interventions provide "
            "incremental classification gains beyond the frozen Stage B-500 incumbent."
        ),

    "incumbent":
        {

            "model":
                "Stage B-500",

            "optimization_dev_accuracy":
                0.840842,

            "optimization_dev_macro_f1":
                0.808201,

            "negative_recall":
                0.721344,

            "neutral_recall":
                0.889754,

            "positive_recall":
                0.801126,
        },

    "data":
        {

            "train_split":
                "optimization_train",

            "train_n":
                10831,

            "dev_split":
                "optimization_dev",

            "dev_n":
                2708,

            "class_counts":
                class_counts,

            "historical_validation_used":
                False,

            "future_f1_f6_used":
                False,
        },

    "conditions":
        STAGE_C_CONDITIONS,

    "training_protocol":
        {

            "fresh_initialization_per_candidate":
                True,

            "base_model":
                "meta-llama/Meta-Llama-3-8B",

            "lora_r":
                8,

            "lora_alpha":
                32,

            "lora_dropout":
                0.1,

            "target_modules":
                [
                    "q_proj",
                    "v_proj",
                ],

            "max_steps":
                500,

            "learning_rate":
                1e-4,

            "lr_scheduler_type":
                "linear",

            "warmup_steps":
                10,

            "train_batch_size":
                1,

            "gradient_accumulation_steps":
                8,

            "effective_batch_size":
                8,

            "seed":
                42,

            "data_seed":
                42,

            "max_seq_length":
                512,
        },

    "selection_protocol":
        {

            "evaluation_split":
                "optimization_dev",

            "primary_metric":
                "accuracy",

            "direction":
                "maximize",

            "guardrails":
                {

                    "macro_f1":
                        "must_not_materially_degrade",

                    "negative_recall":
                        "must_not_materially_degrade",

                    "positive_recall":
                        "must_not_materially_degrade",

                    "invalid_generation_rate":
                        "must_remain_near_zero",
                },

            "tie_break_order":
                [
                    "macro_f1",
                    "minority_recall_floor",
                    "eval_loss",
                    "simpler_sampling_scheme",
                ],
        },

    "firewall":
        {

            "historical_validation_forbidden":
                True,

            "future_f1_f6_forbidden":
                True,
        },

    "freeze_status":
        "STAGE_C_SAMPLING_PROTOCOL_FROZEN",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 10. Formal checks
# --------------------------------------------------------------------------------------------------

stage_c_checks = {

    "day22_root_exists":
        DAY22_RESULTS_ROOT.exists(),

    "stage_c_artifact_dir_exists":
        STAGE_C_ARTIFACT_DIR.exists(),

    "train_n_10831":
        len(
            optimization_train
        )
        == 10831,

    "dev_n_2708":
        len(
            optimization_dev
        )
        == 2708,

    "class_counts_exact":
        class_counts
        ==
        EXPECTED_CLASS_COUNTS,

    "three_conditions":
        set(
            STAGE_C_CONDITIONS.keys()
        )
        ==
        {
            "natural",
            "inverse_sqrt",
            "capped_balanced",
        },

    "natural_reuses_stage_b":
        STAGE_C_CONDITIONS[
            "natural"
        ][
            "training_required"
        ]
        is False,

    "inverse_sqrt_requires_training":
        STAGE_C_CONDITIONS[
            "inverse_sqrt"
        ][
            "training_required"
        ]
        is True,

    "capped_balanced_requires_training":
        STAGE_C_CONDITIONS[
            "capped_balanced"
        ][
            "training_required"
        ]
        is True,

    "cap_3":
        MAX_CLASS_WEIGHT_RATIO
        == 3.0,

    "max_steps_500":
        STAGE_C_PROTOCOL[
            "training_protocol"
        ][
            "max_steps"
        ]
        == 500,

    "learning_rate_1e4":
        STAGE_C_PROTOCOL[
            "training_protocol"
        ][
            "learning_rate"
        ]
        == 1e-4,

    "seed_42":
        STAGE_C_PROTOCOL[
            "training_protocol"
        ][
            "seed"
        ]
        == 42,

    "primary_metric_accuracy":
        STAGE_C_PROTOCOL[
            "selection_protocol"
        ][
            "primary_metric"
        ]
        ==
        "accuracy",

    "historical_validation_firewall":
        STAGE_C_PROTOCOL[
            "data"
        ][
            "historical_validation_used"
        ]
        is False,

    "future_firewall":
        STAGE_C_PROTOCOL[
            "data"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


all_stage_c_checks_pass = bool(
    all(
        stage_c_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Save protocol
# --------------------------------------------------------------------------------------------------

with open(
    STAGE_C_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        STAGE_C_PROTOCOL,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 12. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE C SAMPLING INTERVENTION PROTOCOL")
print("=" * 120)


print("\n[PATHS]")
print(
    f"Day 22 root                : "
    f"{DAY22_RESULTS_ROOT}"
)
print(
    f"Stage C                    : "
    f"{STAGE_C_DIR}"
)


print("\n[STAGE B INCUMBENT]")
print(
    "Model                      : Stage B-500"
)
print(
    "Optimization Dev Accuracy  : 0.840842"
)
print(
    "Macro-F1                   : 0.808201"
)


print("\n[OPTIMIZATION TRAIN CLASS COUNTS]")

for label in STAGE_C_LABELS:

    print(
        f"{label:<27}: "
        f"{class_counts[label]:,}"
    )


print("\n[SAMPLING WEIGHTS]")

display(
    weight_table
)


print("\n[STAGE C CONDITIONS]")
print(
    "1. natural                 : frozen Stage B-500 incumbent"
)
print(
    "2. inverse_sqrt            : NEW 500-step training"
)
print(
    "3. capped_balanced         : NEW 500-step training"
)


print("\n[CONTROLLED VARIABLES]")
print(
    "Base model                 : Meta-Llama-3-8B"
)
print(
    "LoRA                       : r=8, alpha=32, dropout=0.1"
)
print(
    "Training horizon           : 500"
)
print(
    "Learning rate              : 1e-4"
)
print(
    "Scheduler                  : linear"
)
print(
    "Warmup                     : 10"
)
print(
    "Effective batch            : 8"
)
print(
    "Seed                       : 42"
)


print("\n[SELECTION]")
print(
    "Primary metric             : Accuracy"
)
print(
    "Evaluation split           : Optimization Dev"
)
print(
    "Historical Validation      : FORBIDDEN"
)
print(
    "Future F1-F6               : FORBIDDEN"
)


print("\n[PROTOCOL CHECKS]")

for name, passed in stage_c_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Protocol                   : "
    f"{STAGE_C_PROTOCOL_PATH}"
)
print(
    f"Class weights              : "
    f"{STAGE_C_CLASS_WEIGHTS_PATH}"
)


print("\n" + "-" * 120)

if all_stage_c_checks_pass:

    print(
        "DAY 22 CELL 20 STATUS: PASS — "
        "STAGE C SAMPLING PROTOCOL FROZEN"
    )

else:

    print(
        "DAY 22 CELL 20 STATUS: FAIL — "
        "DO NOT INITIALIZE STAGE C TRAINING"
    )

print("=" * 120)


DAY 22 — STAGE C SAMPLING INTERVENTION PROTOCOL

[PATHS]
Day 22 root                : /content/drive/MyDrive/FinGPT_Project/results/day22
Stage C                    : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_c_sampling_audit

[STAGE B INCUMBENT]
Model                      : Stage B-500
Optimization Dev Accuracy  : 0.840842
Macro-F1                   : 0.808201

[OPTIMIZATION TRAIN CLASS COUNTS]
negative                   : 2,276
neutral                    : 6,767
positive                   : 1,788

[SAMPLING WEIGHTS]


,label,count,natural_weight,inverse_sqrt_weight,inverse_frequency_raw,capped_balanced_weight
0,negative,2276,1.0,1.724297,2.973199,2.973199
1,neutral,6767,1.0,1.000000,1.000000,1.000000
2,positive,1788,1.0,1.945424,3.784676,3.000000



[STAGE C CONDITIONS]
1. natural                 : frozen Stage B-500 incumbent
2. inverse_sqrt            : NEW 500-step training
3. capped_balanced         : NEW 500-step training

[CONTROLLED VARIABLES]
Base model                 : Meta-Llama-3-8B
LoRA                       : r=8, alpha=32, dropout=0.1
Training horizon           : 500
Learning rate              : 1e-4
Scheduler                  : linear
Warmup                     : 10
Effective batch            : 8
Seed                       : 42

[SELECTION]
Primary metric             : Accuracy
Evaluation split           : Optimization Dev
Historical Validation      : FORBIDDEN
Future F1-F6               : FORBIDDEN

[PROTOCOL CHECKS]
PASS   — day22_root_exists
PASS   — stage_c_artifact_dir_exists
PASS   — train_n_10831
PASS   — dev_n_2708
PASS   — class_counts_exact
PASS   — three_conditions
PASS   — natural_reuses_stage_b
PASS   — inverse_sqrt_requires_training
PASS   — capped_balanced_requires_training
PASS   — cap_3
PASS   — m

In [22]:
# ==================================================================================================
# DAY 22 — CELL 21
# STAGE C DETERMINISTIC WEIGHTED-SAMPLER CONSTRUCTION + REALIZED COMPOSITION AUDIT
# ==================================================================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from torch.utils.data import WeightedRandomSampler


# --------------------------------------------------------------------------------------------------
# 1. Formal Stage C sampling geometry
#
# per-device batch = 1
# gradient accumulation = 8
# max optimizer steps = 500
#
# Therefore approximately:
#
#     500 * 8 = 4000 sampled training observations
#
# are consumed in one candidate trajectory.
# --------------------------------------------------------------------------------------------------

STAGE_C_MAX_STEPS = int(
    STAGE_C_PROTOCOL[
        "training_protocol"
    ][
        "max_steps"
    ]
)

STAGE_C_MICRO_BATCH = int(
    STAGE_C_PROTOCOL[
        "training_protocol"
    ][
        "train_batch_size"
    ]
)

STAGE_C_GRAD_ACCUM = int(
    STAGE_C_PROTOCOL[
        "training_protocol"
    ][
        "gradient_accumulation_steps"
    ]
)

STAGE_C_EFFECTIVE_BATCH = (
    STAGE_C_MICRO_BATCH
    *
    STAGE_C_GRAD_ACCUM
)

STAGE_C_TOTAL_DRAWS = (
    STAGE_C_MAX_STEPS
    *
    STAGE_C_EFFECTIVE_BATCH
)


assert STAGE_C_TOTAL_DRAWS == 4000


# --------------------------------------------------------------------------------------------------
# 2. Frozen training labels in exact dataset row order
# --------------------------------------------------------------------------------------------------

stage_c_train_labels = (
    optimization_train[
        "sentiment"
    ]
    .astype(str)
    .reset_index(drop=True)
)


assert len(
    stage_c_train_labels
) == 10831


# --------------------------------------------------------------------------------------------------
# 3. Build sample-level weights from frozen class-level weights
# --------------------------------------------------------------------------------------------------

def build_sample_weight_tensor(
    labels,
    class_weight_dict,
):

    values = [
        float(
            class_weight_dict[
                label
            ]
        )
        for label
        in labels
    ]

    return torch.tensor(
        values,
        dtype=torch.double,
    )


inverse_sqrt_sample_weights = (
    build_sample_weight_tensor(
        stage_c_train_labels,
        inverse_sqrt_class_weights,
    )
)


capped_balanced_sample_weights = (
    build_sample_weight_tensor(
        stage_c_train_labels,
        capped_balanced_class_weights,
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Compute theoretical sampling probabilities
#
# Under WeightedRandomSampler:
#
#     P(class c)
#       proportional to
#     n_c * w_c
# --------------------------------------------------------------------------------------------------

def theoretical_class_probabilities(
    class_counts,
    class_weights,
):

    class_mass = {
        label:
            (
                class_counts[
                    label
                ]
                *
                class_weights[
                    label
                ]
            )
        for label
        in STAGE_C_LABELS
    }

    total_mass = sum(
        class_mass.values()
    )

    return {
        label:
            float(
                class_mass[
                    label
                ]
                /
                total_mass
            )
        for label
        in STAGE_C_LABELS
    }


natural_expected_probs = (
    theoretical_class_probabilities(
        class_counts,
        natural_class_weights,
    )
)


inverse_sqrt_expected_probs = (
    theoretical_class_probabilities(
        class_counts,
        inverse_sqrt_class_weights,
    )
)


capped_expected_probs = (
    theoretical_class_probabilities(
        class_counts,
        capped_balanced_class_weights,
    )
)


# --------------------------------------------------------------------------------------------------
# 5. Deterministic sampling simulator
#
# IMPORTANT:
#
# Candidate-specific generators are initialized with the SAME frozen seed.
# Each candidate is an independent experiment from fresh initialization.
#
# These exact sampler settings will later be reused during training.
# --------------------------------------------------------------------------------------------------

STAGE_C_SAMPLER_SEED = 42


def deterministic_weighted_draws(
    sample_weights,
    n_draws,
    seed,
):

    generator = torch.Generator()

    generator.manual_seed(
        int(
            seed
        )
    )

    sampler = WeightedRandomSampler(
        weights=
            sample_weights,

        num_samples=
            int(
                n_draws
            ),

        replacement=
            True,

        generator=
            generator,
    )

    sampled_indices = list(
        sampler
    )

    return sampled_indices


inverse_sqrt_indices = (
    deterministic_weighted_draws(
        sample_weights=
            inverse_sqrt_sample_weights,

        n_draws=
            STAGE_C_TOTAL_DRAWS,

        seed=
            STAGE_C_SAMPLER_SEED,
    )
)


capped_balanced_indices = (
    deterministic_weighted_draws(
        sample_weights=
            capped_balanced_sample_weights,

        n_draws=
            STAGE_C_TOTAL_DRAWS,

        seed=
            STAGE_C_SAMPLER_SEED,
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Realized class composition
# --------------------------------------------------------------------------------------------------

def realized_composition(
    labels,
    sampled_indices,
):

    sampled_labels = (
        labels.iloc[
            sampled_indices
        ]
        .reset_index(drop=True)
    )

    counts = (
        sampled_labels
        .value_counts()
        .reindex(
            STAGE_C_LABELS,
            fill_value=0,
        )
    )

    proportions = (
        counts
        /
        len(
            sampled_labels
        )
    )

    return (
        sampled_labels,
        counts,
        proportions,
    )


(
    inverse_sqrt_sampled_labels,
    inverse_sqrt_realized_counts,
    inverse_sqrt_realized_props,
) = realized_composition(
    labels=
        stage_c_train_labels,

    sampled_indices=
        inverse_sqrt_indices,
)


(
    capped_sampled_labels,
    capped_realized_counts,
    capped_realized_props,
) = realized_composition(
    labels=
        stage_c_train_labels,

    sampled_indices=
        capped_balanced_indices,
)


# --------------------------------------------------------------------------------------------------
# 7. Build formal composition table
# --------------------------------------------------------------------------------------------------

composition_rows = []


for label in STAGE_C_LABELS:

    composition_rows.append(
        {
            "condition":
                "natural",

            "label":
                label,

            "class_weight":
                natural_class_weights[
                    label
                ],

            "expected_probability":
                natural_expected_probs[
                    label
                ],

            "realized_count":
                None,

            "realized_probability":
                None,
        }
    )


for label in STAGE_C_LABELS:

    composition_rows.append(
        {
            "condition":
                "inverse_sqrt",

            "label":
                label,

            "class_weight":
                inverse_sqrt_class_weights[
                    label
                ],

            "expected_probability":
                inverse_sqrt_expected_probs[
                    label
                ],

            "realized_count":
                int(
                    inverse_sqrt_realized_counts[
                        label
                    ]
                ),

            "realized_probability":
                float(
                    inverse_sqrt_realized_props[
                        label
                    ]
                ),
        }
    )


for label in STAGE_C_LABELS:

    composition_rows.append(
        {
            "condition":
                "capped_balanced",

            "label":
                label,

            "class_weight":
                capped_balanced_class_weights[
                    label
                ],

            "expected_probability":
                capped_expected_probs[
                    label
                ],

            "realized_count":
                int(
                    capped_realized_counts[
                        label
                    ]
                ),

            "realized_probability":
                float(
                    capped_realized_props[
                        label
                    ]
                ),
        }
    )


stage_c_sampling_composition = pd.DataFrame(
    composition_rows
)


# --------------------------------------------------------------------------------------------------
# 8. Difference between theoretical and realized proportions
# --------------------------------------------------------------------------------------------------

stage_c_sampling_composition[
    "realized_minus_expected"
] = (
    stage_c_sampling_composition[
        "realized_probability"
    ]
    -
    stage_c_sampling_composition[
        "expected_probability"
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. Duplicate exposure diagnostics
#
# Replacement sampling intentionally permits repeated observations.
# We quantify the number of unique observations exposed in the first 4000 draws.
# --------------------------------------------------------------------------------------------------

inverse_sqrt_unique_indices = len(
    set(
        inverse_sqrt_indices
    )
)

capped_unique_indices = len(
    set(
        capped_balanced_indices
    )
)


inverse_sqrt_unique_fraction = (
    inverse_sqrt_unique_indices
    /
    STAGE_C_TOTAL_DRAWS
)


capped_unique_fraction = (
    capped_unique_indices
    /
    STAGE_C_TOTAL_DRAWS
)


# --------------------------------------------------------------------------------------------------
# 10. Expected and realized audit tables
# --------------------------------------------------------------------------------------------------

inverse_sqrt_audit_table = pd.DataFrame(
    {
        "label":
            STAGE_C_LABELS,

        "original_count":
            [
                class_counts[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "weight":
            [
                inverse_sqrt_class_weights[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "expected_pct":
            [
                100
                *
                inverse_sqrt_expected_probs[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "realized_count_4000":
            [
                int(
                    inverse_sqrt_realized_counts[
                        label
                    ]
                )
                for label
                in STAGE_C_LABELS
            ],

        "realized_pct":
            [
                100
                *
                float(
                    inverse_sqrt_realized_props[
                        label
                    ]
                )
                for label
                in STAGE_C_LABELS
            ],
    }
)


capped_balanced_audit_table = pd.DataFrame(
    {
        "label":
            STAGE_C_LABELS,

        "original_count":
            [
                class_counts[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "weight":
            [
                capped_balanced_class_weights[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "expected_pct":
            [
                100
                *
                capped_expected_probs[
                    label
                ]
                for label
                in STAGE_C_LABELS
            ],

        "realized_count_4000":
            [
                int(
                    capped_realized_counts[
                        label
                    ]
                )
                for label
                in STAGE_C_LABELS
            ],

        "realized_pct":
            [
                100
                *
                float(
                    capped_realized_props[
                        label
                    ]
                )
                for label
                in STAGE_C_LABELS
            ],
    }
)


# --------------------------------------------------------------------------------------------------
# 11. Formal sampler checks
#
# These are protocol-integrity checks, not performance criteria.
# --------------------------------------------------------------------------------------------------

def max_probability_deviation(
    expected_probs,
    realized_props,
):

    return max(
        abs(
            float(
                realized_props[
                    label
                ]
            )
            -
            float(
                expected_probs[
                    label
                ]
            )
        )
        for label
        in STAGE_C_LABELS
    )


inverse_sqrt_max_deviation = (
    max_probability_deviation(
        inverse_sqrt_expected_probs,
        inverse_sqrt_realized_props,
    )
)


capped_max_deviation = (
    max_probability_deviation(
        capped_expected_probs,
        capped_realized_props,
    )
)


stage_c_sampler_checks = {

    "total_draws_4000":
        STAGE_C_TOTAL_DRAWS
        == 4000,

    "inverse_sqrt_weight_length_10831":
        len(
            inverse_sqrt_sample_weights
        )
        == 10831,

    "capped_weight_length_10831":
        len(
            capped_balanced_sample_weights
        )
        == 10831,

    "inverse_sqrt_draw_n_4000":
        len(
            inverse_sqrt_indices
        )
        == 4000,

    "capped_draw_n_4000":
        len(
            capped_balanced_indices
        )
        == 4000,

    "inverse_sqrt_all_classes_present":
        all(
            inverse_sqrt_realized_counts[
                label
            ]
            > 0
            for label
            in STAGE_C_LABELS
        ),

    "capped_all_classes_present":
        all(
            capped_realized_counts[
                label
            ]
            > 0
            for label
            in STAGE_C_LABELS
        ),

    # 3 percentage points is deliberately generous:
    # this only detects a broken sampler, not ordinary Monte Carlo variation.
    "inverse_sqrt_matches_expected":
        inverse_sqrt_max_deviation
        < 0.03,

    "capped_matches_expected":
        capped_max_deviation
        < 0.03,

    "inverse_sqrt_not_majority_dominated":
        float(
            inverse_sqrt_realized_props[
                "neutral"
            ]
        )
        < 0.55,

    "capped_not_majority_dominated":
        float(
            capped_realized_props[
                "neutral"
            ]
        )
        < 0.45,

    "replacement_sampling_preserves_valid_indices":
        (
            min(
                inverse_sqrt_indices
            )
            >= 0
            and
            max(
                inverse_sqrt_indices
            )
            < 10831
            and
            min(
                capped_balanced_indices
            )
            >= 0
            and
            max(
                capped_balanced_indices
            )
            < 10831
        ),

    "seed_42":
        STAGE_C_SAMPLER_SEED
        == 42,

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_stage_c_sampler_checks_pass = bool(
    all(
        stage_c_sampler_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Save artifacts
# --------------------------------------------------------------------------------------------------

STAGE_C_SAMPLER_AUDIT_PATH = (
    STAGE_C_ARTIFACT_DIR
    / "stage_c_sampler_composition_audit.csv"
)

STAGE_C_SAMPLER_SUMMARY_PATH = (
    STAGE_C_ARTIFACT_DIR
    / "stage_c_sampler_audit_summary.json"
)


stage_c_sampling_composition.to_csv(
    STAGE_C_SAMPLER_AUDIT_PATH,
    index=False,
)


stage_c_sampler_summary = {

    "day":
        22,

    "stage":
        "C",

    "artifact":
        "deterministic_sampler_audit",

    "sampler_seed":
        STAGE_C_SAMPLER_SEED,

    "replacement":
        True,

    "trajectory_geometry": {

        "max_steps":
            STAGE_C_MAX_STEPS,

        "effective_batch_size":
            STAGE_C_EFFECTIVE_BATCH,

        "total_draws":
            STAGE_C_TOTAL_DRAWS,
    },

    "inverse_sqrt": {

        "expected_probabilities":
            inverse_sqrt_expected_probs,

        "realized_counts":
            {
                label:
                    int(
                        inverse_sqrt_realized_counts[
                            label
                        ]
                    )
                for label
                in STAGE_C_LABELS
            },

        "realized_probabilities":
            {
                label:
                    float(
                        inverse_sqrt_realized_props[
                            label
                        ]
                    )
                for label
                in STAGE_C_LABELS
            },

        "unique_indices":
            inverse_sqrt_unique_indices,

        "unique_fraction":
            float(
                inverse_sqrt_unique_fraction
            ),

        "max_probability_deviation":
            float(
                inverse_sqrt_max_deviation
            ),
    },

    "capped_balanced": {

        "expected_probabilities":
            capped_expected_probs,

        "realized_counts":
            {
                label:
                    int(
                        capped_realized_counts[
                            label
                        ]
                    )
                for label
                in STAGE_C_LABELS
            },

        "realized_probabilities":
            {
                label:
                    float(
                        capped_realized_props[
                            label
                        ]
                    )
                for label
                in STAGE_C_LABELS
            },

        "unique_indices":
            capped_unique_indices,

        "unique_fraction":
            float(
                capped_unique_fraction
            ),

        "max_probability_deviation":
            float(
                capped_max_deviation
            ),
    },

    "checks": {
        key:
            bool(
                value
            )
        for key, value
        in stage_c_sampler_checks.items()
    },

    "all_checks_pass":
        all_stage_c_sampler_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_C_SAMPLER_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_c_sampler_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 13. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE C DETERMINISTIC WEIGHTED-SAMPLER AUDIT")
print("=" * 120)


print("\n[TRAJECTORY GEOMETRY]")
print(
    f"Optimizer steps             : "
    f"{STAGE_C_MAX_STEPS}"
)
print(
    f"Effective batch             : "
    f"{STAGE_C_EFFECTIVE_BATCH}"
)
print(
    f"Total sampled observations  : "
    f"{STAGE_C_TOTAL_DRAWS:,}"
)
print(
    f"Sampler seed                : "
    f"{STAGE_C_SAMPLER_SEED}"
)
print(
    "Replacement                 : True"
)


print("\n[NATURAL REFERENCE]")
print(
    f"Negative                    : "
    f"{natural_expected_probs['negative']:.2%}"
)
print(
    f"Neutral                     : "
    f"{natural_expected_probs['neutral']:.2%}"
)
print(
    f"Positive                    : "
    f"{natural_expected_probs['positive']:.2%}"
)


print("\n[INVERSE-SQRT SAMPLING]")

display(
    inverse_sqrt_audit_table
)

print(
    f"Unique observations         : "
    f"{inverse_sqrt_unique_indices:,} / "
    f"{STAGE_C_TOTAL_DRAWS:,} "
    f"({inverse_sqrt_unique_fraction:.2%})"
)


print("\n[CAPPED-BALANCED SAMPLING]")

display(
    capped_balanced_audit_table
)

print(
    f"Unique observations         : "
    f"{capped_unique_indices:,} / "
    f"{STAGE_C_TOTAL_DRAWS:,} "
    f"({capped_unique_fraction:.2%})"
)


print("\n[SAMPLER CHECKS]")

for name, passed in stage_c_sampler_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Composition audit           : "
    f"{STAGE_C_SAMPLER_AUDIT_PATH}"
)
print(
    f"Sampler summary             : "
    f"{STAGE_C_SAMPLER_SUMMARY_PATH}"
)


print("\n" + "-" * 120)

if all_stage_c_sampler_checks_pass:

    print(
        "DAY 22 CELL 21 STATUS: PASS — "
        "STAGE C WEIGHTED SAMPLERS VALIDATED; TRAINING PERMITTED"
    )

else:

    print(
        "DAY 22 CELL 21 STATUS: FAIL — "
        "DO NOT START STAGE C TRAINING"
    )

print("=" * 120)

DAY 22 — STAGE C DETERMINISTIC WEIGHTED-SAMPLER AUDIT

[TRAJECTORY GEOMETRY]
Optimizer steps             : 500
Effective batch             : 8
Total sampled observations  : 4,000
Sampler seed                : 42
Replacement                 : True

[NATURAL REFERENCE]
Negative                    : 21.01%
Neutral                     : 62.48%
Positive                    : 16.51%

[INVERSE-SQRT SAMPLING]


,label,original_count,weight,expected_pct,realized_count_4000,realized_pct
0,negative,2276,1.724297,27.69599,1068,26.700
1,neutral,6767,1.000000,47.75610,1917,47.925
2,positive,1788,1.945424,24.54791,1015,25.375


Unique observations         : 3,257 / 4,000 (81.42%)

[CAPPED-BALANCED SAMPLING]


,label,original_count,weight,expected_pct,realized_count_4000,realized_pct
0,negative,2276,2.973199,35.808022,1425,35.625
1,neutral,6767,1.000000,35.808022,1458,36.450
2,positive,1788,3.000000,28.383956,1117,27.925


Unique observations         : 3,182 / 4,000 (79.55%)

[SAMPLER CHECKS]
PASS   — total_draws_4000
PASS   — inverse_sqrt_weight_length_10831
PASS   — capped_weight_length_10831
PASS   — inverse_sqrt_draw_n_4000
PASS   — capped_draw_n_4000
PASS   — inverse_sqrt_all_classes_present
PASS   — capped_all_classes_present
PASS   — inverse_sqrt_matches_expected
PASS   — capped_matches_expected
PASS   — inverse_sqrt_not_majority_dominated
PASS   — capped_not_majority_dominated
PASS   — replacement_sampling_preserves_valid_indices
PASS   — seed_42
PASS   — historical_validation_not_used
PASS   — future_not_used

[ARTIFACTS]
Composition audit           : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_c_sampling_audit/artifacts/stage_c_sampler_composition_audit.csv
Sampler summary             : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_c_sampling_audit/artifacts/stage_c_sampler_audit_summary.json

------------------------------------------------------------------

In [23]:
# ==================================================================================================
# DAY 22 — CELL 22
# STAGE C INVERSE-SQRT CANDIDATE
# FRESH MODEL + WEIGHTED-RANDOM-SAMPLER TRAINER INFRASTRUCTURE
# ==================================================================================================

import gc
import random
from pathlib import Path

import numpy as np
import torch

from torch.utils.data import (
    DataLoader,
    WeightedRandomSampler,
)

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# --------------------------------------------------------------------------------------------------
# 1. Candidate identity
# --------------------------------------------------------------------------------------------------

STAGE_C_CURRENT_CONDITION = "inverse_sqrt"

STAGE_C_INVERSE_DIR = (
    STAGE_C_DIR
    / "inverse_sqrt"
)

STAGE_C_INVERSE_ARTIFACT_DIR = (
    STAGE_C_INVERSE_DIR
    / "artifacts"
)

STAGE_C_INVERSE_MODEL_DIR = (
    STAGE_C_INVERSE_DIR
    / "model"
)

STAGE_C_INVERSE_CHECKPOINT_DIR = (
    STAGE_C_INVERSE_MODEL_DIR
    / "checkpoints"
)


for directory in [
    STAGE_C_INVERSE_ARTIFACT_DIR,
    STAGE_C_INVERSE_MODEL_DIR,
    STAGE_C_INVERSE_CHECKPOINT_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 2. Safety gate — candidate must start clean
# --------------------------------------------------------------------------------------------------

existing_inverse_checkpoints = sorted(
    [
        p
        for p
        in STAGE_C_INVERSE_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_inverse_checkpoints
) == 0, (
    "DAY 22 STOP: existing inverse-sqrt checkpoints detected."
)


# --------------------------------------------------------------------------------------------------
# 3. Release Stage B model/trainer objects from GPU
#
# All Stage B artifacts are already frozen on Drive.
# --------------------------------------------------------------------------------------------------

objects_to_delete = [
    "stage_b_trainer",
    "stage_b_model",
    "stage_b_base_model",
    "stage_b_train_result",
]


for object_name in objects_to_delete:

    if object_name in globals():

        del globals()[
            object_name
        ]


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


gpu_after_cleanup_gb = (
    torch.cuda.memory_allocated()
    / (1024 ** 3)
    if torch.cuda.is_available()
    else None
)


# --------------------------------------------------------------------------------------------------
# 4. Deterministic initialization
# --------------------------------------------------------------------------------------------------

STAGE_C_SEED = 42


random.seed(
    STAGE_C_SEED
)

np.random.seed(
    STAGE_C_SEED
)

torch.manual_seed(
    STAGE_C_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_C_SEED
    )


set_seed(
    STAGE_C_SEED
)


# --------------------------------------------------------------------------------------------------
# 5. Fresh 4-bit base model
# --------------------------------------------------------------------------------------------------

stage_c_bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.float16,
)


stage_c_inverse_base_model = (
    AutoModelForCausalLM.from_pretrained(

        "meta-llama/Meta-Llama-3-8B",

        quantization_config=
            stage_c_bnb_config,

        device_map=
            "auto",

        trust_remote_code=
            True,
    )
)


stage_c_inverse_base_model.config.use_cache = False


stage_c_inverse_base_model = (
    prepare_model_for_kbit_training(

        stage_c_inverse_base_model,

        use_gradient_checkpointing=
            True,
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Fresh LoRA — identical to Stage B
# --------------------------------------------------------------------------------------------------

set_seed(
    STAGE_C_SEED
)


stage_c_inverse_lora_config = LoraConfig(

    r=8,

    lora_alpha=32,

    lora_dropout=0.1,

    target_modules=[
        "q_proj",
        "v_proj",
    ],

    bias="none",

    task_type="CAUSAL_LM",

    inference_mode=False,
)


stage_c_inverse_model = (
    get_peft_model(
        stage_c_inverse_base_model,
        stage_c_inverse_lora_config,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Exact inverse-sqrt sample weights
#
# These are derived from the frozen Optimization Train row order.
# --------------------------------------------------------------------------------------------------

assert len(
    inverse_sqrt_sample_weights
) == len(
    hf_opt_train_model
) == 10831


stage_c_inverse_sampler_weights = (
    inverse_sqrt_sample_weights
    .clone()
    .detach()
    .double()
)


# --------------------------------------------------------------------------------------------------
# 8. Weighted Trainer
#
# We override get_train_dataloader() explicitly so that the training trajectory
# really uses WeightedRandomSampler rather than natural shuffling.
#
# Important:
#
# - replacement=True
# - exactly 4000 draws
# - seed=42
# - batch size = 1
#
# 4000 micro-batches / gradient_accumulation=8 = 500 optimizer steps.
# --------------------------------------------------------------------------------------------------

class StageCWeightedSamplerTrainer(
    Trainer
):

    def __init__(
        self,
        *args,
        sampler_weights=None,
        sampler_num_samples=None,
        sampler_seed=42,
        **kwargs,
    ):

        super().__init__(
            *args,
            **kwargs,
        )

        self.stage_c_sampler_weights = (
            sampler_weights
        )

        self.stage_c_sampler_num_samples = int(
            sampler_num_samples
        )

        self.stage_c_sampler_seed = int(
            sampler_seed
        )


    def get_train_dataloader(
        self,
    ):

        if self.train_dataset is None:

            raise ValueError(
                "Trainer requires a train_dataset."
            )


        generator = torch.Generator()

        generator.manual_seed(
            self.stage_c_sampler_seed
        )


        sampler = WeightedRandomSampler(

            weights=
                self.stage_c_sampler_weights,

            num_samples=
                self.stage_c_sampler_num_samples,

            replacement=
                True,

            generator=
                generator,
        )


        dataloader = DataLoader(

            self.train_dataset,

            batch_size=
                self.args.per_device_train_batch_size,

            sampler=
                sampler,

            collate_fn=
                self.data_collator,

            drop_last=
                self.args.dataloader_drop_last,

            num_workers=
                self.args.dataloader_num_workers,

            pin_memory=
                self.args.dataloader_pin_memory,
        )


        return self.accelerator.prepare(
            dataloader
        )


# --------------------------------------------------------------------------------------------------
# 9. Final-only evaluation/save callback
#
# For Stage C we do not need expensive intermediate Dev evaluations.
#
# Selection is classification-first and occurs after the complete 500-step candidate.
# We only retain the final checkpoint and final eval_loss for diagnostic/tie-break use.
# --------------------------------------------------------------------------------------------------

class FinalStepEvaluationSaveCallback(
    TrainerCallback
):

    def __init__(
        self,
        final_step=500,
    ):

        self.final_step = int(
            final_step
        )

        self.eval_triggered = False
        self.save_triggered = False


    def on_step_end(
        self,
        args,
        state,
        control,
        **kwargs,
    ):

        control.should_evaluate = False
        control.should_save = False

        if int(
            state.global_step
        ) == self.final_step:

            control.should_evaluate = True
            control.should_save = True

            self.eval_triggered = True
            self.save_triggered = True

        return control


stage_c_inverse_callback = (
    FinalStepEvaluationSaveCallback(
        final_step=500
    )
)


# --------------------------------------------------------------------------------------------------
# 10. TrainingArguments
# --------------------------------------------------------------------------------------------------

stage_c_inverse_training_args = TrainingArguments(

    output_dir=
        str(
            STAGE_C_INVERSE_CHECKPOINT_DIR
        ),

    max_steps=500,

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=1e-4,

    lr_scheduler_type="linear",

    warmup_steps=10,

    weight_decay=0.0,

    logging_strategy="steps",

    logging_steps=10,

    # callback-controlled
    eval_strategy="no",

    save_strategy="no",

    load_best_model_at_end=False,

    fp16=True,

    bf16=False,

    gradient_checkpointing=True,

    seed=42,

    data_seed=42,

    report_to=[],

    remove_unused_columns=True,

    label_names=[
        "labels"
    ],

    disable_tqdm=False,
)


# --------------------------------------------------------------------------------------------------
# 11. Build inverse-sqrt Trainer
# --------------------------------------------------------------------------------------------------

stage_c_inverse_trainer = (
    StageCWeightedSamplerTrainer(

        model=
            stage_c_inverse_model,

        args=
            stage_c_inverse_training_args,

        train_dataset=
            hf_opt_train_model,

        eval_dataset=
            hf_opt_dev_model,

        data_collator=
            data_collator,

        processing_class=
            tokenizer,

        callbacks=[
            stage_c_inverse_callback
        ],

        sampler_weights=
            stage_c_inverse_sampler_weights,

        sampler_num_samples=
            STAGE_C_TOTAL_DRAWS,

        sampler_seed=
            STAGE_C_SAMPLER_SEED,
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Parameter audit
# --------------------------------------------------------------------------------------------------

stage_c_inverse_total_params = 0

stage_c_inverse_trainable_params = 0

stage_c_inverse_trainable_names = []


for name, param in stage_c_inverse_model.named_parameters():

    stage_c_inverse_total_params += (
        param.numel()
    )

    if param.requires_grad:

        stage_c_inverse_trainable_params += (
            param.numel()
        )

        stage_c_inverse_trainable_names.append(
            name
        )


stage_c_inverse_unexpected_trainable = [

    name

    for name
    in stage_c_inverse_trainable_names

    if "lora_" not in name
]


# --------------------------------------------------------------------------------------------------
# 13. Dry-run sampler verification
#
# Reconstruct the sampler independently and verify first 4000 indices exactly
# match the deterministic audit from Cell 21.
# --------------------------------------------------------------------------------------------------

dry_generator = torch.Generator()

dry_generator.manual_seed(
    STAGE_C_SAMPLER_SEED
)


dry_sampler = WeightedRandomSampler(

    weights=
        stage_c_inverse_sampler_weights,

    num_samples=
        STAGE_C_TOTAL_DRAWS,

    replacement=True,

    generator=
        dry_generator,
)


dry_indices = list(
    dry_sampler
)


dry_sampler_exact_match = (
    dry_indices
    ==
    inverse_sqrt_indices
)


# --------------------------------------------------------------------------------------------------
# 14. Formal infrastructure checks
# --------------------------------------------------------------------------------------------------

stage_c_inverse_checks = {

    "cuda_available":
        torch.cuda.is_available(),

    "checkpoint_dir_clean":
        len(
            existing_inverse_checkpoints
        )
        == 0,

    "fresh_model_loaded":
        stage_c_inverse_model
        is not None,

    "train_n_10831":
        len(
            hf_opt_train_model
        )
        == 10831,

    "dev_n_2708":
        len(
            hf_opt_dev_model
        )
        == 2708,

    "sampler_weight_length_10831":
        len(
            stage_c_inverse_sampler_weights
        )
        == 10831,

    "sampler_draws_4000":
        STAGE_C_TOTAL_DRAWS
        == 4000,

    "sampler_seed_42":
        STAGE_C_SAMPLER_SEED
        == 42,

    "dry_sampler_exact_match_cell21":
        dry_sampler_exact_match,

    "max_steps_500":
        stage_c_inverse_training_args.max_steps
        == 500,

    "gradient_accumulation_8":
        stage_c_inverse_training_args.gradient_accumulation_steps
        == 8,

    "learning_rate_1e4":
        abs(
            stage_c_inverse_training_args.learning_rate
            - 1e-4
        )
        < 1e-12,

    "linear_scheduler":
        (
            str(
                stage_c_inverse_training_args.lr_scheduler_type.value
                if hasattr(
                    stage_c_inverse_training_args.lr_scheduler_type,
                    "value",
                )
                else
                stage_c_inverse_training_args.lr_scheduler_type
            )
            ==
            "linear"
        ),

    "warmup_10":
        stage_c_inverse_training_args.warmup_steps
        == 10,

    "seed_42":
        stage_c_inverse_training_args.seed
        == 42,

    "lora_r8":
        stage_c_inverse_model.peft_config[
            "default"
        ].r
        == 8,

    "lora_alpha32":
        stage_c_inverse_model.peft_config[
            "default"
        ].lora_alpha
        == 32,

    "trainable_params_match_stage_b":
        stage_c_inverse_trainable_params
        == 3_407_872,

    "only_lora_trainable":
        len(
            stage_c_inverse_unexpected_trainable
        )
        == 0,

    "trainer_global_step_zero":
        int(
            stage_c_inverse_trainer.state.global_step
        )
        == 0,

    "final_eval_not_triggered":
        stage_c_inverse_callback.eval_triggered
        is False,

    "final_save_not_triggered":
        stage_c_inverse_callback.save_triggered
        is False,

    "historical_validation_firewall":
        STAGE_C_PROTOCOL[
            "data"
        ][
            "historical_validation_used"
        ]
        is False,

    "future_firewall":
        STAGE_C_PROTOCOL[
            "data"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


all_stage_c_inverse_checks_pass = bool(
    all(
        stage_c_inverse_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 15. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE C INVERSE-SQRT TRAINER INFRASTRUCTURE")
print("=" * 120)


print("\n[CANDIDATE]")
print(
    "Condition                   : inverse_sqrt"
)
print(
    "Initialization              : fresh base + fresh LoRA"
)
print(
    "Natural Stage B adapter     : NOT LOADED"
)


print("\n[SAMPLER]")
print(
    "Type                        : WeightedRandomSampler"
)
print(
    "Replacement                 : True"
)
print(
    f"Sampler seed                : "
    f"{STAGE_C_SAMPLER_SEED}"
)
print(
    f"Total draws                 : "
    f"{STAGE_C_TOTAL_DRAWS:,}"
)
print(
    f"Exact Cell 21 match         : "
    f"{dry_sampler_exact_match}"
)


print("\n[TRAINING PROTOCOL]")
print(
    f"Max steps                   : "
    f"{stage_c_inverse_training_args.max_steps}"
)
print(
    f"Gradient accumulation       : "
    f"{stage_c_inverse_training_args.gradient_accumulation_steps}"
)
print(
    "Effective batch             : 8"
)
print(
    f"Learning rate               : "
    f"{stage_c_inverse_training_args.learning_rate}"
)
print(
    f"Scheduler                   : "
    f"{stage_c_inverse_training_args.lr_scheduler_type}"
)
print(
    f"Warmup                      : "
    f"{stage_c_inverse_training_args.warmup_steps}"
)
print(
    "Formal Dev evaluation       : step 500 only"
)


print("\n[PARAMETERS]")
print(
    f"Trainable parameters        : "
    f"{stage_c_inverse_trainable_params:,}"
)
print(
    f"Total parameters            : "
    f"{stage_c_inverse_total_params:,}"
)
print(
    f"Unexpected trainable        : "
    f"{len(stage_c_inverse_unexpected_trainable)}"
)


print("\n[PRE-TRAINING STATE]")
print(
    f"Global step                 : "
    f"{stage_c_inverse_trainer.state.global_step}"
)
print(
    f"Final eval triggered        : "
    f"{stage_c_inverse_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_c_inverse_callback.save_triggered}"
)


if gpu_after_cleanup_gb is not None:

    print("\n[GPU]")
    print(
        f"Allocated after cleanup     : "
        f"{gpu_after_cleanup_gb:.2f} GB"
    )

    print(
        f"Allocated after model load  : "
        f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB"
    )


print("\n[INFRASTRUCTURE CHECKS]")

for name, passed in stage_c_inverse_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n" + "-" * 120)

if all_stage_c_inverse_checks_pass:

    print(
        "DAY 22 CELL 22 STATUS: PASS — "
        "INVERSE-SQRT WEIGHTED TRAINER READY; TRAINING NOT YET STARTED"
    )

else:

    print(
        "DAY 22 CELL 22 STATUS: FAIL — "
        "DO NOT START INVERSE-SQRT TRAINING"
    )

print("=" * 120)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

DAY 22 — STAGE C INVERSE-SQRT TRAINER INFRASTRUCTURE

[CANDIDATE]
Condition                   : inverse_sqrt
Initialization              : fresh base + fresh LoRA
Natural Stage B adapter     : NOT LOADED

[SAMPLER]
Type                        : WeightedRandomSampler
Replacement                 : True
Sampler seed                : 42
Total draws                 : 4,000
Exact Cell 21 match         : True

[TRAINING PROTOCOL]
Max steps                   : 500
Gradient accumulation       : 8
Effective batch             : 8
Learning rate               : 0.0001
Scheduler                   : SchedulerType.LINEAR
Warmup                      : 10
Formal Dev evaluation       : step 500 only

[PARAMETERS]
Trainable parameters        : 3,407,872
Total parameters            : 4,544,008,192
Unexpected trainable        : 0

[PRE-TRAINING STATE]
Global step                 : 0
Final eval triggered        : False
Final save triggered        : False

[GPU]
Allocated after cleanup     : 1.97 GB
Allocated

In [24]:
# ==================================================================================================
# DAY 22 — CELL 23
# STAGE C INVERSE-SQRT CANDIDATE
# CONTROLLED 500-STEP WEIGHTED-SAMPLING TRAINING
# ==================================================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch


# --------------------------------------------------------------------------------------------------
# 1. Formal artifact paths
# --------------------------------------------------------------------------------------------------

STAGE_C_INVERSE_TRAINING_LOG_PATH = (
    STAGE_C_INVERSE_ARTIFACT_DIR
    / "inverse_sqrt_training_log.csv"
)

STAGE_C_INVERSE_TRAINING_SUMMARY_PATH = (
    STAGE_C_INVERSE_ARTIFACT_DIR
    / "inverse_sqrt_training_summary.json"
)

STAGE_C_INVERSE_TRAINING_COMPLETE_PATH = (
    STAGE_C_INVERSE_ARTIFACT_DIR
    / "inverse_sqrt_training_complete.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Pre-training safety gate
# --------------------------------------------------------------------------------------------------

assert int(
    stage_c_inverse_trainer.state.global_step
) == 0, (
    "DAY 22 STOP: inverse-sqrt trainer has already advanced beyond step 0."
)


assert (
    stage_c_inverse_callback.eval_triggered
    is False
), (
    "DAY 22 STOP: final evaluation callback was already triggered."
)


assert (
    stage_c_inverse_callback.save_triggered
    is False
), (
    "DAY 22 STOP: final save callback was already triggered."
)


existing_checkpoints_before_training = sorted(
    [
        p
        for p
        in STAGE_C_INVERSE_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


assert len(
    existing_checkpoints_before_training
) == 0, (
    "DAY 22 STOP: inverse-sqrt checkpoint directory is not clean."
)


assert not STAGE_C_INVERSE_TRAINING_COMPLETE_PATH.exists(), (
    "DAY 22 STOP: inverse-sqrt completion marker already exists."
)


# --------------------------------------------------------------------------------------------------
# 3. Reset deterministic seeds immediately before training
# --------------------------------------------------------------------------------------------------

random.seed(
    STAGE_C_SEED
)

np.random.seed(
    STAGE_C_SEED
)

torch.manual_seed(
    STAGE_C_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        STAGE_C_SEED
    )

    torch.cuda.reset_peak_memory_stats()


set_seed(
    STAGE_C_SEED
)


# --------------------------------------------------------------------------------------------------
# 4. Train
# --------------------------------------------------------------------------------------------------

training_start_time = time.time()


stage_c_inverse_train_result = (
    stage_c_inverse_trainer.train()
)


training_end_time = time.time()


training_runtime_seconds = (
    training_end_time
    -
    training_start_time
)

training_runtime_minutes = (
    training_runtime_seconds
    /
    60.0
)


# --------------------------------------------------------------------------------------------------
# 5. Collect Trainer log history
# --------------------------------------------------------------------------------------------------

stage_c_inverse_log_history = (
    stage_c_inverse_trainer.state.log_history
)


stage_c_inverse_log_df = pd.DataFrame(
    stage_c_inverse_log_history
)


stage_c_inverse_log_df.to_csv(
    STAGE_C_INVERSE_TRAINING_LOG_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 6. Final state
# --------------------------------------------------------------------------------------------------

final_global_step = int(
    stage_c_inverse_trainer.state.global_step
)


final_training_loss = float(
    stage_c_inverse_train_result.training_loss
)


# --------------------------------------------------------------------------------------------------
# 7. Extract final Dev evaluation
# --------------------------------------------------------------------------------------------------

final_eval_rows = (
    stage_c_inverse_log_df.loc[
        stage_c_inverse_log_df[
            "eval_loss"
        ].notna()
    ]
    .copy()
)


if len(
    final_eval_rows
) > 0:

    final_eval_row = (
        final_eval_rows
        .sort_values(
            "step"
        )
        .iloc[-1]
    )

    final_eval_step = int(
        final_eval_row[
            "step"
        ]
    )

    final_eval_loss = float(
        final_eval_row[
            "eval_loss"
        ]
    )

    final_eval_runtime = float(
        final_eval_row.get(
            "eval_runtime",
            np.nan,
        )
    )

else:

    final_eval_step = None
    final_eval_loss = None
    final_eval_runtime = None


# --------------------------------------------------------------------------------------------------
# 8. Learning-rate trajectory
# --------------------------------------------------------------------------------------------------

lr_rows = (
    stage_c_inverse_log_df.loc[
        stage_c_inverse_log_df[
            "learning_rate"
        ].notna()
    ]
    .copy()
)


def get_logged_lr_at_step(
    step,
):

    rows = (
        lr_rows.loc[
            lr_rows[
                "step"
            ]
            ==
            step
        ]
    )

    if len(
        rows
    ) == 0:

        return None

    return float(
        rows.iloc[-1][
            "learning_rate"
        ]
    )


lr_step_100 = get_logged_lr_at_step(
    100
)

lr_step_200 = get_logged_lr_at_step(
    200
)

lr_step_300 = get_logged_lr_at_step(
    300
)

lr_step_400 = get_logged_lr_at_step(
    400
)

lr_step_500 = get_logged_lr_at_step(
    500
)


# --------------------------------------------------------------------------------------------------
# 9. Live scheduler end-state
# --------------------------------------------------------------------------------------------------

final_scheduler_epoch = None
final_scheduler_lr = None


if (
    stage_c_inverse_trainer.lr_scheduler
    is not None
):

    final_scheduler_epoch = int(
        stage_c_inverse_trainer.lr_scheduler.last_epoch
    )

    final_scheduler_lr = [
        float(
            value
        )
        for value
        in stage_c_inverse_trainer.lr_scheduler.get_last_lr()
    ]


# --------------------------------------------------------------------------------------------------
# 10. Check final checkpoint
# --------------------------------------------------------------------------------------------------

inverse_final_checkpoint_dir = (
    STAGE_C_INVERSE_CHECKPOINT_DIR
    / "checkpoint-500"
)


checkpoint_500_checks = {

    "checkpoint_exists":
        inverse_final_checkpoint_dir.exists(),

    "adapter_config_exists":
        (
            inverse_final_checkpoint_dir
            / "adapter_config.json"
        ).exists(),

    "adapter_weights_exist":
        (
            (
                inverse_final_checkpoint_dir
                / "adapter_model.safetensors"
            ).exists()
            or
            (
                inverse_final_checkpoint_dir
                / "adapter_model.bin"
            ).exists()
        ),

    "trainer_state_exists":
        (
            inverse_final_checkpoint_dir
            / "trainer_state.json"
        ).exists(),

    "optimizer_state_exists":
        (
            inverse_final_checkpoint_dir
            / "optimizer.pt"
        ).exists(),

    "scheduler_state_exists":
        (
            inverse_final_checkpoint_dir
            / "scheduler.pt"
        ).exists(),
}


# --------------------------------------------------------------------------------------------------
# 11. Detect any unintended intermediate checkpoints
# --------------------------------------------------------------------------------------------------

all_inverse_checkpoints = sorted(
    [
        p.name
        for p
        in STAGE_C_INVERSE_CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if p.is_dir()
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. GPU state
# --------------------------------------------------------------------------------------------------

peak_gpu_gb = None


if torch.cuda.is_available():

    peak_gpu_gb = (
        torch.cuda.max_memory_allocated()
        /
        (1024 ** 3)
    )


# --------------------------------------------------------------------------------------------------
# 13. Formal post-training checks
# --------------------------------------------------------------------------------------------------

stage_c_inverse_training_checks = {

    "training_completed_500_steps":
        final_global_step
        == 500,

    "training_loss_finite":
        np.isfinite(
            final_training_loss
        ),

    "training_log_nonempty":
        len(
            stage_c_inverse_log_df
        )
        > 0,

    "final_eval_triggered":
        stage_c_inverse_callback.eval_triggered
        is True,

    "final_save_triggered":
        stage_c_inverse_callback.save_triggered
        is True,

    "exactly_one_eval":
        len(
            final_eval_rows
        )
        == 1,

    "eval_at_step_500":
        final_eval_step
        == 500,

    "eval_loss_finite":
        (
            final_eval_loss
            is not None
            and
            np.isfinite(
                final_eval_loss
            )
        ),

    "only_checkpoint_500":
        all_inverse_checkpoints
        ==
        [
            "checkpoint-500"
        ],

    "checkpoint_500_complete":
        all(
            checkpoint_500_checks.values()
        ),

    "lr_step_100_observed":
        lr_step_100
        is not None,

    "lr_step_200_observed":
        lr_step_200
        is not None,

    "lr_step_300_observed":
        lr_step_300
        is not None,

    "lr_step_400_observed":
        lr_step_400
        is not None,

    "lr_step_500_observed":
        lr_step_500
        is not None,

    "lr_positive_before_end":
        (
            lr_step_100 is not None
            and
            lr_step_200 is not None
            and
            lr_step_300 is not None
            and
            lr_step_400 is not None
            and
            lr_step_100 > 0
            and
            lr_step_200 > 0
            and
            lr_step_300 > 0
            and
            lr_step_400 > 0
        ),

    "scheduler_reaches_end_at_500":
        (
            final_scheduler_epoch
            == 500
        ),

    "historical_validation_not_used":
        STAGE_C_PROTOCOL[
            "data"
        ][
            "historical_validation_used"
        ]
        is False,

    "future_not_used":
        STAGE_C_PROTOCOL[
            "data"
        ][
            "future_f1_f6_used"
        ]
        is False,
}


all_inverse_training_checks_pass = bool(
    all(
        stage_c_inverse_training_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Save summary
# --------------------------------------------------------------------------------------------------

stage_c_inverse_training_summary = {

    "day":
        22,

    "stage":
        "C",

    "condition":
        "inverse_sqrt",

    "sampling":
        {

            "sampler":
                "WeightedRandomSampler",

            "replacement":
                True,

            "seed":
                STAGE_C_SAMPLER_SEED,

            "draws":
                STAGE_C_TOTAL_DRAWS,

            "class_weights":
                inverse_sqrt_class_weights,
        },

    "training":
        {

            "final_global_step":
                final_global_step,

            "training_loss":
                final_training_loss,

            "runtime_seconds":
                float(
                    training_runtime_seconds
                ),

            "runtime_minutes":
                float(
                    training_runtime_minutes
                ),

            "peak_gpu_gb":
                (
                    float(
                        peak_gpu_gb
                    )
                    if peak_gpu_gb
                    is not None
                    else None
                ),
        },

    "evaluation":
        {

            "step":
                final_eval_step,

            "eval_loss":
                final_eval_loss,

            "eval_runtime":
                final_eval_runtime,
        },

    "learning_rate":
        {

            "step_100":
                lr_step_100,

            "step_200":
                lr_step_200,

            "step_300":
                lr_step_300,

            "step_400":
                lr_step_400,

            "step_500":
                lr_step_500,

            "final_scheduler_epoch":
                final_scheduler_epoch,

            "final_scheduler_lr":
                final_scheduler_lr,
        },

    "checkpoint":
        {

            "directory":
                str(
                    inverse_final_checkpoint_dir
                ),

            "all_checkpoint_dirs":
                all_inverse_checkpoints,

            "checks":
                {
                    key:
                        bool(
                            value
                        )
                    for key, value
                    in checkpoint_500_checks.items()
                },
        },

    "checks":
        {
            key:
                bool(
                    value
                )
            for key, value
            in stage_c_inverse_training_checks.items()
        },

    "all_checks_pass":
        all_inverse_training_checks_pass,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_C_INVERSE_TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage_c_inverse_training_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Completion marker
# --------------------------------------------------------------------------------------------------

if all_inverse_training_checks_pass:

    completion_payload = {

        "status":
            "PASS",

        "condition":
            "inverse_sqrt",

        "global_step":
            final_global_step,

        "checkpoint":
            str(
                inverse_final_checkpoint_dir
            ),

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    with open(
        STAGE_C_INVERSE_TRAINING_COMPLETE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            completion_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )


# --------------------------------------------------------------------------------------------------
# 16. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE C INVERSE-SQRT 500-STEP TRAINING")
print("=" * 120)


print("\n[TRAINING]")
print(
    f"Final global step           : "
    f"{final_global_step}"
)
print(
    f"Final training loss         : "
    f"{final_training_loss:.6f}"
)
print(
    f"Runtime                     : "
    f"{training_runtime_minutes:.2f} minutes"
)

if peak_gpu_gb is not None:

    print(
        f"Peak GPU allocation         : "
        f"{peak_gpu_gb:.2f} GB"
    )


print("\n[FINAL DEV EVALUATION]")
print(
    f"Evaluation step             : "
    f"{final_eval_step}"
)
print(
    f"Evaluation loss             : "
    f"{final_eval_loss:.6f}"
)
print(
    f"Evaluation runtime          : "
    f"{final_eval_runtime:.2f} sec"
)


print("\n[LEARNING-RATE TRAJECTORY]")
print(
    f"Step 100                    : "
    f"{lr_step_100}"
)
print(
    f"Step 200                    : "
    f"{lr_step_200}"
)
print(
    f"Step 300                    : "
    f"{lr_step_300}"
)
print(
    f"Step 400                    : "
    f"{lr_step_400}"
)
print(
    f"Step 500                    : "
    f"{lr_step_500}"
)
print(
    f"Final scheduler epoch       : "
    f"{final_scheduler_epoch}"
)
print(
    f"Final scheduler LR          : "
    f"{final_scheduler_lr}"
)


print("\n[CALLBACK]")
print(
    f"Final eval triggered        : "
    f"{stage_c_inverse_callback.eval_triggered}"
)
print(
    f"Final save triggered        : "
    f"{stage_c_inverse_callback.save_triggered}"
)


print("\n[CHECKPOINT]")
print(
    f"Checkpoint directory        : "
    f"{inverse_final_checkpoint_dir}"
)
print(
    f"All checkpoints             : "
    f"{all_inverse_checkpoints}"
)

for name, passed in checkpoint_500_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[POST-TRAINING CHECKS]")

for name, passed in stage_c_inverse_training_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Training log               : "
    f"{STAGE_C_INVERSE_TRAINING_LOG_PATH}"
)
print(
    f"Training summary           : "
    f"{STAGE_C_INVERSE_TRAINING_SUMMARY_PATH}"
)
print(
    f"Completion marker          : "
    f"{STAGE_C_INVERSE_TRAINING_COMPLETE_PATH}"
)


print("\n" + "-" * 120)

if all_inverse_training_checks_pass:

    print(
        "DAY 22 CELL 23 STATUS: PASS — "
        "INVERSE-SQRT 500-STEP TRAINING COMPLETE; "
        "CLASSIFICATION EVALUATION PENDING"
    )

else:

    print(
        "DAY 22 CELL 23 STATUS: FAIL — "
        "DO NOT EVALUATE OR PROCEED"
    )

print("=" * 120)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


DAY 22 — STAGE C INVERSE-SQRT 500-STEP TRAINING

[TRAINING]
Final global step           : 500
Final training loss         : 0.462005
Runtime                     : 60.41 minutes
Peak GPU allocation         : 8.80 GB

[FINAL DEV EVALUATION]
Evaluation step             : 500
Evaluation loss             : 0.228352
Evaluation runtime          : 520.91 sec

[LEARNING-RATE TRAJECTORY]
Step 100                    : 8.183673469387756e-05
Step 200                    : 6.142857142857143e-05
Step 300                    : 4.102040816326531e-05
Step 400                    : 2.0612244897959186e-05
Step 500                    : 2.0408163265306124e-07
Final scheduler epoch       : 500
Final scheduler LR          : [0.0, 0.0]

[CALLBACK]
Final eval triggered        : True
Final save triggered        : True

[CHECKPOINT]
Checkpoint directory        : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_c_sampling_audit/inverse_sqrt/model/checkpoints/checkpoint-500
All checkpoints             : 

In [25]:
# ==================================================================================================
# DAY 22 — CELL 24
# STAGE C INVERSE-SQRT
# OPTIMIZATION DEV CLASSIFICATION INFERENCE + INCUMBENT COMPARISON
# ==================================================================================================

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)


# --------------------------------------------------------------------------------------------------
# 1. Formal paths
# --------------------------------------------------------------------------------------------------

STAGE_C_INVERSE_PREDICTIONS_PATH = (
    STAGE_C_INVERSE_ARTIFACT_DIR
    / "inverse_sqrt_optimization_dev_predictions.csv"
)

STAGE_C_INVERSE_METRICS_PATH = (
    STAGE_C_INVERSE_ARTIFACT_DIR
    / "inverse_sqrt_optimization_dev_metrics.json"
)

STAGE_C_INVERSE_CLASS_METRICS_PATH = (
    STAGE_C_INVERSE_ARTIFACT_DIR
    / "inverse_sqrt_optimization_dev_class_metrics.csv"
)

STAGE_C_INVERSE_CONFUSION_PATH = (
    STAGE_C_INVERSE_ARTIFACT_DIR
    / "inverse_sqrt_optimization_dev_confusion_matrix.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen inference protocol
# --------------------------------------------------------------------------------------------------

STAGE_C_INFERENCE_BATCH_SIZE = 8
STAGE_C_MAX_NEW_TOKENS = 4

STAGE_C_VALID_LABELS = [
    "negative",
    "neutral",
    "positive",
]


# --------------------------------------------------------------------------------------------------
# 3. Safety gate
# --------------------------------------------------------------------------------------------------

prechecks = {

    "training_complete_marker_exists":
        STAGE_C_INVERSE_TRAINING_COMPLETE_PATH.exists(),

    "model_available":
        stage_c_inverse_model is not None,

    "optimization_dev_n_2708":
        len(
            optimization_dev
        ) == 2708,

    "historical_validation_not_used":
        STAGE_C_PROTOCOL[
            "data"
        ][
            "historical_validation_used"
        ] is False,

    "future_not_used":
        STAGE_C_PROTOCOL[
            "data"
        ][
            "future_f1_f6_used"
        ] is False,
}


assert all(
    prechecks.values()
), (
    "DAY 22 STOP: inverse-sqrt classification evaluation prechecks failed."
)


# --------------------------------------------------------------------------------------------------
# 4. Prepare model/tokenizer for inference
# --------------------------------------------------------------------------------------------------

stage_c_inverse_model.eval()

stage_c_inverse_model.config.use_cache = True

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token


# --------------------------------------------------------------------------------------------------
# 5. Exact Optimization Dev frame
# --------------------------------------------------------------------------------------------------

inverse_dev_frame = (
    optimization_dev[
        [
            "observation_id",
            "channel_username",
            "message_id",
            "date",
            "text",
            "sentiment",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


inverse_dev_prompts = [
    build_inference_prompt(
        text
    )
    for text
    in inverse_dev_frame[
        "text"
    ].tolist()
]


# --------------------------------------------------------------------------------------------------
# 6. Greedy batched inference
# --------------------------------------------------------------------------------------------------

inverse_raw_predictions = []


for start_idx in range(
    0,
    len(
        inverse_dev_prompts
    ),
    STAGE_C_INFERENCE_BATCH_SIZE,
):

    end_idx = min(
        start_idx
        + STAGE_C_INFERENCE_BATCH_SIZE,
        len(
            inverse_dev_prompts
        ),
    )

    batch_prompts = (
        inverse_dev_prompts[
            start_idx:end_idx
        ]
    )

    encoded = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )

    encoded = {
        key:
            value.to(
                stage_c_inverse_model.device
            )
        for key, value
        in encoded.items()
    }

    input_length = (
        encoded[
            "input_ids"
        ].shape[1]
    )

    with torch.inference_mode():

        generated = stage_c_inverse_model.generate(
            **encoded,
            max_new_tokens=
                STAGE_C_MAX_NEW_TOKENS,
            do_sample=
                False,
            pad_token_id=
                tokenizer.eos_token_id,
            eos_token_id=
                tokenizer.eos_token_id,
        )

    generated_only = (
        generated[
            :,
            input_length:
        ]
    )

    decoded = tokenizer.batch_decode(
        generated_only,
        skip_special_tokens=True,
    )

    inverse_raw_predictions.extend(
        [
            text.strip().lower()
            for text
            in decoded
        ]
    )


assert len(
    inverse_raw_predictions
) == 2708


# --------------------------------------------------------------------------------------------------
# 7. Prediction validity
# --------------------------------------------------------------------------------------------------

inverse_predictions = (
    inverse_dev_frame
    .copy()
)


inverse_predictions[
    "prediction_raw"
] = inverse_raw_predictions


inverse_predictions[
    "prediction_valid"
] = (
    inverse_predictions[
        "prediction_raw"
    ]
    .isin(
        STAGE_C_VALID_LABELS
    )
)


inverse_predictions[
    "prediction"
] = (
    inverse_predictions[
        "prediction_raw"
    ]
    .where(
        inverse_predictions[
            "prediction_valid"
        ],
        other="INVALID",
    )
)


inverse_invalid_n = int(
    (
        ~inverse_predictions[
            "prediction_valid"
        ]
    ).sum()
)


inverse_invalid_rate = float(
    inverse_invalid_n
    /
    len(
        inverse_predictions
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Classification metrics
# --------------------------------------------------------------------------------------------------

y_true = (
    inverse_predictions[
        "sentiment"
    ]
    .astype(str)
    .tolist()
)


y_pred = (
    inverse_predictions[
        "prediction"
    ]
    .astype(str)
    .tolist()
)


inverse_accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)


inverse_macro_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=
            STAGE_C_VALID_LABELS,
        average=
            "macro",
        zero_division=
            0,
    )
)


inverse_weighted_f1 = float(
    f1_score(
        y_true,
        y_pred,
        labels=
            STAGE_C_VALID_LABELS,
        average=
            "weighted",
        zero_division=
            0,
    )
)


(
    inverse_precision,
    inverse_recall,
    inverse_class_f1,
    inverse_support,
) = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=
        STAGE_C_VALID_LABELS,
    zero_division=
        0,
)


inverse_class_metrics = pd.DataFrame(
    {
        "label":
            STAGE_C_VALID_LABELS,

        "precision":
            inverse_precision,

        "recall":
            inverse_recall,

        "f1":
            inverse_class_f1,

        "support":
            inverse_support.astype(int),
    }
)


inverse_negative_recall = float(
    inverse_class_metrics.loc[
        inverse_class_metrics[
            "label"
        ] == "negative",
        "recall",
    ].iloc[0]
)


inverse_neutral_recall = float(
    inverse_class_metrics.loc[
        inverse_class_metrics[
            "label"
        ] == "neutral",
        "recall",
    ].iloc[0]
)


inverse_positive_recall = float(
    inverse_class_metrics.loc[
        inverse_class_metrics[
            "label"
        ] == "positive",
        "recall",
    ].iloc[0]
)


inverse_minority_floor = min(
    inverse_negative_recall,
    inverse_positive_recall,
)


# --------------------------------------------------------------------------------------------------
# 9. Confusion matrix
# --------------------------------------------------------------------------------------------------

CONFUSION_LABELS = [
    "negative",
    "neutral",
    "positive",
    "INVALID",
]


inverse_cm = confusion_matrix(
    y_true,
    y_pred,
    labels=
        CONFUSION_LABELS,
)


inverse_confusion_df = pd.DataFrame(
    inverse_cm,
    index=[
        f"true_{label}"
        for label
        in CONFUSION_LABELS
    ],
    columns=[
        f"pred_{label}"
        for label
        in CONFUSION_LABELS
    ],
)


# --------------------------------------------------------------------------------------------------
# 10. Frozen Stage B incumbent
# --------------------------------------------------------------------------------------------------

STAGE_B_INCUMBENT = {

    "accuracy":
        0.840842,

    "macro_f1":
        0.808201,

    "negative_recall":
        0.721344,

    "neutral_recall":
        0.889754,

    "positive_recall":
        0.801126,
}


stage_b_minority_floor = min(
    STAGE_B_INCUMBENT[
        "negative_recall"
    ],
    STAGE_B_INCUMBENT[
        "positive_recall"
    ],
)


# --------------------------------------------------------------------------------------------------
# 11. Candidate minus incumbent deltas
# --------------------------------------------------------------------------------------------------

delta_accuracy = (
    inverse_accuracy
    -
    STAGE_B_INCUMBENT[
        "accuracy"
    ]
)


delta_macro_f1 = (
    inverse_macro_f1
    -
    STAGE_B_INCUMBENT[
        "macro_f1"
    ]
)


delta_negative_recall = (
    inverse_negative_recall
    -
    STAGE_B_INCUMBENT[
        "negative_recall"
    ]
)


delta_neutral_recall = (
    inverse_neutral_recall
    -
    STAGE_B_INCUMBENT[
        "neutral_recall"
    ]
)


delta_positive_recall = (
    inverse_positive_recall
    -
    STAGE_B_INCUMBENT[
        "positive_recall"
    ]
)


delta_minority_floor = (
    inverse_minority_floor
    -
    stage_b_minority_floor
)


# --------------------------------------------------------------------------------------------------
# 12. Formal candidate assessment
#
# Primary metric is Accuracy.
# No final Stage C winner is selected yet because capped-balanced has not been trained.
# --------------------------------------------------------------------------------------------------

beats_incumbent_accuracy = (
    delta_accuracy
    > 0
)


# --------------------------------------------------------------------------------------------------
# 13. Save artifacts
# --------------------------------------------------------------------------------------------------

inverse_predictions.to_csv(
    STAGE_C_INVERSE_PREDICTIONS_PATH,
    index=False,
)


inverse_class_metrics.to_csv(
    STAGE_C_INVERSE_CLASS_METRICS_PATH,
    index=False,
)


inverse_confusion_df.to_csv(
    STAGE_C_INVERSE_CONFUSION_PATH,
    index=True,
)


inverse_metrics_payload = {

    "day":
        22,

    "stage":
        "C",

    "condition":
        "inverse_sqrt",

    "evaluation_population":
        "optimization_dev",

    "n":
        2708,

    "eval_loss":
        float(
            final_eval_loss
        ),

    "accuracy":
        inverse_accuracy,

    "macro_f1":
        inverse_macro_f1,

    "weighted_f1":
        inverse_weighted_f1,

    "class_recall": {

        "negative":
            inverse_negative_recall,

        "neutral":
            inverse_neutral_recall,

        "positive":
            inverse_positive_recall,
    },

    "minority_recall_floor":
        inverse_minority_floor,

    "invalid_prediction_n":
        inverse_invalid_n,

    "invalid_prediction_rate":
        inverse_invalid_rate,

    "delta_vs_stage_b": {

        "accuracy":
            float(
                delta_accuracy
            ),

        "macro_f1":
            float(
                delta_macro_f1
            ),

        "negative_recall":
            float(
                delta_negative_recall
            ),

        "neutral_recall":
            float(
                delta_neutral_recall
            ),

        "positive_recall":
            float(
                delta_positive_recall
            ),

        "minority_recall_floor":
            float(
                delta_minority_floor
            ),
    },

    "beats_stage_b_accuracy":
        bool(
            beats_incumbent_accuracy
        ),

    "firewall": {

        "historical_validation_used":
            False,

        "future_f1_f6_used":
            False,
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    STAGE_C_INVERSE_METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        inverse_metrics_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 14. Formal checks
# --------------------------------------------------------------------------------------------------

inverse_eval_checks = {

    "prediction_n_2708":
        len(
            inverse_predictions
        )
        == 2708,

    "observation_ids_preserved":
        inverse_predictions[
            "observation_id"
        ].tolist()
        ==
        optimization_dev[
            "observation_id"
        ].tolist(),

    "accuracy_finite":
        np.isfinite(
            inverse_accuracy
        ),

    "macro_f1_finite":
        np.isfinite(
            inverse_macro_f1
        ),

    "negative_recall_finite":
        np.isfinite(
            inverse_negative_recall
        ),

    "positive_recall_finite":
        np.isfinite(
            inverse_positive_recall
        ),

    "historical_validation_not_used":
        True,

    "future_not_used":
        True,
}


all_inverse_eval_checks_pass = bool(
    all(
        inverse_eval_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 15. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — STAGE C INVERSE-SQRT CLASSIFICATION EVALUATION")
print("=" * 120)


print("\n[OPTIMIZATION DEV METRICS]")
print(
    f"Accuracy                    : "
    f"{inverse_accuracy:.6f}"
)
print(
    f"Macro-F1                    : "
    f"{inverse_macro_f1:.6f}"
)
print(
    f"Weighted-F1                 : "
    f"{inverse_weighted_f1:.6f}"
)
print(
    f"Negative recall             : "
    f"{inverse_negative_recall:.6f}"
)
print(
    f"Neutral recall              : "
    f"{inverse_neutral_recall:.6f}"
)
print(
    f"Positive recall             : "
    f"{inverse_positive_recall:.6f}"
)
print(
    f"Minority recall floor       : "
    f"{inverse_minority_floor:.6f}"
)


print("\n[GENERATION VALIDITY]")
print(
    f"Invalid predictions         : "
    f"{inverse_invalid_n}"
)
print(
    f"Invalid rate                : "
    f"{inverse_invalid_rate:.4%}"
)


print("\n[CLASS METRICS]")

display(
    inverse_class_metrics
)


print("\n[CONFUSION MATRIX]")

display(
    inverse_confusion_df
)


print("\n[VS STAGE B-500 INCUMBENT]")
print(
    f"Accuracy delta              : "
    f"{delta_accuracy:+.6f} "
    f"({delta_accuracy * 100:+.2f} pp)"
)
print(
    f"Macro-F1 delta              : "
    f"{delta_macro_f1:+.6f}"
)
print(
    f"Negative recall delta       : "
    f"{delta_negative_recall:+.6f}"
)
print(
    f"Neutral recall delta        : "
    f"{delta_neutral_recall:+.6f}"
)
print(
    f"Positive recall delta       : "
    f"{delta_positive_recall:+.6f}"
)
print(
    f"Minority floor delta        : "
    f"{delta_minority_floor:+.6f}"
)
print(
    f"Beats incumbent Accuracy    : "
    f"{beats_incumbent_accuracy}"
)


print("\n[EVALUATION CHECKS]")

for name, passed in inverse_eval_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[ARTIFACTS]")
print(
    f"Predictions                 : "
    f"{STAGE_C_INVERSE_PREDICTIONS_PATH}"
)
print(
    f"Metrics                     : "
    f"{STAGE_C_INVERSE_METRICS_PATH}"
)
print(
    f"Class metrics               : "
    f"{STAGE_C_INVERSE_CLASS_METRICS_PATH}"
)
print(
    f"Confusion matrix            : "
    f"{STAGE_C_INVERSE_CONFUSION_PATH}"
)


print("\n" + "-" * 120)

if all_inverse_eval_checks_pass:

    print(
        "DAY 22 CELL 24 STATUS: PASS — "
        "INVERSE-SQRT CLASSIFICATION EVALUATION COMPLETE; "
        "CAPPED-BALANCED CANDIDATE STILL PENDING"
    )

else:

    print(
        "DAY 22 CELL 24 STATUS: FAIL — "
        "DO NOT INTERPRET INVERSE-SQRT PERFORMANCE"
    )

print("=" * 120)

[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=4) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

DAY 22 — STAGE C INVERSE-SQRT CLASSIFICATION EVALUATION

[OPTIMIZATION DEV METRICS]
Accuracy                    : 0.822009
Macro-F1                    : 0.796423
Weighted-F1                 : 0.824807
Negative recall             : 0.810277
Neutral recall              : 0.823847
Positive recall             : 0.827392
Minority recall floor       : 0.810277

[GENERATION VALIDITY]
Invalid predictions         : 0
Invalid rate                : 0.0000%

[CLASS METRICS]


,label,precision,recall,f1,support
0,negative,0.710572,0.810277,0.757156,506
1,neutral,0.906394,0.823847,0.863151,1669
2,positive,0.718241,0.827392,0.768963,533



[CONFUSION MATRIX]


,pred_negative,pred_neutral,pred_positive,pred_INVALID
true_negative,410,75,21,0
true_neutral,142,1375,152,0
true_positive,25,67,441,0
true_INVALID,0,0,0,0



[VS STAGE B-500 INCUMBENT]
Accuracy delta              : -0.018833 (-1.88 pp)
Macro-F1 delta              : -0.011778
Negative recall delta       : +0.088933
Neutral recall delta        : -0.065907
Positive recall delta       : +0.026266
Minority floor delta        : +0.088933
Beats incumbent Accuracy    : False

[EVALUATION CHECKS]
PASS   — prediction_n_2708
PASS   — observation_ids_preserved
PASS   — accuracy_finite
PASS   — macro_f1_finite
PASS   — negative_recall_finite
PASS   — positive_recall_finite
PASS   — historical_validation_not_used
PASS   — future_not_used

[ARTIFACTS]
Predictions                 : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_c_sampling_audit/inverse_sqrt/artifacts/inverse_sqrt_optimization_dev_predictions.csv
Metrics                     : /content/drive/MyDrive/FinGPT_Project/results/day22/runs/stage_c_sampling_audit/inverse_sqrt/artifacts/inverse_sqrt_optimization_dev_metrics.json
Class metrics               : /content/drive/MyDrive/Fi

In [26]:
# ==================================================================================================
# DAY 22 — CELL 25
# PART 1 FORMAL CHECKPOINT + RESUME MANIFEST
# ==================================================================================================

import json
from pathlib import Path
from datetime import datetime, timezone


# --------------------------------------------------------------------------------------------------
# 1. Day 22 root
# --------------------------------------------------------------------------------------------------

DAY22_RESULTS_ROOT = (PROJECT_ROOT / 'results' / 'day22')

DAY22_PART1_CHECKPOINT_DIR = (
    DAY22_RESULTS_ROOT
    / "checkpoints"
)

DAY22_PART1_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DAY22_PART1_CHECKPOINT_PATH = (
    DAY22_PART1_CHECKPOINT_DIR
    / "day22_part1_checkpoint.json"
)

DAY22_PART1_RESUME_PATH = (
    DAY22_PART1_CHECKPOINT_DIR
    / "day22_part1_resume_manifest.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Required completed artifacts
# --------------------------------------------------------------------------------------------------

required_artifacts = {

    # Stage A
    "stage_a_config":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_a_baseline' / 'artifacts' / 'stage_a_baseline_config.json'),

    "stage_a_selected_adapter":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_a_baseline' / 'model' / 'selected_adapter'),

    "stage_a_dev_metrics":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_a_baseline' / 'artifacts' / 'stage_a_200_optimization_dev_metrics.json'),

    # Stage B
    "stage_b_revised_protocol":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_b_horizon_audit' / 'artifacts' / 'stage_b_horizon_audit_config_revised.json'),

    "stage_b_selection":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_b_horizon_audit' / 'artifacts' / 'stage_b_horizon_selection_summary.json'),

    "stage_b_selected_adapter":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_b_horizon_audit' / 'model' / 'selected_adapter'),

    "stage_b_dev_metrics":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_b_horizon_audit' / 'artifacts' / 'stage_b_500_optimization_dev_metrics.json'),

    # Fair Stage A vs B comparison
    "stage_a_vs_b_comparison":
        (PROJECT_ROOT / 'results' / 'day22' / 'final' / 'stage_a_vs_stage_b_optimization_dev_comparison.json'),

    # Stage C protocol / sampler
    "stage_c_protocol":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_c_sampling_audit' / 'artifacts' / 'stage_c_sampling_protocol.json'),

    "stage_c_sampler_audit":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_c_sampling_audit' / 'artifacts' / 'stage_c_sampler_audit_summary.json'),

    # inverse-sqrt candidate
    "inverse_sqrt_training_complete":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_c_sampling_audit' / 'inverse_sqrt' / 'artifacts' / 'inverse_sqrt_training_complete.json'),

    "inverse_sqrt_metrics":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_c_sampling_audit' / 'inverse_sqrt' / 'artifacts' / 'inverse_sqrt_optimization_dev_metrics.json'),

    "inverse_sqrt_checkpoint_500":
        (PROJECT_ROOT / 'results' / 'day22' / 'runs' / 'stage_c_sampling_audit' / 'inverse_sqrt' / 'model' / 'checkpoints' / 'checkpoint-500'),
}


# --------------------------------------------------------------------------------------------------
# 3. Artifact audit
# --------------------------------------------------------------------------------------------------

artifact_checks = {

    name:
        path.exists()

    for name, path
    in required_artifacts.items()
}


all_artifacts_present = bool(
    all(
        artifact_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Frozen Part 1 scientific state
# --------------------------------------------------------------------------------------------------

part1_state = {

    "day":
        22,

    "part":
        1,

    "status":
        "COMPLETE",

    "research_objective":
        "Improve historical financial sentiment model accuracy before renewed temporal evaluation.",

    "completed_stages": {

        "stage_a": {

            "status":
                "COMPLETE",

            "role":
                "200-step historical baseline",

            "optimization_dev_accuracy":
                0.779542,

            "optimization_dev_macro_f1":
                0.726037,
        },

        "stage_b": {

            "status":
                "COMPLETE",

            "role":
                "training horizon adequacy audit",

            "selected_horizon":
                500,

            "selected_eval_loss":
                0.215775,

            "optimization_dev_accuracy":
                0.840842,

            "optimization_dev_macro_f1":
                0.808201,

            "accuracy_gain_vs_stage_a_pp":
                6.13,

            "conclusion":
                (
                    "The original 200-step training budget was materially insufficient. "
                    "A controlled 500-step trajectory substantially improved classification performance."
                ),
        },

        "stage_c_inverse_sqrt": {

            "status":
                "COMPLETE",

            "sampling":
                "inverse_sqrt",

            "optimization_dev_accuracy":
                0.822009,

            "optimization_dev_macro_f1":
                0.796423,

            "negative_recall":
                0.810277,

            "neutral_recall":
                0.823847,

            "positive_recall":
                0.827392,

            "accuracy_delta_vs_stage_b_pp":
                -1.88,

            "beats_stage_b_accuracy":
                False,

            "interpretation":
                (
                    "Moderate class rebalancing improved minority-class recall "
                    "but reduced neutral recall and overall accuracy."
                ),
        },
    },

    "current_incumbent": {

        "model":
            "Stage B-500 natural sampling",

        "accuracy":
            0.840842,

        "macro_f1":
            0.808201,

        "negative_recall":
            0.721344,

        "neutral_recall":
            0.889754,

        "positive_recall":
            0.801126,
    },

    "data_firewall": {

        "historical_validation_used_for_tuning":
            False,

        "future_f1_f6_used_for_tuning":
            False,

        "status":
            "INTACT",
    },

    "stage_c_status": {

        "natural":
            "COMPLETE_INCUMBENT",

        "inverse_sqrt":
            "COMPLETE_NOT_WINNER",

        "capped_balanced":
            "PENDING",
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 5. Resume manifest
# --------------------------------------------------------------------------------------------------

resume_manifest = {

    "day":
        22,

    "resume_from":
        "Stage C capped-balanced candidate",

    "next_formal_task":
        "Initialize fresh capped-balanced weighted-sampling model and trainer.",

    "do_not_repeat": [

        "Stage A baseline training",
        "Stage B 500-step horizon audit",
        "Stage A vs Stage B fair comparison",
        "Stage C protocol freeze",
        "Stage C deterministic sampler audit",
        "inverse-sqrt 500-step training",
        "inverse-sqrt Optimization Dev classification evaluation",
    ],

    "pending_sequence": [

        "Stage C capped-balanced infrastructure",
        "Stage C capped-balanced 500-step training",
        "Stage C capped-balanced classification evaluation",
        "Stage C three-condition selection",

        "Stage D learning-rate audit",
        "Stage D winner selection",

        "Stage E LoRA-capacity audit",
        "Stage E winner selection",

        "multi-seed confirmation",

        "final configuration freeze",

        "one-shot Historical Validation evaluation",

        "Day 22 final artifact audit and notebook completion",
    ],

    "current_incumbent":
        "Stage B-500 natural sampling",

    "primary_selection_metric":
        "accuracy",

    "optimization_dev_only":
        True,

    "historical_validation_locked":
        True,

    "future_f1_f6_locked":
        True,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 6. Save checkpoint artifacts
# --------------------------------------------------------------------------------------------------

with open(
    DAY22_PART1_CHECKPOINT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        part1_state,
        f,
        indent=2,
        ensure_ascii=False,
    )


with open(
    DAY22_PART1_RESUME_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        resume_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 7. Formal checks
# --------------------------------------------------------------------------------------------------

checkpoint_checks = {

    "day22_root_exists":
        DAY22_RESULTS_ROOT.exists(),

    "all_required_artifacts_present":
        all_artifacts_present,

    "checkpoint_written":
        DAY22_PART1_CHECKPOINT_PATH.exists(),

    "resume_manifest_written":
        DAY22_PART1_RESUME_PATH.exists(),

    "stage_a_complete":
        True,

    "stage_b_complete":
        True,

    "stage_c_protocol_frozen":
        True,

    "inverse_sqrt_complete":
        True,

    "inverse_sqrt_not_incumbent":
        True,

    "capped_balanced_pending":
        True,

    "incumbent_stage_b_500":
        True,

    "historical_validation_locked":
        True,

    "future_locked":
        True,
}


all_checkpoint_checks_pass = bool(
    all(
        checkpoint_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Report
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("DAY 22 — PART 1 FORMAL CHECKPOINT")
print("=" * 120)


print("\n[COMPLETED]")
print(
    "Stage A                    : COMPLETE"
)
print(
    "Stage B                    : COMPLETE"
)
print(
    "Stage C protocol           : FROZEN"
)
print(
    "Stage C inverse-sqrt       : COMPLETE"
)


print("\n[CURRENT INCUMBENT]")
print(
    "Model                      : Stage B-500 natural sampling"
)
print(
    "Optimization Dev Accuracy  : 0.840842"
)
print(
    "Macro-F1                   : 0.808201"
)


print("\n[INVERSE-SQRT RESULT]")
print(
    "Accuracy                   : 0.822009"
)
print(
    "Delta vs incumbent         : -1.88 pp"
)
print(
    "Beats incumbent            : False"
)


print("\n[PENDING]")
print(
    "Next candidate             : capped-balanced"
)
print(
    "Historical Validation      : LOCKED"
)
print(
    "Future F1-F6               : LOCKED"
)


print("\n[ARTIFACT AUDIT]")

for name, passed in artifact_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[CHECKPOINT CHECKS]")

for name, passed in checkpoint_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )


print("\n[RESUME POINT]")
print(
    "Tomorrow resume at         : Stage C capped-balanced candidate infrastructure"
)


print("\n[ARTIFACTS]")
print(
    f"Part 1 checkpoint          : "
    f"{DAY22_PART1_CHECKPOINT_PATH}"
)
print(
    f"Resume manifest            : "
    f"{DAY22_PART1_RESUME_PATH}"
)


print("\n" + "-" * 120)

if all_checkpoint_checks_pass:

    print(
        "DAY 22 PART 1 STATUS: PASS — "
        "FORMAL CHECKPOINT COMPLETE; SAFE TO STOP RUNTIME"
    )

else:

    print(
        "DAY 22 PART 1 STATUS: FAIL — "
        "DO NOT TERMINATE RUNTIME UNTIL FAILED ARTIFACTS ARE REVIEWED"
    )

print("=" * 120)

DAY 22 — PART 1 FORMAL CHECKPOINT

[COMPLETED]
Stage A                    : COMPLETE
Stage B                    : COMPLETE
Stage C protocol           : FROZEN
Stage C inverse-sqrt       : COMPLETE

[CURRENT INCUMBENT]
Model                      : Stage B-500 natural sampling
Optimization Dev Accuracy  : 0.840842
Macro-F1                   : 0.808201

[INVERSE-SQRT RESULT]
Accuracy                   : 0.822009
Delta vs incumbent         : -1.88 pp
Beats incumbent            : False

[PENDING]
Next candidate             : capped-balanced
Historical Validation      : LOCKED
Future F1-F6               : LOCKED

[ARTIFACT AUDIT]
PASS   — stage_a_config
PASS   — stage_a_selected_adapter
PASS   — stage_a_dev_metrics
PASS   — stage_b_revised_protocol
PASS   — stage_b_selection
PASS   — stage_b_selected_adapter
PASS   — stage_b_dev_metrics
PASS   — stage_a_vs_b_comparison
PASS   — stage_c_protocol
PASS   — stage_c_sampler_audit
PASS   — inverse_sqrt_training_complete
PASS   — inverse_sqrt_metri